In [ ]:

import sys

print("Python version:", sys.version)

import pandas as pd
import numpy as np
import sklearn

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nJUDGEBERT project environment is ready!")



Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Pandas: 2.2.3
NumPy: 2.1.3
Scikit-learn: 1.6.1

JUDGEBERT project environment is ready!


In [ ]:
!pip -q install datasets pandas

In [ ]:
!python prepare_lmp_dataset.py --output /content/lmp_silver_dataset.csv --rows 1500

python3: can't open file '/content/prepare_lmp_dataset.py': [Errno 2] No such file or directory


In [ ]:
from google.colab import files
files.download('/content/lmp_silver_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip -q install datasets pandas

from datasets import load_dataset

dataset = load_dataset(
    "mohitskaushal/MILDSum_Dataset_Original_V2_with_eng_and_hindi_summary"
)

print(dataset)
print("\nAvailable columns:")
print(dataset["train"].column_names)

print("\nFirst example:")
example = dataset["train"][0]

for column, value in example.items():
    print(f"\n--- {column} ---")
    print(str(value)[:1200])

README.md:   0%|          | 0.00/564 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 30.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 8.26MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/val-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.16MB            

data/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1506 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/438 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/430 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['judgement', 'english_summary', 'hindi_summary'],
        num_rows: 1506
    })
    test: Dataset({
        features: ['judgement', 'english_summary', 'hindi_summary'],
        num_rows: 438
    })
    val: Dataset({
        features: ['judgement', 'english_summary', 'hindi_summary'],
        num_rows: 430
    })
})

Available columns:
['judgement', 'english_summary', 'hindi_summary']

First example:

--- judgement ---
1.     The revisionist herein is an advocate who has been fastened with a
criminal liability under section 420, 468, 471, 120B of IPC for providing false
and improper legal advice which was instrumental in sanction of a bank loan
to a company which turned NPA.
2.     Petitioner filed this application under section 401 read with section 482
of the Code of Criminal Procedure, 1973 praying for quashing of proceedings in
GR Case No. 4105 of 2013, arising out of Central Bureau of Investigation,
Enforcement Offences Wing, K

In [ ]:
import pandas as pd

# Use the training split
df = dataset["train"].to_pandas()

print("Original dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# Find the actual text columns
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].dropna().astype(str).head(2).tolist())

Original dataset shape: (1506, 3)
Columns: ['judgement', 'english_summary', 'hindi_summary']

judgement:
['1.     The revisionist herein is an advocate who has been fastened with a\ncriminal liability under section 420, 468, 471, 120B of IPC for providing false\nand improper legal advice which was instrumental in sanction of a bank loan\nto a company which turned NPA.\n2.     Petitioner filed this application under section 401 read with section 482\nof the Code of Criminal Procedure, 1973 praying for quashing of proceedings in\nGR Case No. 4105 of 2013, arising out of Central Bureau of Investigation,\nEnforcement Offences Wing, Kolkata Case No. RC 07/E/011-Kolkata dated\n16.12.2011 under sections 420, 468, 471 read with section 120B of the Indian\nPenal Code, pending before Learned Metropolitan Magistrate 22nd Court at\n3.     Mr. Partha Das Chowdhury and Smt. Baisali Mukherjee the two directors\nof Emotions Infomedia Private Limited applied for cash credit loan facility\nbefore Opposi

In [ ]:
# Show text lengths correctly
text_columns = ["judgement", "english_summary", "hindi_summary"]

for col in text_columns:
    lengths = clean_df[col].str.len()
    print(f"\n{col}:")
    print(lengths.describe())

# Save the cleaned dataset
output_path = "/content/legal_text_dataset_v1.csv"
clean_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nDataset rows:", len(clean_df))
print("Saved to:", output_path)

# Download the CSV
from google.colab import files
files.download(output_path)

NameError: name 'clean_df' is not defined

In [ ]:
import pandas as pd
import re

# Inspect a few actual records
print("Dataset size:", len(clean_df))

for i in range(min(3, len(clean_df))):
    row = clean_df.iloc[i]

    print(f"\n{'='*60}")
    print("Record:", row["record_id"])
    print("\nENGLISH SUMMARY:\n", row["english_summary"][:700])
    print("\nHINDI SUMMARY:\n", row["hindi_summary"][:700])

# Split one judgment into rough English-style sentences for inspection
sample_judgment = clean_df.iloc[0]["judgement"]
sentences = re.split(r'(?<=[.!?])\s+', sample_judgment)

print("\nFirst 10 extracted text segments:")
for sentence in sentences[:10]:
    if sentence.strip():
        print("-", sentence[:300])

In [ ]:
import pandas as pd

# Start from the cleaned dataset already in memory
pairs = clean_df.copy()

# Keep only records with all three text fields
text_cols = ["judgement", "english_summary", "hindi_summary"]
for col in text_cols:
    pairs[col] = pairs[col].fillna("").astype(str).str.strip()

pairs = pairs[
    pairs["judgement"].str.len().ge(100) &
    pairs["english_summary"].str.len().ge(50) &
    pairs["hindi_summary"].str.len().ge(50)
].copy()

# Add provenance and explicit label status
pairs["pair_type"] = "judgment_and_summaries"
pairs["meaning_preserved"] = pd.NA
pairs["label_status"] = "unlabelled_not_human_verified"

output_path = "/content/legal_text_pairs_v1.csv"
pairs.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Records prepared:", len(pairs))
print("Columns:", pairs.columns.tolist())
print("Labels intentionally left blank: summaries are not verified LMP labels.")

from google.colab import files
files.download(output_path)

In [ ]:
!pip -q install scikit-learn pandas
!python step2_build_candidate_pairs_colab.py

In [ ]:
from IPython.display import display
import pandas as pd

pairs_v2 = pd.read_csv("/content/lmp_candidate_pairs_v2.csv")

display(pairs_v2.head(10))

print("\nLanguage counts:")
print(pairs_v2["language"].value_counts())

print("\nLabel counts:")
print(pairs_v2.groupby(["language", "meaning_preserved"]).size())

In [ ]:
from google.colab import files
files.download("/content/lmp_candidate_pairs_v2.csv")

In [ ]:
import pandas as pd
import re

df = pd.read_csv("/content/lmp_candidate_pairs_v2.csv")
new_rows = []

hindi_negations = [
    ("नहीं करेगा", "करेगा"),
    ("नहीं करेगी", "करेगी"),
    ("नहीं करेंगे", "करेंगे"),
    ("नहीं होगा", "होगा"),
    ("नहीं होगी", "होगी"),
    ("नहीं है", "है"),
    ("नहीं", "")
]

for _, row in df[df["language"] == "hi"].drop_duplicates(
    subset=["source_record_id"]
).iterrows():
    text = str(row["original_text"]).strip()
    candidate = None

    for old, new in hindi_negations:
        if old in text:
            changed = text.replace(old, new, 1).strip()
            if len(changed) >= 40 and changed != text:
                candidate = changed
                break

    if candidate is None:
        match = re.search(r"\d+", text)
        if match:
            number = int(match.group())
            candidate = (
                text[:match.start()] + str(number + 1) +
                text[match.end():]
            )

    if candidate and candidate != text:
        new_rows.append({
            "pair_id": f"HI_SYN_{len(new_rows)+1:06d}",
            "source_record_id": row["source_record_id"],
            "language": "hi",
            "original_text": text,
            "candidate_text": candidate,
            "meaning_preserved": 0,
            "pair_type": "synthetic_changed_candidate",
            "label_basis": "automatic_hindi_edit_rule",
            "human_validated": False,
            "split": "train"
        })

hindi_negatives = pd.DataFrame(new_rows)
combined = pd.concat([df, hindi_negatives], ignore_index=True)

combined.to_csv(
    "/content/lmp_candidate_pairs_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Total examples:", len(combined))
print("\nLanguage and label counts:")
print(combined.groupby(["language", "meaning_preserved"]).size())
print("\nHindi negative examples added:", len(hindi_negatives))

In [ ]:
!python lmp_corrected_source_material_v1.py

In [ ]:
!pip -q install datasets pandas

from datasets import load_dataset

legal_pairs = load_dataset(
    "Karthikrv/adaption-legal-clause-simplification"
)

print(legal_pairs)
print("\nColumns:")
print(legal_pairs["train"].column_names)

print("\nExamples:")
for row in legal_pairs["train"].select(range(5)):
    print("\nORIGINAL:", row.get("Input", ""))
    print("SIMPLIFIED:", row.get("Output", ""))

In [ ]:
print("Dataset columns:")
print(legal_pairs["train"].column_names)

print("\nFirst 3 examples:")
for i in range(min(3, len(legal_pairs["train"]))):
    row = legal_pairs["train"][i]
    for key, value in row.items():
        print(f"{key}: {str(value)[:500]}")
    print("-" * 50)

In [ ]:
import pandas as pd

# Use the dataset already loaded in Colab
raw = legal_pairs["train"].to_pandas()

print("Available columns:", raw.columns.tolist())

# Confirm the expected fields exist
required = ["Input", "Output"]
missing = [c for c in required if c not in raw.columns]

if missing:
    print("Expected columns not found:", missing)
else:
    pairs = raw[["Input", "Output"]].copy()
    pairs.columns = ["original_text", "simplified_text"]

    pairs["original_text"] = (
        pairs["original_text"].fillna("").astype(str).str.strip()
    )
    pairs["simplified_text"] = (
        pairs["simplified_text"].fillna("").astype(str).str.strip()
    )

    # Remove empty entries and exact duplicates
    pairs = pairs[
        (pairs["original_text"].str.len() >= 20) &
        (pairs["simplified_text"].str.len() >= 20)
    ].drop_duplicates().reset_index(drop=True)

    pairs.insert(
        0, "pair_id",
        [f"EN_{i:05d}" for i in range(1, len(pairs) + 1)]
    )
    pairs["language"] = "en"
    pairs["lmp_score_1_to_10"] = pd.NA
    pairs["score_status"] = "not_yet_rated"
    pairs["source_dataset"] = (
        "Karthikrv/adaption-legal-clause-simplification"
    )

    output_path = "/content/legal_simplification_pairs_en_v1.csv"
    pairs.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("Clean English pairs:", len(pairs))
    print(pairs.head(5).to_string(index=False))

    from google.colab import files
    files.download(output_path)

In [ ]:
import pandas as pd

pairs = pd.read_csv(
    "/content/legal_simplification_pairs_en_v1.csv"
)

pairs["original_len"] = pairs["original_text"].str.len()
pairs["simplified_len"] = pairs["simplified_text"].str.len()

pairs["identical"] = (
    pairs["original_text"].str.strip()
    == pairs["simplified_text"].str.strip()
)

pairs["simplification_ratio"] = (
    pairs["simplified_len"] / pairs["original_len"].clip(lower=1)
)

print("Total pairs:", len(pairs))
print("Identical pairs:", pairs["identical"].sum())
print("Different pairs:", (~pairs["identical"]).sum())
print(
    "Simplified text shorter:",
    (pairs["simplified_len"] < pairs["original_len"]).sum()
)
print("\nLength statistics:")
print(pairs[["original_len", "simplified_len", "simplification_ratio"]].describe())

print("\nSample pairs:")
for _, row in pairs.head(5).iterrows():
    print("\nORIGINAL:", row["original_text"][:400])
    print("SIMPLIFIED:", row["simplified_text"][:400])

In [ ]:
import pandas as pd

pairs = pd.read_csv(
    "/content/legal_simplification_pairs_en_v1.csv"
)

pairs["original_text"] = pairs["original_text"].fillna("").astype(str).str.strip()
pairs["simplified_text"] = pairs["simplified_text"].fillna("").astype(str).str.strip()

# Remove empty rows and duplicate pairs
pairs = pairs[
    (pairs["original_text"].str.len() >= 20) &
    (pairs["simplified_text"].str.len() >= 20)
].drop_duplicates(subset=["original_text", "simplified_text"]).copy()

# Keep reasonable length ratios for an initial inspection set.
# This does not prove that meaning is preserved.
pairs["length_ratio"] = (
    pairs["simplified_text"].str.len()
    / pairs["original_text"].str.len().clip(lower=1)
)

filtered = pairs[
    pairs["length_ratio"].between(0.25, 2.0)
].copy()

print("Pairs before filtering:", len(pairs))
print("Pairs after filtering:", len(filtered))
print("\nLength ratio statistics:")
print(filtered["length_ratio"].describe())

print("\nInspect 10 pairs:")
for _, row in filtered.sample(
    min(10, len(filtered)), random_state=42
).iterrows():
    print("\nORIGINAL:", row["original_text"][:500])
    print("SIMPLIFIED:", row["simplified_text"][:500])
    print("-" * 60)

filtered.to_csv(
    "/content/legal_simplification_pairs_en_filtered.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved: /content/legal_simplification_pairs_en_filtered.csv")


In [ ]:
import pandas as pd

pairs = pd.read_csv(
    "/content/legal_simplification_pairs_en_filtered.csv"
)

# Inspect pairs that may not be legal clauses
keywords = [
    "tenant", "landlord", "contract", "agreement", "court",
    "claim", "insurer", "shall", "must", "liability",
    "party", "payment", "law", "policy", "employee"
]

def legal_keyword_count(text):
    text = str(text).lower()
    return sum(word in text for word in keywords)

pairs["legal_keyword_count"] = (
    pairs["original_text"].apply(legal_keyword_count)
)

print("Total pairs:", len(pairs))
print("\nPairs with no legal keywords:",
      (pairs["legal_keyword_count"] == 0).sum())

print("\nExamples with no legal keywords:")
display(
    pairs[pairs["legal_keyword_count"] == 0][
        ["original_text", "simplified_text"]
    ].head(10)
)

print("\nExamples with legal keywords:")
display(
    pairs[pairs["legal_keyword_count"] > 0][
        ["original_text", "simplified_text"]
    ].head(10)
)

In [ ]:
!pip -q install datasets pandas

from datasets import load_dataset
import pandas as pd

legal_hi = load_dataset("Wasserstoff-AI/legalTransEn_Indic")

print(legal_hi)
print("Columns:", legal_hi["train"].column_names)

# Inspect actual fields before extracting anything
for row in legal_hi["train"].select(range(3)):
    print("\n", row)

In [ ]:
import pandas as pd
import re

# Convert the loaded dataset into a DataFrame
df_hi = legal_hi["train"].to_pandas()

# Keep Hindi entries that contain Devanagari text
hindi_pattern = re.compile(r'[\u0900-\u097F]')

df_hi["hindi_length"] = df_hi["hindi"].fillna("").str.len()

hindi_candidates = df_hi[
    df_hi["hindi"].fillna("").str.contains(hindi_pattern)
    & (df_hi["hindi_length"] >= 100)
].copy()

print("Total dataset rows:", len(df_hi))
print("Hindi entries at least 100 characters long:", len(hindi_candidates))

# Show 10 examples
for i, text in enumerate(hindi_candidates["hindi"].head(10), start=1):
    print(f"\n--- Example {i} ---\n{text}")

In [ ]:
import re
import pandas as pd

# Start with the longer Hindi entries
hi = hindi_candidates.copy()

# Remove empty text and obvious citation/name fragments
hi["hindi"] = hi["hindi"].fillna("").str.strip()
hi = hi[hi["hindi"].str.len() >= 100]

# Keep entries containing common legal terms in Hindi
legal_terms = (
    r"न्यायालय|अदालत|अधिनियम|धारा|याचिका|"
    r"प्रतिवादी|वादी|आदेश|अपील|कानून|अधिकार|"
    r"अनुबंध|समझौता|दंड|न्यायाधीश|साक्ष्य"
)

hi_legal = hi[
    hi["hindi"].str.contains(legal_terms, regex=True, na=False)
].copy()

# Remove duplicate passages
hi_legal = hi_legal.drop_duplicates(subset=["hindi"])

print("Long Hindi entries:", len(hi))
print("Entries matching legal terms:", len(hi_legal))

# Inspect 10 examples
for i, text in enumerate(hi_legal["hindi"].head(10), start=1):
    print(f"\n--- Legal passage {i} ---\n{text}")

In [ ]:
import re
import pandas as pd

legal_keywords = re.compile(
    r"न्यायालय|अदालत|अधिनियम|धारा|याचिका|"
    r"प्रतिवादी|वादी|आदेश|अपील|कानून|अधिकार|"
    r"अनुबंध|समझौता|दंड|न्यायाधीश|साक्ष्य"
)

candidates = []

for text in hi_legal["hindi"].dropna():
    # Split at Hindi/English sentence-ending punctuation
    sentences = re.split(r'(?<=[।!?])\s+', text.strip())

    for sentence in sentences:
        sentence = sentence.strip()

        if (
            len(sentence) >= 80
            and legal_keywords.search(sentence)
            and re.search(r'[।!?]$', sentence)
        ):
            candidates.append(sentence)

hi_sentences = pd.DataFrame(
    {"original_text": candidates}
).drop_duplicates()

print("Unique Hindi sentence candidates:", len(hi_sentences))

for i, sentence in enumerate(hi_sentences["original_text"].head(10), 1):
    print(f"\n--- Sentence {i} ---\n{sentence}")

In [ ]:
import pandas as pd
import re

clean_sentences = []

for text in hi_sentences["original_text"].dropna():
    text = re.sub(r"\s+", " ", text).strip()

    # Remove leading paragraph numbering such as "5." or "2.5."
    text = re.sub(r"^\d+(?:\.\d+)*\s*", "", text).strip()

    # Keep plausible sentence lengths
    if 80 <= len(text) <= 700:
        # Require Hindi text and a sentence-ending mark
        if re.search(r"[\u0900-\u097F]", text) and re.search(r"[।!?]$", text):
            clean_sentences.append(text)

hi_originals = pd.DataFrame({
    "pair_id": [f"hi_{i:05d}" for i in range(1, len(clean_sentences) + 1)],
    "original_text": clean_sentences,
    "simplified_text": "",
    "language": "hi",
    "lmp_score_1_to_10": "",
    "score_status": "not_yet_rated",
    "source_dataset": "Wasserstoff-AI/legalTransEn_Indic"
}).drop_duplicates(subset=["original_text"])

output_path = "/content/hindi_original_candidates_v1.csv"
hi_originals.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Hindi original candidates:", len(hi_originals))
display(hi_originals.head(10))

from google.colab import files
files.download(output_path)

In [ ]:
!pip -q install transformers sentencepiece accelerate

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the Hindi original candidates created in the previous step
input_path = "/content/hindi_original_candidates_v1.csv"
df = pd.read_csv(input_path)

# Use a multilingual model capable of Hindi text generation
model_name = "google/byt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Rows loaded:", len(df))
print("Model loaded:", model_name)

In [ ]:
# Test the model on just 5 Hindi sentences first
test_df = df.head(5).copy()

def generate_candidate(text):
    prompt = "simplify: " + str(text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=160,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

test_df["generated_simplification"] = test_df["original_text"].apply(
    generate_candidate
)

for _, row in test_df.iterrows():
    print("\nORIGINAL HINDI:")
    print(row["original_text"])
    print("\nGENERATED CANDIDATE:")
    print(row["generated_simplification"])
    print("-" * 70)

In [ ]:
!pip -q install huggingface_hub

from huggingface_hub import HfApi

api = HfApi()

queries = [
    "Hindi text simplification",
    "Indic text simplification",
    "Hindi paraphrase",
    "Indic paraphrase"
]

for query in queries:
    print(f"\n--- Search: {query} ---")
    results = api.list_models(
        search=query,
        limit=8,
        sort="downloads"
    )

    for model_info in results:
        print(model_info.id)

In [ ]:
!pip -q install transformers sentencepiece

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

candidate_name = "Abhi964/Paraphrase_IndicBERT_onfull_FT1"

try:
    candidate_tokenizer = AutoTokenizer.from_pretrained(candidate_name)
    candidate_model = AutoModelForSeq2SeqLM.from_pretrained(candidate_name)
    print("Model loaded successfully!")
except Exception as e:
    print("Could not load this model:")
    print(type(e).__name__, str(e)[:1000])

In [ ]:
from transformers import AutoConfig

candidate_name = "Abhi964/Paraphrase_IndicBERT_onfull_FT1"

config = AutoConfig.from_pretrained(candidate_name)

print("Model architecture:", config.architectures)
print("Model type:", config.model_type)
print("Configuration class:", type(config).__name__)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "ai4bharat/IndicBART"

try:
    hindi_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    hindi_model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    print("IndicBART loaded successfully!")
except Exception as e:
    print("Loading failed:", type(e).__name__, str(e)[:1000])

In [ ]:
import pandas as pd

# Reload the Hindi originals from the saved CSV
df = pd.read_csv("/content/hindi_original_candidates_v1.csv")

print("Hindi originals loaded:", len(df))
print("Columns:", df.columns.tolist())

# Select 3 original Hindi sentences
test_texts = df["original_text"].dropna().head(3).tolist()

hindi_tokenizer.src_lang = "hi_IN"
hindi_tokenizer.tgt_lang = "hi_IN"

for i, original in enumerate(test_texts, start=1):
    inputs = hindi_tokenizer(
        original,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    output_ids = hindi_model.generate(
        **inputs,
        max_length=160,
        num_beams=4
    )

    generated = hindi_tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True
    )[0]

    print(f"\n--- Example {i} ---")
    print("ORIGINAL HINDI:")
    print(original)
    print("\nGENERATED TEXT:")
    print(generated)
    print("-" * 60)

In [ ]:
import os

for filename in os.listdir("/content"):
    if "legal_simplification_pairs" in filename:
        print(filename)

In [ ]:
import pandas as pd

path = "/content/legal_simplification_pairs_en_v1.csv"
en_df = pd.read_csv(path)

print("Total rows:", len(en_df))
print("\nColumns:")
print(en_df.columns.tolist())

print("\nFirst 5 original/simplified pairs:")
display(en_df[["original_text", "simplified_text"]].head(5))

In [ ]:
import pandas as pd

# Check whether text and score fields are present
print("Missing original texts:", en_df["original_text"].isna().sum())
print("Missing simplified texts:", en_df["simplified_text"].isna().sum())

print("\nScore status counts:")
print(en_df["score_status"].value_counts(dropna=False))

print("\nExisting LMP scores:")
print(en_df["lmp_score_1_to_10"].value_counts(dropna=False).head(15))

# Display 10 complete pairs for manual review
display(en_df[["original_text", "simplified_text"]].sample(
    n=min(10, len(en_df)), random_state=42
))

In [ ]:
import re
import pandas as pd

legal_terms = re.compile(
    r"\blaw\b|\blegal\b|\bcourt\b|\bclaim\b|\binsurer\b|"
    r"\binsurance\b|\blandlord\b|\btenant\b|\blease\b|"
    r"\bcontract\b|\bagreement\b|\bliability\b|\bstatute\b|"
    r"\bshall\b|\bmust\b|\bparty\b|\bemployee\b|\bconfidentiality\b",
    re.IGNORECASE
)

# Keep rows where either text contains a legal-related term
mask = (
    en_df["original_text"].fillna("").str.contains(legal_terms)
    | en_df["simplified_text"].fillna("").str.contains(legal_terms)
)

legal_review = en_df.loc[mask].copy()

print("Total pairs:", len(en_df))
print("Pairs selected for legal review:", len(legal_review))

display(
    legal_review[
        ["pair_id", "original_text", "simplified_text"]
    ].head(10)
)

In [ ]:
# Show 5 pairs with the full text, not truncated table cells
for _, row in legal_review.head(5).iterrows():
    print("\nPAIR ID:", row["pair_id"])
    print("\nORIGINAL:")
    print(row["original_text"])
    print("\nSIMPLIFIED:")
    print(row["simplified_text"])
    print("\n" + "=" * 80)


In [ ]:
rubric = """
JUDGEBERT: LEGAL MEANING PRESERVATION RUBRIC

Purpose:
Compare an original legal text with its simplified version
in the SAME language. Do not compare translations.

Score 10:
All material legal meaning is preserved. Wording may be simpler.

Score 8-9:
Meaning is substantially preserved; only minor, non-material
wording differences exist.

Score 6-7:
Mostly preserved, but one potentially important detail is
unclear, weakened, or omitted.

Score 4-5:
A material condition, scope, party, exception, or obligation
is changed or omitted.

Score 2-3:
Major legal meaning is changed, reversed, or lost.

Score 1:
The simplified text contradicts the original or loses nearly
all of its material legal meaning.

Review these dimensions:
1. Parties and roles
2. Rights and obligations (may, must, shall)
3. Negation and exceptions
4. Conditions and scope
5. Amounts, dates, deadlines, and legal references
6. Material consequences

Important:
- A shorter sentence is not automatically better.
- Similar wording does not prove equivalent legal meaning.
- Do not assign a score when the evidence is insufficient.
- These scores require human review or a validated evaluation
  procedure; they must not be invented automatically.
"""

with open("/content/lmp_scoring_rubric_v1.txt", "w", encoding="utf-8") as f:
    f.write(rubric)

print(rubric)

from google.colab import files
files.download("/content/lmp_scoring_rubric_v1.txt")

In [ ]:
import pandas as pd
from google.colab import files

# Select the first 20 candidate pairs for review
review_df = legal_review.head(20).copy()

# Create blank fields for human assessment
review_df["review_score_1_to_10"] = ""
review_df["meaning_change_notes"] = ""
review_df["review_status"] = "pending_review"

# Keep only useful review columns
review_df = review_df[
    [
        "pair_id",
        "original_text",
        "simplified_text",
        "review_score_1_to_10",
        "meaning_change_notes",
        "review_status"
    ]
]

output_path = "/content/lmp_human_review_20_pairs.csv"
review_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Review sheet created:", len(review_df), "pairs")
display(review_df.head(5))

files.download(output_path)

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

filename = next(iter(uploaded))
review_df = pd.read_csv(filename)

print("Rows:", len(review_df))
print("Columns:", review_df.columns.tolist())
print("Average human score:",
      review_df["review_score_1_to_10"].mean())
print("Reviewed pairs:",
      (review_df["review_status"] == "reviewed").sum())

display(review_df.head(5))

In [ ]:
!pip -q install sentence-transformers scikit-learn

In [ ]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error

# Use the reviewed CSV loaded in the previous cell.
# If your variable has a different name, tell me the error.
df = review_df.copy()

# Keep rows with both texts and valid human scores.
df["review_score_1_to_10"] = pd.to_numeric(
    df["review_score_1_to_10"], errors="coerce"
)

df = df.dropna(
    subset=["original_text", "simplified_text", "review_score_1_to_10"]
).copy()

# Load a multilingual semantic-embedding model.
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

originals = df["original_text"].astype(str).tolist()
simplified = df["simplified_text"].astype(str).tolist()

original_embeddings = model.encode(
    originals, convert_to_numpy=True, show_progress_bar=True
)
simplified_embeddings = model.encode(
    simplified, convert_to_numpy=True, show_progress_bar=True
)

# Cosine similarity measures semantic closeness, not legal correctness.
similarities = np.diag(
    cosine_similarity(original_embeddings, simplified_embeddings)
)

# Convert similarity to a preliminary 1–10 score.
# This is a baseline mapping, not a calibrated legal score.
df["baseline_similarity"] = similarities
df["baseline_score_1_to_10"] = (
    1 + 9 * np.clip(similarities, 0, 1)
).round(2)

# Compare against human scores.
df["review_score_1_to_10"] = df["review_score_1_to_10"].astype(float)

mae = mean_absolute_error(
    df["review_score_1_to_10"],
    df["baseline_score_1_to_10"]
)

print("Pairs evaluated:", len(df))
print("Baseline mean absolute error:", round(mae, 2))
display(
    df[[
        "pair_id",
        "review_score_1_to_10",
        "baseline_similarity",
        "baseline_score_1_to_10"
    ]].head(10)
)


In [ ]:

output_columns = [
    "pair_id",
    "original_text",
    "simplified_text",
    "review_score_1_to_10",
    "meaning_change_notes",
    "baseline_similarity",
    "baseline_score_1_to_10"
]

results_df = df[output_columns].copy()

output_path = "/content/judgebert_baseline_results.csv"
results_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(results_df))

from google.colab import files
files.download(output_path)


In [ ]:

import pandas as pd

diagnostic_pairs = [
    {
        "category": "Negation",
        "original_text": "The tenant must pay the rent.",
        "simplified_text": "The tenant must not pay the rent."
    },
    {
        "category": "Obligation",
        "original_text": "The tenant must notify the landlord.",
        "simplified_text": "The tenant may notify the landlord."
    },
    {
        "category": "Deadline",
        "original_text": "The appeal must be filed within 30 days.",
        "simplified_text": "The appeal must be filed within 60 days."
    },
    {
        "category": "Amount",
        "original_text": "The penalty shall not exceed 5000 rupees.",
        "simplified_text": "The penalty shall not exceed 500 rupees."
    },
    {
        "category": "Meaning preserved",
        "original_text": "The landlord may enter the property after giving reasonable notice.",
        "simplified_text": "The landlord may enter the property after providing reasonable notice."
    }
]

test_df = pd.DataFrame(diagnostic_pairs)

original_embeddings = model.encode(
    test_df["original_text"].tolist(),
    convert_to_numpy=True
)

simplified_embeddings = model.encode(
    test_df["simplified_text"].tolist(),
    convert_to_numpy=True
)

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

test_df["similarity"] = np.diag(
    cosine_similarity(original_embeddings, simplified_embeddings)
)

test_df["baseline_score_1_to_10"] = (
    1 + 9 * np.clip(test_df["similarity"], 0, 1)
).round(2)

display(test_df)


In [ ]:

import re
import pandas as pd

def extract_legal_markers(text):
    text = str(text).lower()

    # Preserve important legal wording and numeric values.
    markers = {
        "negation": bool(re.search(
            r"\b(not|no|never|neither|unless|except)\b", text
        )),
        "must": bool(re.search(r"\bmust\b", text)),
        "may": bool(re.search(r"\bmay\b", text)),
        "shall": bool(re.search(r"\bshall\b", text)),
        "should": bool(re.search(r"\bshould\b", text)),
        "numbers": re.findall(r"\b\d+(?:,\d{3})*(?:\.\d+)?\b", text),
        "time_terms": re.findall(
            r"\b(days?|weeks?|months?|years?|hours?|deadline|within|before|after)\b",
            text
        )
    }
    return markers

def compare_legal_markers(original, simplified):
    a = extract_legal_markers(original)
    b = extract_legal_markers(simplified)

    flags = []

    for key in ["negation", "must", "may", "shall", "should"]:
        if a[key] != b[key]:
            flags.append(f"Possible change in {key}")

    if a["numbers"] != b["numbers"]:
        flags.append("Numeric value changed or removed")

    if a["time_terms"] != b["time_terms"]:
        flags.append("Possible deadline or time-condition change")

    return "; ".join(flags) if flags else "No marker change detected"

test_df["legal_risk_flags"] = test_df.apply(
    lambda row: compare_legal_markers(
        row["original_text"], row["simplified_text"]
    ),
    axis=1
)

display(test_df[
    ["category", "baseline_score_1_to_10", "legal_risk_flags"]
])


In [ ]:

def calculate_lmp_score(row):
    score = float(row["baseline_score_1_to_10"])
    flags = row["legal_risk_flags"]

    # Preliminary heuristic penalties, not learned model weights.
    if flags != "No marker change detected":
        if "negation" in flags:
            score -= 4
        if "must" in flags or "may" in flags or "shall" in flags or "should" in flags:
            score -= 3
        if "Numeric value" in flags:
            score -= 3
        if "deadline" in flags or "time-condition" in flags:
            score -= 2

    return round(max(1, min(10, score)), 2)

test_df["preliminary_lmp_score_1_to_10"] = test_df.apply(
    calculate_lmp_score, axis=1
)

display(test_df[
    [
        "category",
        "baseline_score_1_to_10",
        "legal_risk_flags",
        "preliminary_lmp_score_1_to_10"
    ]
])


In [ ]:

# Save the test results to a CSV file
output_path = "/content/judgebert_lmp_test_results.csv"

test_df.to_csv(output_path, index=False)

print("Results saved to:", output_path)
print("Total test cases:", len(test_df))

from google.colab import files
files.download(output_path)


In [ ]:

import pandas as pd
import os

# Find the reviewed file in the Colab session
review_path = "/content/lmp_human_review_20_pairs.csv"

if not os.path.exists(review_path):
    print("File not found:", review_path)
    print("Files currently available:")
    print([f for f in os.listdir("/content") if f.endswith(".csv")])
else:
    review_df = pd.read_csv(review_path)

    print("Reviewed pairs:", len(review_df))
    print("Columns:", review_df.columns.tolist())
    display(review_df.head(5))


In [ ]:

# Check whether the human scores are numeric and available
print("Human score column:")
print(review_df["review_score_1_to_10"].value_counts(dropna=False))

print("\nMissing human scores:",
      review_df["review_score_1_to_10"].isna().sum())

print("\nReview notes sample:")
display(review_df[
    ["pair_id", "review_score_1_to_10",
     "meaning_change_notes", "review_status"]
].head(20))


In [ ]:

import os
import pandas as pd

for f in os.listdir("/content"):
    if "lmp_human_review_20_pairs" in f and f.endswith(".csv"):
        print("\nFile:", f)
        temp = pd.read_csv("/content/" + f)
        print("Rows:", len(temp))
        print("Columns:", temp.columns.tolist())
        print("Numeric scores:",
              pd.to_numeric(temp["review_score_1_to_10"], errors="coerce").notna().sum())
        display(temp[["pair_id", "review_score_1_to_10",
                      "meaning_change_notes"]].head(5))


In [ ]:

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

# Load the correct human-reviewed file
review_df = pd.read_csv("/content/lmp_human_review_20_pairs.csv")

# Load the baseline results generated earlier
baseline_df = pd.read_csv("/content/judgebert_baseline_results.csv")

print("Human review columns:", review_df.columns.tolist())
print("Baseline columns:", baseline_df.columns.tolist())

display(baseline_df.head())


In [ ]:

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

# Merge human scores and baseline predictions by pair ID
evaluation_df = review_df[
    ["pair_id", "review_score_1_to_10"]
].merge(
    baseline_df[["pair_id", "baseline_score_1_to_10"]],
    on="pair_id",
    how="inner"
)

# Ensure scores are numeric and remove missing values
evaluation_df["review_score_1_to_10"] = pd.to_numeric(
    evaluation_df["review_score_1_to_10"], errors="coerce"
)
evaluation_df["baseline_score_1_to_10"] = pd.to_numeric(
    evaluation_df["baseline_score_1_to_10"], errors="coerce"
)
evaluation_df = evaluation_df.dropna()

y_true = evaluation_df["review_score_1_to_10"]
y_pred = evaluation_df["baseline_score_1_to_10"]

print("Matched pairs:", len(evaluation_df))
print("Mean Absolute Error (MAE):",
      round(mean_absolute_error(y_true, y_pred), 3))
print("Root Mean Squared Error (RMSE):",
      round(np.sqrt(mean_squared_error(y_true, y_pred)), 3))

if y_true.nunique() > 1 and y_pred.nunique() > 1:
    print("Pearson correlation:",
          round(pearsonr(y_true, y_pred).statistic, 3))
    print("Spearman correlation:",
          round(spearmanr(y_true, y_pred).statistic, 3))
else:
    print("Correlation unavailable: one score column has no variation.")

display(evaluation_df)


In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
import numpy as np

y_true = evaluation_df["review_score_1_to_10"].astype(float)
y_pred = evaluation_df["baseline_score_1_to_10"].astype(float)

print("EVALUATION SUMMARY")
print("------------------")
print("Matched pairs:", len(evaluation_df))
print("MAE:", round(mean_absolute_error(y_true, y_pred), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y_true, y_pred)), 3))

print("Pearson correlation:",
      round(pearsonr(y_true, y_pred).statistic, 3))
print("Spearman correlation:",
      round(spearmanr(y_true, y_pred).statistic, 3))


In [ ]:

# Save the evaluated pairs and predictions
evaluation_df.to_csv(
    "/content/judgebert_lmp_evaluation_v1.csv",
    index=False
)

print("Saved evaluation results.")
print("Rows:", len(evaluation_df))
print("File: /content/judgebert_lmp_evaluation_v1.csv")

from google.colab import files
files.download("/content/judgebert_lmp_evaluation_v1.csv")


In [ ]:

import pandas as pd
import os

path = "/content/judgebert_lmp_evaluation_v1.csv"

if os.path.exists(path):
    df = pd.read_csv(path)
    print("File found!")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    display(df.head(5))
else:
    print("File not found in this Colab session.")
    print("Please upload judgebert_lmp_evaluation_v1.csv")


In [ ]:

import pandas as pd
import os

path = "/content/lmp_human_review_20_pairs.csv"

if os.path.exists(path):
    df = pd.read_csv(path)

    print("Total reviewed pairs:", len(df))
    print("Human scores present:",
          pd.to_numeric(
              df["review_score_1_to_10"],
              errors="coerce"
          ).notna().sum())

    print("\nScore distribution:")
    print(
        pd.to_numeric(
            df["review_score_1_to_10"],
            errors="coerce"
        ).value_counts().sort_index()
    )

    print("\nReview status:")
    print(df["review_status"].value_counts(dropna=False))

else:
    print("File not found. Check the Colab Files panel.")


In [ ]:

import pandas as pd
import os

print("=== DATASET INVENTORY ===")

# Existing human-reviewed pairs
review_path = "/content/lmp_human_review_20_pairs.csv"

if os.path.exists(review_path):
    reviewed = pd.read_csv(review_path)
    reviewed["review_score_1_to_10"] = pd.to_numeric(
        reviewed["review_score_1_to_10"], errors="coerce"
    )
    reviewed = reviewed.dropna(
        subset=["original_text", "simplified_text", "review_score_1_to_10"]
    )
    print("Reviewed pairs with scores:", len(reviewed))
else:
    print("Reviewed-pairs file not found.")

# English simplification candidates
candidate_paths = [
    "/content/legal_simplification_pairs_en_filtered.csv",
    "/content/legal_simplification_pairs_en_v1.csv",
]

for path in candidate_paths:
    if os.path.exists(path):
        candidates = pd.read_csv(path)
        print("\nEnglish candidate file:", os.path.basename(path))
        print("Candidate rows:", len(candidates))
        print("Columns:", candidates.columns.tolist())
        break
else:
    print("\nNo English candidate file found.")

# Hindi candidates: inspect only; do not count them as Hindi pairs yet
hindi_path = "/content/hindi_original_candidates_v1.csv"

if os.path.exists(hindi_path):
    hindi = pd.read_csv(hindi_path)
    print("\nHindi source candidates:", len(hindi))
    print("Hindi columns:", hindi.columns.tolist())
    print("These count as LMP pairs only after a valid Hindi simplification exists.")


In [ ]:

import pandas as pd

hindi_path = "/content/hindi_original_candidates_v1.csv"
hindi = pd.read_csv(hindi_path)

print("Rows:", len(hindi))
print("\nLanguage counts:")
print(hindi["language"].value_counts(dropna=False))

print("\nScore status counts:")
print(hindi["score_status"].value_counts(dropna=False))

print("\nFirst 10 Hindi candidate pairs:")
display(
    hindi[["pair_id", "original_text", "simplified_text",
           "language", "lmp_score_1_to_10"]].head(10)
)


In [ ]:

import pandas as pd

hindi = pd.read_csv("/content/hindi_original_candidates_v1.csv")

# Keep only non-empty Hindi originals
hindi = hindi[
    hindi["original_text"].notna()
    & (hindi["original_text"].astype(str).str.strip() != "")
].copy()

# Select a small pilot for inspection
pilot = hindi.head(10).copy()

pilot["simplified_text"] = ""
pilot["lmp_score_1_to_10"] = pd.NA
pilot["score_status"] = "pending_hindi_simplification"
pilot["review_status"] = "pending"

output_path = "/content/hindi_simplification_pilot_v1.csv"
pilot.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Hindi originals selected:", len(pilot))
print("Simplified texts still blank:", pilot["simplified_text"].eq("").sum())
print("Saved:", output_path)

display(pilot[["pair_id", "original_text", "simplified_text",
               "lmp_score_1_to_10", "review_status"]])


In [ ]:

import pandas as pd

hindi = pd.read_csv("/content/hindi_original_candidates_v1.csv")

for i, row in hindi.head(5).iterrows():
    print(f"\n{'=' * 70}")
    print("PAIR ID:", row["pair_id"])
    print("ORIGINAL HINDI TEXT:")
    print(str(row["original_text"]))


In [ ]:

import pandas as pd
import re

input_path = "/content/hindi_original_candidates_v1.csv"
output_path = "/content/hindi_source_pilot_300.csv"

df = pd.read_csv(input_path)

# Keep non-empty Hindi text with a minimum length.
df["original_text"] = df["original_text"].fillna("").astype(str).str.strip()

selected = df[
    (df["original_text"].str.len() >= 100) &
    (df["original_text"].str.contains(r"[\u0900-\u097F]", regex=True))
].copy()

# Remove exact duplicate texts.
selected = selected.drop_duplicates(subset=["original_text"])

# Keep at most 300 source texts.
selected = selected.head(300).copy()

# Prepare fields for later simplification and human review.
selected["simplified_text"] = ""
selected["lmp_score_1_to_10"] = pd.NA
selected["score_status"] = "not_yet_reviewed"

selected.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Eligible Hindi source texts:", len(selected))
print("Saved to:", output_path)
print("Simplified texts completed:", (selected["simplified_text"] != "").sum())
print("Human scores present:", selected["lmp_score_1_to_10"].notna().sum())

display(selected[["pair_id", "original_text"]].head(10))


In [ ]:

import os
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from google.colab import files

# ---------- 1. Load and validate the Hindi source texts ----------
source_path = "/content/hindi_source_pilot_300.csv"
english_path = "/content/legal_simplification_pairs_en_v1.csv"
review_path = "/content/lmp_human_review_20_pairs.csv"

assert os.path.exists(source_path), (
    f"Missing {source_path}. Run the previous Hindi selection cell first."
)

hi = pd.read_csv(source_path).fillna("")
hi["original_text"] = hi["original_text"].astype(str).str.strip()
hi = hi[hi["original_text"] != ""].drop_duplicates(
    subset=["original_text"]
).copy()

# Keep the expected schema consistent.
for col, default in {
    "pair_id": "",
    "simplified_text": "",
    "lmp_score_1_to_10": "",
    "score_status": "not_yet_reviewed",
}.items():
    if col not in hi.columns:
        hi[col] = default

hi["language"] = "hi"
hi["simplified_text"] = ""
hi["lmp_score_1_to_10"] = ""
hi["score_status"] = "not_yet_reviewed"
hi["meaning_change_notes"] = ""

# ---------- 2. Load English candidates and human-reviewed pilot ----------
en_count = None
en_scored_count = None
review_count = None
review_scored_count = None

if os.path.exists(english_path):
    en = pd.read_csv(english_path)
    en_count = len(en)
    if "lmp_score_1_to_10" in en.columns:
        en_scored_count = pd.to_numeric(
            en["lmp_score_1_to_10"], errors="coerce"
        ).notna().sum()

if os.path.exists(review_path):
    review = pd.read_csv(review_path)
    review_count = len(review)
    if "review_score_1_to_10" in review.columns:
        review_scored_count = pd.to_numeric(
            review["review_score_1_to_10"], errors="coerce"
        ).notna().sum()

# ---------- 3. Define annotation rubric ----------
rubric = pd.DataFrame([
    [10, "Meaning fully preserved; wording may be simpler."],
    [8,  "Main legal meaning preserved; only minor nuance may differ."],
    [6,  "Some potentially important legal detail is weakened or unclear."],
    [4,  "A material obligation, right, exception, party, or condition changes."],
    [2,  "Major legal meaning is lost or changed."],
    [1,  "Meaning is contradicted or almost entirely lost."],
], columns=["Score", "Guidance"])

# ---------- 4. Create workbook ----------
output_path = "/content/JUDGEBERT_LMP_annotation_workbook.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    hi.to_excel(writer, sheet_name="Hindi_Source_300", index=False)
    hi.head(50).to_excel(writer, sheet_name="Hindi_Pilot_50", index=False)
    rubric.to_excel(writer, sheet_name="Scoring_Rubric", index=False)

    status = pd.DataFrame([
        ["Hindi source texts prepared", len(hi)],
        ["Hindi simplified versions completed", 0],
        ["Hindi human-rated pairs", 0],
        ["English candidate pairs", en_count if en_count is not None else "File missing"],
        ["English candidate rows with scores",
         en_scored_count if en_scored_count is not None else "Not available"],
        ["English reviewed pilot rows",
         review_count if review_count is not None else "File missing"],
        ["English reviewed pilot rows with numeric scores",
         review_scored_count if review_scored_count is not None else "Not available"],
    ], columns=["Metric", "Value"])
    status.to_excel(writer, sheet_name="Project_Status", index=False)

    # Make sheets easier to read in Excel.
    for ws in writer.book.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor="23476A")
            cell.alignment = Alignment(wrap_text=True, vertical="top")
        for col_cells in ws.columns:
            col_letter = get_column_letter(col_cells[0].column)
            header = str(col_cells[0].value or "")
            if header in ("original_text", "simplified_text", "meaning_change_notes", "Guidance"):
                ws.column_dimensions[col_letter].width = 65
            else:
                ws.column_dimensions[col_letter].width = min(
                    max(max(len(str(c.value or "")) for c in col_cells[:30]) + 2, 14),
                    32
                )
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

print("WORKBOOK CREATED:", output_path)
print("Hindi source texts:", len(hi))
print("Hindi simplifications: 0 (intentionally blank)")
print("Hindi human scores: 0 (intentionally blank)")
print("\nProject status:")
display(status)

files.download(output_path)


In [ ]:

import pandas as pd
from google.colab import files

source_path = "/content/hindi_source_pilot_300.csv"
df = pd.read_csv(source_path).fillna("")

# Select 10 source texts for a small, manageable pilot.
pilot = df[df["original_text"].astype(str).str.strip() != ""].head(10).copy()

pilot["language"] = "hi"
pilot["simplified_text"] = ""
pilot["lmp_score_1_to_10"] = ""
pilot["meaning_change_notes"] = ""
pilot["review_status"] = "pending"

pilot = pilot[
    [
        "pair_id",
        "language",
        "original_text",
        "simplified_text",
        "lmp_score_1_to_10",
        "meaning_change_notes",
        "review_status",
    ]
]

output_path = "/content/JUDGEBERT_Hindi_simplification_pilot_10.csv"
pilot.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Hindi pilot rows:", len(pilot))
print("Simplified texts completed:", pilot["simplified_text"].ne("").sum())
print("Scores assigned:", pilot["lmp_score_1_to_10"].ne("").sum())
display(pilot[["pair_id", "original_text", "simplified_text"]])

files.download(output_path)


In [ ]:

import pandas as pd
from google.colab import files

input_path = "/content/JUDGEBERT_Hindi_simplification_pilot_10.csv"
df = pd.read_csv(input_path).fillna("")

# Draft simplifications for readable source passages only.
# These are preliminary drafts and need legal review.
hindi_drafts = {
    "hi_00002": (
        "इस समझौते पर बोर्ड की ओर से प्रशासक और सरकार की ओर से "
        "उत्तर प्रदेश सरकार के अपर सचिव ने हस्ताक्षर किए हैं।"
    ),
    "hi_00003": (
        "जांच पूरी होने के बाद, अपीलकर्ता सहित चार आरोपियों के खिलाफ "
        "आईपीसी की धारा 304बी के तहत दहेज मृत्यु के अपराध का आरोपपत्र "
        "दाखिल किया गया।"
    ),
    "hi_00004": (
        "स्पष्टीकरण: इस धारा में “कुरुता” का अर्थ वही होगा जो "
        "भारतीय दंड संहिता की धारा 498ए में दिया गया है।"
    ),
    "hi_00005": (
        "इसलिए, आदेश जारी किया गया कि सुनीता देवी को ऊपर बताए गए "
        "अपराध के मुकदमे में पेश होने के लिए बुलाया जाए।"
    ),
}

for i, row in df.iterrows():
    pair_id = str(row.get("pair_id", "")).strip()
    original = str(row.get("original_text", "")).strip()

    if pair_id in hindi_drafts and original:
        df.at[i, "simplified_text"] = hindi_drafts[pair_id]
        df.at[i, "review_status"] = "draft_needs_legal_review"
        df.at[i, "meaning_change_notes"] = (
            "Preliminary simplification; verify legal meaning and references."
        )
    else:
        df.at[i, "review_status"] = "needs_simplification"

# Never invent human scores.
df["lmp_score_1_to_10"] = ""

output_path = "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Pilot rows:", len(df))
print("Draft simplifications filled:", df["simplified_text"].astype(str).str.strip().ne("").sum())
print("Human scores assigned: 0")
display(df[["pair_id", "original_text", "simplified_text", "review_status"]])

files.download(output_path)


In [ ]:

import pandas as pd

path = "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts.csv"
df = pd.read_csv(path).fillna("")

remaining = df[df["simplified_text"].astype(str).str.strip() == ""]

print("Remaining rows:", len(remaining))
for _, row in remaining.iterrows():
    print("\n" + "=" * 70)
    print("PAIR ID:", row["pair_id"])
    print("ORIGINAL HINDI:")
    print(row["original_text"])


In [ ]:

import pandas as pd
from google.colab import files

input_path = "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts.csv"
df = pd.read_csv(input_path).fillna("")

hindi_drafts = {
    "hi_00006": (
        "अपीलकर्ता की ओर से उपस्थित वरिष्ठ अधिवक्ता श्री नवीन पाहवा "
        "और प्रत्यर्थी संख्या 2 की ओर से उपस्थित वरिष्ठ अधिवक्ता "
        "श्री शैलेश शर्मा को सुना गया।"
    ),
    "hi_00007": (
        "अब इस विषय पर कानून स्पष्ट है। न्यायालय के कई निर्णयों से "
        "यह बात अच्छी तरह स्थापित हो चुकी है।"
    ),
    "hi_00008": (
        "क्या संहिता की धारा 200 के तहत चल रही कार्यवाही को, "
        "2002 में संशोधन द्वारा जोड़ी गई एनआई अधिनियम की धारा 45 "
        "के आधार पर समाप्त किया जा सकता है?"
    ),
    "hi_00009": (
        "शिकायतकर्ता को अदालत में बुलाना और शपथ पर उसकी जांच करना "
        "अनिवार्य नहीं है, यदि यह तय करना हो कि कार्यवाही आगे "
        "जारी रखनी है या नहीं।"
    ),
    "hi_00011": (
        "उप-धारा (3) के अनुसार, इस प्रावधान के अंतर्गत आने वाले "
        "मामलों में बोर्ड का निर्णय अंतिम होगा।"
    ),
}

for i, row in df.iterrows():
    pair_id = str(row.get("pair_id", "")).strip()
    original = str(row.get("original_text", "")).strip()

    if pair_id in hindi_drafts and original:
        df.at[i, "simplified_text"] = hindi_drafts[pair_id]
        df.at[i, "review_status"] = "draft_needs_legal_review"
        df.at[i, "meaning_change_notes"] = (
            "Draft only; compare against the complete original, "
            "especially legal references and procedural conditions."
        )

# hi_000010 remains blank because its source was truncated in the screenshot.
df["lmp_score_1_to_10"] = ""

output_path = "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts_v2.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Total pilot rows:", len(df))
print(
    "Drafts filled:",
    df["simplified_text"].astype(str).str.strip().ne("").sum()
)
print("Scores assigned: 0")
print("\nRows still needing simplification:")
display(
    df.loc[
        df["simplified_text"].astype(str).str.strip().eq(""),
        ["pair_id", "original_text", "review_status"]
    ]
)

files.download(output_path)


In [ ]:
# Show the complete original text for the missing Hindi row
import pandas as pd

path = "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts_v2.csv"
df = pd.read_csv(path).fillna("")

row = df[df["pair_id"].astype(str) == "hi_000010"]

if len(row):
    print("PAIR ID:", row.iloc[0]["pair_id"])
    print("FULL ORIGINAL HINDI TEXT:\n")
    print(row.iloc[0]["original_text"])
else:
    print("hi_000010 was not found. Check the CSV filename.")

In [ ]:
import os
import glob
import pandas as pd

# Find available Hindi pilot CSV files automatically
files = glob.glob("/content/*Hindi*simplification*pilot*.csv")
files += glob.glob("/content/*hindi*simplification*pilot*.csv")
files = list(dict.fromkeys(files))

print("Candidate files found:")
for f in files:
    print("-", os.path.basename(f))

found = False

for path in files:
    try:
        df = pd.read_csv(path).fillna("")
        if "pair_id" not in df.columns or "original_text" not in df.columns:
            continue

        missing = df[df["pair_id"].astype(str).str.strip() == "hi_000010"]

        if not missing.empty:
            print("\nUsing file:", os.path.basename(path))
            print("Columns:", df.columns.tolist())
            print("\nPAIR ID:", missing.iloc[0]["pair_id"])
            print("\nFULL ORIGINAL HINDI TEXT:\n")
            print(missing.iloc[0]["original_text"])
            print("\nCurrent simplified text:")
            print(missing.iloc[0].get("simplified_text", ""))
            found = True
            break
    except Exception as e:
        print("Could not read", os.path.basename(path), ":", e)

if not found:
    print("\nThe row was not found in the candidate files.")
    print("CSV files currently available:")
    for f in glob.glob("/content/*.csv"):
        print("-", os.path.basename(f))

In [ ]:
import pandas as pd
import glob
import os

files = glob.glob("/content/*Hindi*simplification*pilot*.csv")
files += glob.glob("/content/hindi_simplification_pilot_v1.csv")
files = list(dict.fromkeys(files))

for path in files:
    try:
        df = pd.read_csv(path).fillna("")
        print("\n" + "=" * 75)
        print("FILE:", os.path.basename(path))
        print("ROWS:", len(df))
        print("COLUMNS:", df.columns.tolist())

        if "pair_id" in df.columns:
            print("PAIR IDs:", df["pair_id"].astype(str).head(15).tolist())

        if "review_status" in df.columns:
            pending = df[df["review_status"].astype(str).str.contains(
                "needs_simplification", case=False, na=False
            )]
            if len(pending):
                print("\nROWS STILL NEEDING SIMPLIFICATION:")
                for _, row in pending.iterrows():
                    print("ID:", row.get("pair_id", ""))
                    print("ORIGINAL:", row.get("original_text", ""))
                    print("CURRENT DRAFT:", row.get("simplified_text", ""))

    except Exception as e:
        print("Could not read file:", e)

In [ ]:
import pandas as pd
import os
from google.colab import files

source = "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts_v2.csv"
output = "/content/JUDGEBERT_Hindi_simplification_pilot_10_completed.csv"

df = pd.read_csv(source).fillna("")

# Fill only the two rows shown as blank in the v2 screenshot.
drafts = {
    "hi_000010": (
        "इस शक्ति का उपयोग संविधान के अनुच्छेद 226 या धारा 482, "
        "दंड प्रक्रिया संहिता के तहत किया जा सकता है। इसका उद्देश्य "
        "न्यायालय की प्रक्रिया के दुरुपयोग को रोकना या न्याय सुनिश्चित करना है।"
    ),
    "hi_000011": (
        "उप-धारा (3) कहती है कि इस प्रावधान के अंतर्गत आने वाले मामलों "
        "में बोर्ड का निर्णय अंतिम होगा।"
    )
}

for pair_id, draft in drafts.items():
    mask = df["pair_id"].astype(str).str.strip().eq(pair_id)
    if mask.any() and df.loc[mask, "simplified_text"].astype(str).str.strip().eq("").any():
        df.loc[mask, "simplified_text"] = draft
        if "review_status" in df.columns:
            df.loc[mask, "review_status"] = "draft_needs_human_review"

# Do not invent human scores.
if "lmp_score_1_to_10" in df.columns:
    df["lmp_score_1_to_10"] = df["lmp_score_1_to_10"].replace("", pd.NA)

df.to_csv(output, index=False, encoding="utf-8-sig")

print("Saved:", output)
print("Total rows:", len(df))
print("Non-empty simplified texts:",
      df["simplified_text"].astype(str).str.strip().ne("").sum())
print("Human scores present:",
      pd.to_numeric(df["lmp_score_1_to_10"], errors="coerce").notna().sum()
      if "lmp_score_1_to_10" in df.columns else 0)
print("\nReview the drafts before treating them as validated pairs:")
display(df[["pair_id", "original_text", "simplified_text"]])

files.download(output)

In [ ]:
import pandas as pd
from google.colab import files

source = "/content/JUDGEBERT_Hindi_simplification_pilot_10_completed.csv"
df = pd.read_csv(source).fillna("")

# Check that the expected columns exist
required = ["pair_id", "original_text", "simplified_text"]
missing_cols = [c for c in required if c not in df.columns]

if missing_cols:
    print("Missing required columns:", missing_cols)
else:
    df["original_text"] = df["original_text"].astype(str).str.strip()
    df["simplified_text"] = df["simplified_text"].astype(str).str.strip()

    df["pair_complete"] = (
        df["original_text"].ne("") & df["simplified_text"].ne("")
    )

    print("=== HINDI PILOT AUDIT ===")
    print("Total rows:", len(df))
    print("Complete text pairs:", int(df["pair_complete"].sum()))
    print("Incomplete pairs:", int((~df["pair_complete"]).sum()))

    if "lmp_score_1_to_10" in df.columns:
        scores = pd.to_numeric(df["lmp_score_1_to_10"], errors="coerce")
        print("Human scores assigned:", int(scores.notna().sum()))

    print("\nPairs needing attention:")
    display(df.loc[~df["pair_complete"],
                   ["pair_id", "original_text", "simplified_text"]])

    # Save audit copy without inventing scores
    audit_path = "/content/JUDGEBERT_Hindi_pilot_audit.csv"
    df.to_csv(audit_path, index=False, encoding="utf-8-sig")
    print("\nAudit saved:", audit_path)
    files.download(audit_path)

In [ ]:

import pandas as pd
from transformers import pipeline
from IPython.display import display

pilot_path = "/content/JUDGEBERT_Hindi_simplification_pilot_10_completed.csv"
source_path = "/content/hindi_original_candidates_v1.csv"

pilot = pd.read_csv(pilot_path).fillna("")
source = pd.read_csv(source_path).fillna("")

pair_id = "hi_000010"

# Find the incomplete row
mask = pilot["pair_id"].astype(str).eq(pair_id)
if not mask.any():
    raise ValueError(f"{pair_id} not found in the pilot file.")

# Recover the original text from the source file if necessary
original = str(pilot.loc[mask, "original_text"].iloc[0]).strip()

if not original or len(original) < 30:
    source_row = source[source["pair_id"].astype(str).eq(pair_id)]
    if source_row.empty:
        raise ValueError("Could not recover the original text from the source CSV.")
    original = str(source_row["original_text"].iloc[0]).strip()

print("Original Hindi text:\n", original)

# Load a multilingual text-to-text model
generator = pipeline(
    "text2text-generation",
    model="google/mt5-small"
)

prompt = (
    "सरल और स्पष्ट हिंदी में लिखें। "
    "कानूनी अर्थ, शर्तें, अपवाद, धाराओं के नंबर और कानूनी प्रभाव न बदलें। "
    "कुछ भी न जोड़ें या हटाएँ। केवल सरल हिंदी में पुनर्लेखन दें।\n\n"
    + original
)

result = generator(
    prompt,
    max_new_tokens=256,
    do_sample=False
)[0]["generated_text"]

pilot.loc[mask, "original_text"] = original
pilot.loc[mask, "simplified_text"] = result.strip()

# Keep the score blank until human review
if "lmp_score_1_to_10" in pilot.columns:
    pilot.loc[mask, "lmp_score_1_to_10"] = ""

if "review_status" in pilot.columns:
    pilot.loc[mask, "review_status"] = "needs_human_legal_review"

output = "/content/JUDGEBERT_Hindi_simplification_pilot_10_repaired.csv"
pilot.to_csv(output, index=False, encoding="utf-8-sig")

print("\nDraft created. Human legal review is still required.")
display(pilot.loc[mask, ["pair_id", "original_text", "simplified_text"]])
print("\nSaved:", output)


In [ ]:

import pandas as pd
import glob
import os
from IPython.display import display

files_to_check = [
    "/content/JUDGEBERT_Hindi_simplification_pilot_10_completed.csv",
    "/content/JUDGEBERT_Hindi_simplification_pilot_10.csv",
    "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts_v2.csv",
    "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts.csv",
    "/content/hindi_simplification_pilot_v1.csv",
    "/content/hindi_original_candidates_v1.csv",
]

found = False

for path in files_to_check:
    if not os.path.exists(path):
        continue

    df = pd.read_csv(path, dtype=str).fillna("")
    if not {"pair_id", "original_text", "simplified_text"}.issubset(df.columns):
        continue

    incomplete = df[
        df["original_text"].str.strip().eq("")
        | df["simplified_text"].str.strip().eq("")
    ]

    print(f"\nFILE: {os.path.basename(path)} | Rows: {len(df)}")
    print("Incomplete pairs:", len(incomplete))

    if len(incomplete):
        display(incomplete[["pair_id", "original_text", "simplified_text"]])
        found = True

if not found:
    print("No incomplete rows found in the checked files.")
    print("CSV files available:")
    print([os.path.basename(p) for p in glob.glob("/content/*.csv")])


In [ ]:

import pandas as pd
import os
from IPython.display import display

# Find the most recent Hindi pilot draft file
candidates = [
    "/content/JUDGEBERT_Hindi_simplification_pilot_10_completed.csv",
    "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts_v2.csv",
    "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts.csv",
    "/content/JUDGEBERT_Hindi_simplification_pilot_10.csv",
    "/content/hindi_simplification_pilot_v1.csv",
]

pilot_path = next((p for p in candidates if os.path.exists(p)), None)

if pilot_path is None:
    raise FileNotFoundError("No Hindi pilot file found.")

pilot = pd.read_csv(pilot_path, dtype=str).fillna("")
print("Using:", os.path.basename(pilot_path))
print("Rows:", len(pilot))
print("Columns:", pilot.columns.tolist())

# Display incomplete rows so we can identify the correct row and source ID
incomplete_mask = (
    pilot["original_text"].str.strip().eq("")
    | pilot["simplified_text"].str.strip().eq("")
)

print("Incomplete rows:", int(incomplete_mask.sum()))
display(pilot.loc[incomplete_mask, [
    c for c in ["pair_id", "original_text", "simplified_text", "review_status"]
    if c in pilot.columns
]])

# Save a clean working copy without inventing text or scores
output_path = "/content/JUDGEBERT_Hindi_pilot_repair_working_copy.csv"
pilot.to_csv(output_path, index=False, encoding="utf-8-sig")
print("\nSaved working copy:", output_path)
print("No scores or missing simplifications were invented.")


In [ ]:

import pandas as pd
from IPython.display import display

pilot_path = "/content/JUDGEBERT_Hindi_pilot_repair_working_copy.csv"
source_path = "/content/hindi_original_candidates_v1.csv"

pilot = pd.read_csv(pilot_path, dtype=str).fillna("")
source = pd.read_csv(source_path, dtype=str).fillna("")

pair_id = "hi_00010"

pilot_match = pilot["pair_id"].astype(str).str.strip() == pair_id
source_match = source["pair_id"].astype(str).str.strip() == pair_id

print("Pilot rows matching ID:", int(pilot_match.sum()))
print("Source rows matching ID:", int(source_match.sum()))

if pilot_match.sum() != 1:
    print("Could not identify exactly one pilot row. No changes made.")
elif source_match.sum() != 1:
    print("Could not identify exactly one matching source row. No changes made.")
else:
    original = source.loc[source_match, "original_text"].iloc[0]
    print("\nFULL ORIGINAL HINDI TEXT:\n")
    print(original)

    # Restore the original only; do not invent a simplification or score.
    pilot.loc[pilot_match, "original_text"] = original
    pilot.loc[pilot_match, "simplified_text"] = ""
    pilot.loc[pilot_match, "lmp_score_1_to_10"] = ""
    pilot.loc[pilot_match, "review_status"] = "needs_simplification"

    output_path = "/content/JUDGEBERT_Hindi_pilot_recovered_v1.csv"
    pilot.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("\nRecovered source text and saved:", output_path)
    print("Simplified text and human score remain blank until reviewed.")


In [ ]:

import os
import glob
import pandas as pd
from IPython.display import display

print("Available Hindi-related CSV files:")
for path in glob.glob("/content/*Hindi*.csv") + glob.glob("/content/hindi*.csv"):
    print("-", os.path.basename(path))

# Choose an existing pilot file, preferring the completed version
candidates = [
    "/content/JUDGEBERT_Hindi_simplification_pilot_10_completed.csv",
    "/content/JUDGEBERT_Hindi_simplification_pilot_10.csv",
    "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts_v2.csv",
    "/content/JUDGEBERT_Hindi_simplification_pilot_10_drafts.csv",
    "/content/hindi_simplification_pilot_v1.csv",
]

pilot_path = next((p for p in candidates if os.path.exists(p)), None)

if pilot_path is None:
    raise FileNotFoundError(
        "No pilot CSV found. Check the filenames printed above."
    )

pilot = pd.read_csv(pilot_path, dtype=str).fillna("")

required = {"pair_id", "original_text", "simplified_text"}
if not required.issubset(pilot.columns):
    raise ValueError(f"Missing required columns: {required - set(pilot.columns)}")

print("\nSelected:", os.path.basename(pilot_path))
print("Rows:", len(pilot))

# Preserve data; do not assign scores or invent simplifications
incomplete = (
    pilot["original_text"].str.strip().eq("")
    | pilot["simplified_text"].str.strip().eq("")
)

print("Incomplete pairs:", int(incomplete.sum()))
display(pilot.loc[incomplete, [
    c for c in [
        "pair_id", "original_text", "simplified_text", "review_status"
    ] if c in pilot.columns
]])

output_path = "/content/JUDGEBERT_Hindi_pilot_recovered_v1.csv"
pilot.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nWorking copy saved:", output_path)
print("Scores have not been fabricated.")


In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# Create a project folder inside Colab
project_dir = Path("/content/JUDGEBERT_LMP")
project_dir.mkdir(parents=True, exist_ok=True)

# Copy all available CSV and TXT files without overwriting existing copies
copied = 0
skipped = 0

for source in Path("/content").glob("*"):
    if source.is_file() and source.suffix.lower() in [".csv", ".txt"]:
        destination = project_dir / source.name

        if destination.exists():
            skipped += 1
            continue

        shutil.copy2(source, destination)
        copied += 1

print("Project folder:", project_dir)
print("New files copied:", copied)
print("Existing files preserved:", skipped)
print("\nFiles currently in the project folder:")

for file in sorted(project_dir.iterdir()):
    print(f"{file.name} — {file.stat().st_size:,} bytes")

In [ ]:

from pathlib import Path
import pandas as pd

root = Path("/content")
out = root / "JUDGEBERT_LMP"
out.mkdir(exist_ok=True)

# Find available source files
files = list(root.glob("*.csv"))
print("CSV files found:", len(files))

reports = []
for f in files:
    try:
        d = pd.read_csv(f, dtype=str).fillna("")
        cols = set(d.columns)
        original = next((c for c in ["original_text", "Input"] if c in cols), None)
        simplified = next((c for c in ["simplified_text", "Output"] if c in cols), None)
        score = next((c for c in ["lmp_score_1_to_10"] if c in cols), None)

        if original and simplified:
            valid = (
                d[original].str.strip().ne("") &
                d[simplified].str.strip().ne("")
            )
            reports.append({
                "file": f.name,
                "rows": len(d),
                "complete_pairs": int(valid.sum()),
                "scored_pairs": int(
                    pd.to_numeric(d[score], errors="coerce").notna().sum()
                ) if score else 0
            })
    except Exception as e:
        print("Skipped:", f.name, str(e)[:100])

report = pd.DataFrame(reports)
display(report)

report.to_csv(out / "dataset_audit.csv", index=False)
print("\nAudit saved:", out / "dataset_audit.csv")
print("Important: blank human scores remain blank.")


In [ ]:

!pip -q install sentence-transformers pandas

import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from pathlib import Path

# Use a multilingual model for English and Hindi
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)

# Test examples: original and simplified text in the SAME language
tests = [
    {
        "language": "English",
        "original": "The tenant must pay the rent within 30 days.",
        "simplified": "The tenant must pay rent within 30 days."
    },
    {
        "language": "English",
        "original": "The tenant must pay the rent within 30 days.",
        "simplified": "The tenant may pay rent within 90 days."
    },
    {
        "language": "Hindi",
        "original": "किरायेदार को 30 दिनों के भीतर किराया देना होगा।",
        "simplified": "किरायेदार को 30 दिनों के भीतर किराया देना होगा।"
    },
    {
        "language": "Hindi",
        "original": "किरायेदार को 30 दिनों के भीतर किराया देना होगा।",
        "simplified": "किरायेदार 90 दिनों के भीतर किराया दे सकता है।"
    }
]

results = []

for item in tests:
    embeddings = model.encode(
        [item["original"], item["simplified"]],
        convert_to_tensor=True
    )
    similarity = util.cos_sim(embeddings[0], embeddings[1]).item()

    results.append({
        **item,
        "semantic_similarity": round(similarity, 4),
        "preliminary_score_1_to_10": round(
            max(1, min(10, 1 + 9 * similarity)), 2
        )
    })

results_df = pd.DataFrame(results)

display(results_df[
    ["language", "original", "simplified",
     "semantic_similarity", "preliminary_score_1_to_10"]
])

output = "/content/JUDGEBERT_LMP/baseline_demo_results.csv"
Path(output).parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output, index=False, encoding="utf-8-sig")

print("\nBaseline demo saved to:", output)
print("Note: these are similarity-based demo scores, NOT validated legal scores.")


In [ ]:

import re
import pandas as pd
from pathlib import Path

def legal_risk_flags(original, simplified, language):
    o = original.lower()
    s = simplified.lower()
    flags = []

    # English negation
    en_neg = r"\b(not|no|never|neither|nor|cannot|can't|without)\b"
    if language == "English":
        if bool(re.search(en_neg, o)) != bool(re.search(en_neg, s)):
            flags.append("Possible negation change")

        must = r"\b(must|shall|required to|obligated to)\b"
        may = r"\b(may|might|can|could|permitted to)\b"
        if bool(re.search(must, o)) != bool(re.search(must, s)):
            flags.append("Possible obligation change")
        if bool(re.search(may, o)) != bool(re.search(may, s)):
            flags.append("Possible permission change")

    # Hindi checks: preliminary keyword rules
    else:
        en_neg_hi = ["नहीं", "न", "कभी नहीं", "बिना"]
        if any(x in o for x in en_neg_hi) != any(x in s for x in en_neg_hi):
            flags.append("Possible Hindi negation change")

        obligation_hi = ["होगा", "होगी", "होंगे", "अनिवार्य", "करना होगा"]
        permission_hi = ["सकता है", "सकती है", "सकते हैं", "अनुमति"]
        if any(x in o for x in obligation_hi) != any(x in s for x in obligation_hi):
            flags.append("Possible Hindi obligation change")
        if any(x in o for x in permission_hi) != any(x in s for x in permission_hi):
            flags.append("Possible Hindi permission change")

    # Flag changes in numeric values, dates, and percentages
    nums_o = re.findall(r"\d+(?:\.\d+)?", o)
    nums_s = re.findall(r"\d+(?:\.\d+)?", s)
    if nums_o != nums_s:
        flags.append("Possible number/date change")

    return flags

# Use the results created by the previous cell
path = Path("/content/JUDGEBERT_LMP/baseline_demo_results.csv")
df = pd.read_csv(path)

df["legal_risk_flags"] = df.apply(
    lambda r: "; ".join(
        legal_risk_flags(r["original"], r["simplified"], r["language"])
    ) or "No rule-based flag detected",
    axis=1
)

# Conservative prototype adjustment, not a trained legal score
df["risk_adjusted_demo_score"] = df.apply(
    lambda r: max(
        1,
        r["preliminary_score_1_to_10"]
        - 3 * len(r["legal_risk_flags"].split("; "))
        if r["legal_risk_flags"] != "No rule-based flag detected"
        else r["preliminary_score_1_to_10"]
    ),
    axis=1
)

display(df[[
    "language", "original", "simplified",
    "preliminary_score_1_to_10",
    "legal_risk_flags", "risk_adjusted_demo_score"
]])

out = "/content/JUDGEBERT_LMP/baseline_with_legal_checks.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")
print("\nSaved:", out)
print("Rule-based prototype only; human legal review is still required.")


In [ ]:

!pip -q install sentence-transformers pandas

from pathlib import Path
import pandas as pd
import re
import shutil
from sentence_transformers import SentenceTransformer, util
from google.colab import files

# Recreate project folder after the Colab restart
out_dir = Path("/content/JUDGEBERT_LMP")
out_dir.mkdir(parents=True, exist_ok=True)

# Four test cases: English and Hindi, same-language comparisons
tests = [
    ("English", "The tenant must pay the rent within 30 days.",
     "The tenant must pay rent within 30 days."),
    ("English", "The tenant must pay the rent within 30 days.",
     "The tenant may pay rent within 90 days."),
    ("Hindi", "किरायेदार को 30 दिनों के भीतर किराया देना होगा।",
     "किरायेदार को 30 दिनों के भीतर किराया देना होगा।"),
    ("Hindi", "किरायेदार को 30 दिनों के भीतर किराया देना होगा।",
     "किरायेदार 90 दिनों के भीतर किराया दे सकता है।")
]

print("Loading multilingual model...")
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

def risk_flags(original, simplified, language):
    o, s = original.lower(), simplified.lower()
    flags = []

    if language == "English":
        if bool(re.search(r"\b(not|no|never|cannot|can't|without)\b", o)) != \
           bool(re.search(r"\b(not|no|never|cannot|can't|without)\b", s)):
            flags.append("Possible negation change")

        if bool(re.search(r"\b(must|shall|required to|obligated to)\b", o)) != \
           bool(re.search(r"\b(must|shall|required to|obligated to)\b", s)):
            flags.append("Possible obligation change")

        if bool(re.search(r"\b(may|might|can|could|permitted to)\b", o)) != \
           bool(re.search(r"\b(may|might|can|could|permitted to)\b", s)):
            flags.append("Possible permission change")
    else:
        neg = ["नहीं", "कभी नहीं", "बिना"]
        obligation = ["होगा", "होगी", "होंगे", "अनिवार्य"]
        permission = ["सकता है", "सकती है", "सकते हैं", "अनुमति"]

        if any(x in o for x in neg) != any(x in s for x in neg):
            flags.append("Possible Hindi negation change")
        if any(x in o for x in obligation) != any(x in s for x in obligation):
            flags.append("Possible Hindi obligation change")
        if any(x in o for x in permission) != any(x in s for x in permission):
            flags.append("Possible Hindi permission change")

    if re.findall(r"\d+(?:\.\d+)?", o) != re.findall(r"\d+(?:\.\d+)?", s):
        flags.append("Possible number/date change")

    return flags

rows = []
for language, original, simplified in tests:
    emb = model.encode([original, simplified], convert_to_tensor=True)
    similarity = float(util.cos_sim(emb[0], emb[1]).item())
    base_score = round(max(1, min(10, 1 + 9 * similarity)), 2)
    flags = risk_flags(original, simplified, language)

    rows.append({
        "language": language,
        "original": original,
        "simplified": simplified,
        "semantic_similarity": round(similarity, 4),
        "preliminary_similarity_score": base_score,
        "legal_risk_flags": "; ".join(flags) if flags else "No rule-based flag detected",
        "risk_adjusted_demo_score": round(max(1, base_score - 3 * len(flags)), 2)
    })

df = pd.DataFrame(rows)
csv_path = out_dir / "baseline_with_legal_checks.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

# Include a readable project status note in the backup
(out_dir / "README.txt").write_text(
    "JUDGEBERT LMP baseline demo\n"
    "Scores are experimental similarity-based values, not validated legal judgments.\n"
    "Rule-based checks are incomplete and may miss meaning changes.\n"
    "Human-reviewed labeled data is still required for proper training and evaluation.\n",
    encoding="utf-8"
)

display(df)
zip_path = shutil.make_archive(
    "/content/JUDGEBERT_LMP_backup", "zip", root_dir=out_dir
)

print("\nBackup created:", zip_path)
files.download(zip_path)


In [ ]:

from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

source = Path("/content/JUDGEBERT_LMP_backup.zip")
destination = Path(
    "/content/drive/MyDrive/JUDGEBERT_LMP_backup.zip"
)

if source.exists():
    shutil.copy2(source, destination)
    print("Permanent backup saved to:", destination)
else:
    print("Backup ZIP not found. Run the recovery cell again.")


In [ ]:

from pathlib import Path
import zipfile

backup = Path("/content/drive/MyDrive/JUDGEBERT_LMP_backup.zip")
project = Path("/content/JUDGEBERT_LMP")

with zipfile.ZipFile(backup, "r") as z:
    z.extractall(project)

print("Backup extracted.")
print("CSV files found:")
for f in sorted(project.rglob("*.csv")):
    print(f"{f.name}: {f.stat().st_size:,} bytes")


In [ ]:

import pandas as pd
from pathlib import Path

root = Path("/content/JUDGEBERT_LMP")

for name in [
    "legal_simplification_pairs_en_v1.csv",
    "JUDGEBERT_Hindi_simplification_pilot_10_completed.csv",
    "hindi_original_candidates_v1.csv",
]:
    matches = list(root.rglob(name))
    print(f"\n--- {name} ---")
    if not matches:
        print("Not found in backup")
        continue

    df = pd.read_csv(matches[0]).fillna("")
    print("Rows:", len(df))
    print("Columns:", list(df.columns))

    for col in ["original_text", "simplified_text"]:
        if col in df.columns:
            print(f"{col} non-empty:", (df[col].astype(str).str.strip() != "").sum())
    if "lmp_score_1_to_10" in df.columns:
        print("Human scores:", pd.to_numeric(
            df["lmp_score_1_to_10"], errors="coerce"
        ).notna().sum())



--- legal_simplification_pairs_en_v1.csv ---
Not found in backup

--- JUDGEBERT_Hindi_simplification_pilot_10_completed.csv ---
Not found in backup

--- hindi_original_candidates_v1.csv ---
Not found in backup


In [ ]:

from pathlib import Path

search_roots = [
    Path("/content"),
    Path("/content/drive/MyDrive")
]

wanted = {
    "legal_simplification_pairs_en_v1.csv",
    "JUDGEBERT_Hindi_simplification_pilot_10_completed.csv",
    "hindi_original_candidates_v1.csv",
    "legal_text_pairs_v1.csv",
    "lmp_candidate_pairs_v2.csv",
}

print("Searching for existing project datasets...\n")

found = set()

for root in search_roots:
    if not root.exists():
        continue
    for path in root.rglob("*.csv"):
        if path.name in wanted:
            print(path)
            found.add(path.name)

print("\nFiles found:", len(found), "of", len(wanted))
print("Missing:", sorted(wanted - found))


Searching for existing project datasets...

/content/hindi_original_candidates_v1.csv
/content/JUDGEBERT_Hindi_simplification_pilot_10_completed.csv
/content/lmp_candidate_pairs_v2.csv
/content/legal_simplification_pairs_en_v1.csv
/content/legal_text_pairs_v1.csv

Files found: 5 of 5
Missing: []


In [ ]:

from pathlib import Path
import shutil
import zipfile

source_dir = Path("/content")
drive_dir = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
drive_dir.mkdir(parents=True, exist_ok=True)

files_to_save = [
    "hindi_original_candidates_v1.csv",
    "JUDGEBERT_Hindi_simplification_pilot_10_completed.csv",
    "lmp_candidate_pairs_v2.csv",
    "legal_simplification_pairs_en_v1.csv",
    "legal_text_pairs_v1.csv",
]

for name in files_to_save:
    src = source_dir / name
    if src.exists():
        shutil.copy2(src, drive_dir / name)
        print("Saved:", name)
    else:
        print("MISSING:", name)

# Create a fresh backup containing these files
backup = Path("/content/drive/MyDrive/JUDGEBERT_LMP_backup.zip")
with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as z:
    for name in files_to_save:
        src = drive_dir / name
        if src.exists():
            z.write(src, arcname=name)

print("\nDrive folder:", drive_dir)
print("Backup:", backup)
print("Backup size:", round(backup.stat().st_size / 1024, 1), "KB")


Saved: hindi_original_candidates_v1.csv
Saved: JUDGEBERT_Hindi_simplification_pilot_10_completed.csv
Saved: lmp_candidate_pairs_v2.csv
Saved: legal_simplification_pairs_en_v1.csv
Saved: legal_text_pairs_v1.csv

Drive folder: /content/drive/MyDrive/JUDGEBERT_LMP
Backup: /content/drive/MyDrive/JUDGEBERT_LMP_backup.zip
Backup size: 26017.5 KB


In [ ]:

import pandas as pd
from pathlib import Path

root = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

files = {
    "english": "legal_simplification_pairs_en_v1.csv",
    "hindi_pilot": "JUDGEBERT_Hindi_simplification_pilot_10_completed.csv",
    "hindi_candidates": "hindi_original_candidates_v1.csv",
    "mixed_candidates": "lmp_candidate_pairs_v2.csv",
}

frames = []

for source_type, filename in files.items():
    path = root / filename
    if not path.exists():
        print("Skipping missing file:", filename)
        continue

    df = pd.read_csv(path).fillna("")
    print(f"\n{filename}: {len(df)} rows")

    # Identify common column names
    original_col = next(
        (c for c in ["original_text", "original", "Input"] if c in df.columns),
        None
    )
    simplified_col = next(
        (c for c in ["simplified_text", "simplified", "Output"] if c in df.columns),
        None
    )

    if not original_col or not simplified_col:
        print("  Skipped: original/simplified columns not identified.")
        continue

    out = pd.DataFrame({
        "original_text": df[original_col].astype(str).str.strip(),
        "simplified_text": df[simplified_col].astype(str).str.strip(),
    })

    if "language" in df.columns:
        out["language"] = df["language"].astype(str).str.strip().str.lower()
    else:
        out["language"] = "en" if source_type == "english" else "hi"

    if "lmp_score_1_to_10" in df.columns:
        out["lmp_score_1_to_10"] = df["lmp_score_1_to_10"]
    else:
        out["lmp_score_1_to_10"] = ""

    out["source_file"] = filename
    out["source_type"] = source_type
    frames.append(out)

if not frames:
    raise ValueError("No compatible datasets found. Check the filenames and columns.")

combined = pd.concat(frames, ignore_index=True)

# Keep only complete, same-language candidate pairs
combined = combined[
    (combined["original_text"] != "") &
    (combined["simplified_text"] != "") &
    (combined["original_text"] != combined["simplified_text"])
].copy()

# Remove exact duplicate pairs within the same language
combined = combined.drop_duplicates(
    subset=["language", "original_text", "simplified_text"]
).reset_index(drop=True)

combined.insert(0, "pair_id", [
    f"LMP_{i:06d}" for i in range(1, len(combined) + 1)
])

output = root / "JUDGEBERT_LMP_candidate_dataset_v1.csv"
combined.to_csv(output, index=False, encoding="utf-8-sig")

print("\n=== CONSOLIDATED CANDIDATE DATASET ===")
print("Rows:", len(combined))
print("\nRows by language:")
print(combined["language"].value_counts(dropna=False))
print("\nHuman scores present:",
      pd.to_numeric(combined["lmp_score_1_to_10"], errors="coerce").notna().sum())
print("Saved to:", output)
display(combined.head(5))



legal_simplification_pairs_en_v1.csv: 6686 rows

JUDGEBERT_Hindi_simplification_pilot_10_completed.csv: 10 rows

hindi_original_candidates_v1.csv: 1639 rows

lmp_candidate_pairs_v2.csv: 4455 rows
  Skipped: original/simplified columns not identified.

=== CONSOLIDATED CANDIDATE DATASET ===
Rows: 6695

Rows by language:
language
en    6686
hi       9
Name: count, dtype: int64

Human scores present: 0
Saved to: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_candidate_dataset_v1.csv


,pair_id,original_text,simplified_text,language,lmp_score_1_to_10,source_file,source_type
0,LMP_000001,The Insurer may require repair estimates befor...,The insurer may require repair estimates befor...,en,,legal_simplification_pairs_en_v1.csv,english
1,LMP_000002,The Landlord may enter the Premises with reaso...,The landlord may enter the property with reaso...,en,,legal_simplification_pairs_en_v1.csv,english
2,LMP_000003,The Tenant shall notify the Landlord of extend...,The tenant must inform the landlord about long...,en,,legal_simplification_pairs_en_v1.csv,english
3,LMP_000004,The Consumer shall ensure that all promotional...,You must provide proof if the company asks for...,en,,legal_simplification_pairs_en_v1.csv,english
4,LMP_000005,The Insurer may deny claims that do not meet p...,The insurer may reject claims that do not meet...,en,,legal_simplification_pairs_en_v1.csv,english


In [ ]:

from pathlib import Path
import pandas as pd
import re
import random

random.seed(42)

# Find project files in Google Drive or the current Colab session
roots = [
    Path("/content/drive/MyDrive/JUDGEBERT_LMP"),
    Path("/content"),
]

def find_csv(filename):
    for root in roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    return None

english_path = find_csv("legal_simplification_pairs_en_v1.csv")
hindi_path = find_csv("JUDGEBERT_Hindi_simplification_pilot_10_completed.csv")

if english_path is None:
    raise FileNotFoundError(
        "English pairs CSV not found. Check the Files panel in Colab."
    )

out_dir = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Clean English source pairs
en = pd.read_csv(english_path).fillna("")

def get_col(df, candidates):
    lookup = {c.lower().strip(): c for c in df.columns}
    for name in candidates:
        if name in lookup:
            return lookup[name]
    return None

orig_col = get_col(en, ["original_text", "input", "original"])
simp_col = get_col(en, ["simplified_text", "output", "simplified"])

if orig_col is None or simp_col is None:
    raise ValueError(f"Could not identify text columns: {en.columns.tolist()}")

en = en[[orig_col, simp_col]].copy()
en.columns = ["original_text", "simplified_text"]

for c in ["original_text", "simplified_text"]:
    en[c] = en[c].astype(str).str.strip()

en = en[
    (en.original_text.str.len() >= 20) &
    (en.simplified_text.str.len() >= 15) &
    (en.original_text.str.len() <= 5000) &
    (en.simplified_text.str.len() <= 5000) &
    (en.original_text != en.simplified_text)
].copy()

ratio = (
    en.simplified_text.str.len() /
    en.original_text.str.len().clip(lower=1)
)
en = en[(ratio >= 0.25) & (ratio <= 2.0)].copy()
en = en.drop_duplicates(
    subset=["original_text", "simplified_text"]
).reset_index(drop=True)

en["language"] = "en"
en["source_type"] = "paired_source_unverified"
en["label_source"] = "source_pair_weak_label"
en["preservation_label"] = 1

# 2. Load complete Hindi pilot pairs, if present
hi_rows = []

if hindi_path is not None:
    hi = pd.read_csv(hindi_path).fillna("")
    oc = get_col(hi, ["original_text", "original"])
    sc = get_col(hi, ["simplified_text", "simplified"])

    if oc and sc:
        for _, row in hi.iterrows():
            original = str(row[oc]).strip()
            simplified = str(row[sc]).strip()

            if len(original) >= 10 and len(simplified) >= 10:
                hi_rows.append({
                    "original_text": original,
                    "simplified_text": simplified,
                    "language": "hi",
                    "source_type": "hindi_pilot_needs_review",
                    "label_source": "pilot_pair_unverified",
                    "preservation_label": 1
                })

hi_df = pd.DataFrame(hi_rows)

# 3. Create controlled negative examples.
# These deliberately alter meaning; they are synthetic test cases.
def make_negative(text, lang):
    if lang == "en":
        patterns = [
            (r"\bmust\b", "may"),
            (r"\bshall\b", "may"),
            (r"\bwithin\s+(\d+)\s+days\b",
             lambda m: f"within {int(m.group(1)) + 30} days"),
            (r"\bnot\b", ""),
        ]
    else:
        patterns = [
            ("नहीं", ""),
            ("अनिवार्य है", "वैकल्पिक है"),
            ("करना होगा", "कर सकता है"),
            ("30", "90"),
            ("३०", "९०"),
        ]

    for pattern, replacement in patterns:
        if lang == "en":
            changed, n = re.subn(
                pattern, replacement, text,
                count=1, flags=re.IGNORECASE
            )
        else:
            changed = text.replace(pattern, replacement, 1)
            n = int(changed != text)

        if n and changed.strip() and changed.strip() != text.strip():
            return changed.strip()

    return None

rows = []

# Source-pair positives: weak labels, not expert-confirmed labels
for i, r in en.iterrows():
    rows.append({
        **r.to_dict(),
        "pair_id": f"EN_{i:06d}",
        "human_score_1_to_10": "",
        "review_status": "needs_human_legal_review"
    })

# Hindi pilot positives: also require review
for i, r in hi_df.iterrows():
    rows.append({
        **r.to_dict(),
        "pair_id": f"HI_{i:06d}",
        "human_score_1_to_10": "",
        "review_status": "needs_human_legal_review"
    })

# Negative variants of English pairs
for i, r in en.iterrows():
    changed = make_negative(r["simplified_text"], "en")
    if changed:
        rows.append({
            "pair_id": f"EN_NEG_{i:06d}",
            "original_text": r["original_text"],
            "simplified_text": changed,
            "language": "en",
            "source_type": "synthetic_meaning_change",
            "label_source": "rule_generated_negative",
            "preservation_label": 0,
            "human_score_1_to_10": "",
            "review_status": "synthetic_needs_review"
        })

# Negative variants from available Hindi pilot pairs
for i, r in hi_df.iterrows():
    changed = make_negative(r["simplified_text"], "hi")
    if changed:
        rows.append({
            "pair_id": f"HI_NEG_{i:06d}",
            "original_text": r["original_text"],
            "simplified_text": changed,
            "language": "hi",
            "source_type": "synthetic_meaning_change",
            "label_source": "rule_generated_negative",
            "preservation_label": 0,
            "human_score_1_to_10": "",
            "review_status": "synthetic_needs_review"
        })

data = pd.DataFrame(rows)

if data.empty:
    raise ValueError("No usable pairs were created.")

# Split by original-text group to reduce leakage between variants
data["group_key"] = (
    data["language"] + "|" + data["original_text"].str.lower().str.strip()
)
groups = data["group_key"].drop_duplicates().tolist()
random.shuffle(groups)

n = len(groups)
train_groups = set(groups[:int(n * 0.8)])
val_groups = set(groups[int(n * 0.8):int(n * 0.9)])

data["split"] = data["group_key"].apply(
    lambda g: "train" if g in train_groups
    else ("validation" if g in val_groups else "test")
)
data = data.drop(columns=["group_key"])

# Save dataset and audit
dataset_path = out_dir / "JUDGEBERT_LMP_dataset_v1.csv"
audit_path = out_dir / "JUDGEBERT_LMP_dataset_audit_v1.csv"

data.to_csv(dataset_path, index=False, encoding="utf-8-sig")

audit = (
    data.groupby(["language", "split", "preservation_label"])
    .size().reset_index(name="rows")
)
audit.to_csv(audit_path, index=False, encoding="utf-8-sig")

print("DATASET CREATED")
print("Dataset:", dataset_path)
print("Audit:", audit_path)
print("Total rows:", len(data))
print("\nRows by language:")
print(data["language"].value_counts().to_string())
print("\nRows by label:")
print(data.groupby(["language", "preservation_label"]).size())
print("\nRows by split:")
print(data["split"].value_counts().to_string())
print("\nHuman scores assigned:", data["human_score_1_to_10"].ne("").sum())
print("\nSample:")
display(data[[
    "pair_id", "language", "original_text", "simplified_text",
    "preservation_label", "label_source", "split"
]].head(10))


DATASET CREATED
Dataset: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_dataset_v1.csv
Audit: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_dataset_audit_v1.csv
Total rows: 5687

Rows by language:
language
en    5676
hi      11

Rows by label:
language  preservation_label
en        0                     1468
          1                     4208
hi        0                        2
          1                        9
dtype: int64

Rows by split:
split
train         4544
validation     577
test           566

Human scores assigned: 0

Sample:


,pair_id,language,original_text,simplified_text,preservation_label,label_source,split
0,EN_000000,en,The Insurer may require repair estimates befor...,The insurer may require repair estimates befor...,1,source_pair_weak_label,train
1,EN_000001,en,The Landlord may enter the Premises with reaso...,The landlord may enter the property with reaso...,1,source_pair_weak_label,validation
2,EN_000002,en,The Tenant shall notify the Landlord of extend...,The tenant must inform the landlord about long...,1,source_pair_weak_label,test
3,EN_000003,en,The Consumer shall ensure that all promotional...,You must provide proof if the company asks for...,1,source_pair_weak_label,train
4,EN_000004,en,The Insurer may deny claims that do not meet p...,The insurer may reject claims that do not meet...,1,source_pair_weak_label,train
5,EN_000005,en,The Company may revoke access to licensed soft...,The company may remove access to licensed soft...,1,source_pair_weak_label,train
6,EN_000006,en,The Company may limit access to premium features.,The company may restrict access to premium fea...,1,source_pair_weak_label,validation
7,EN_000007,en,The Tenant shall report broken handrails immed...,The tenant must immediately report broken hand...,1,source_pair_weak_label,train
8,EN_000008,en,The Employee shall accurately report project p...,The employee must accurately report project st...,1,source_pair_weak_label,test
9,EN_000009,en,User data may be used to improve account manag...,User data may be used to improve account manag...,1,source_pair_weak_label,train


In [ ]:
from pathlib import Path
import pandas as pd

p = Path("/content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_dataset_v1.csv")

print("Dataset exists:", p.exists())

if p.exists():
    df = pd.read_csv(p)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("\nLanguage counts:")
    print(df["language"].value_counts())
    print("\nSplit counts:")
    print(df["split"].value_counts())

Dataset exists: True
Rows: 5687
Columns: ['original_text', 'simplified_text', 'language', 'source_type', 'label_source', 'preservation_label', 'pair_id', 'human_score_1_to_10', 'review_status', 'split']

Language counts:
language
en    5676
hi      11
Name: count, dtype: int64

Split counts:
split
train         4544
validation     577
test           566
Name: count, dtype: int64


In [ ]:

!pip -q install sentence-transformers

from pathlib import Path
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
input_file = project / "JUDGEBERT_LMP_dataset_v1.csv"
df = pd.read_csv(input_file).fillna("")

# Multilingual sentence embeddings support English and Hindi
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def risk_flags(original, simplified, language):
    flags = []
    o, s = original.lower(), simplified.lower()

    # English legal-risk indicators
    if language == "en":
        patterns = {
            "negation": r"\b(not|never|no|neither|unless|without)\b",
            "obligation": r"\b(must|shall|required|may|can|optional)\b",
            "number_or_deadline": r"\b\d+(?:\.\d+)?\b",
            "party_or_right": r"\b(tenant|landlord|insurer|insured|claimant|company|court|employee)\b"
        }
        for name, pattern in patterns.items():
            if bool(re.search(pattern, o)) != bool(re.search(pattern, s)):
                flags.append(name + "_presence_changed")

        nums_o = re.findall(r"\b\d+(?:\.\d+)?\b", o)
        nums_s = re.findall(r"\b\d+(?:\.\d+)?\b", s)
        if nums_o != nums_s:
            flags.append("numbers_changed")

        # Flag modal-word differences for review
        modals = ["must", "shall", "may", "can", "cannot", "should"]
        if any((w in o) != (w in s) for w in modals):
            flags.append("legal_modal_changed")

    else:
        # Hindi indicators; these are simple review heuristics
        for term in ["नहीं", "न", "मत"]:
            if (term in o) != (term in s):
                flags.append("possible_negation_change")
                break
        if re.findall(r"\d+", o) != re.findall(r"\d+", s):
            flags.append("numbers_changed")
        if any(term in o for term in ["करना होगा", "अनिवार्य"]) != any(
            term in s for term in ["करना होगा", "अनिवार्य"]
        ):
            flags.append("possible_obligation_change")

    return sorted(set(flags))

# Score examples in manageable batches
scores, similarities, all_flags = [], [], []

for start in range(0, len(df), 64):
    batch = df.iloc[start:start+64]
    originals = batch["original_text"].astype(str).tolist()
    simplified = batch["simplified_text"].astype(str).tolist()

    emb_o = model.encode(originals, normalize_embeddings=True, show_progress_bar=False)
    emb_s = model.encode(simplified, normalize_embeddings=True, show_progress_bar=False)
    sims = np.sum(emb_o * emb_s, axis=1)

    for (_, row), sim in zip(batch.iterrows(), sims):
        flags = risk_flags(
            str(row["original_text"]),
            str(row["simplified_text"]),
            str(row["language"])
        )

        # Preliminary heuristic only, not a calibrated legal score
        score = float(np.clip(sim, 0, 1) * 10)
        if flags:
            score = min(score, 5.0)

        similarities.append(round(float(sim), 4))
        scores.append(round(score, 2))
        all_flags.append("; ".join(flags) if flags else "no_rule_flag")

df["semantic_similarity"] = similarities
df["preliminary_lmp_score_1_to_10"] = scores
df["legal_risk_flags"] = all_flags
df["score_status"] = "automatic_preliminary_not_legally_validated"

output = project / "JUDGEBERT_LMP_baseline_results_v1.csv"
df.to_csv(output, index=False, encoding="utf-8-sig")

print("Saved:", output)
print("Rows scored:", len(df))
print("\nScore summary:")
print(df["preliminary_lmp_score_1_to_10"].describe().round(2))
print("\nRisk flags:")
print(df["legal_risk_flags"].value_counts().head(10))
display(df[[
    "language", "original_text", "simplified_text",
    "semantic_similarity", "legal_risk_flags",
    "preliminary_lmp_score_1_to_10"
]].head(10))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_baseline_results_v1.csv
Rows scored: 5687

Score summary:
count    5687.00
mean        6.01
std         1.88
min         0.31
25%         5.00
50%         5.00
75%         6.97
max         9.98
Name: preliminary_lmp_score_1_to_10, dtype: float64

Risk flags:
legal_risk_flags
legal_modal_changed                                                            2719
no_rule_flag                                                                   1688
legal_modal_changed; obligation_presence_changed                                549
legal_modal_changed; negation_presence_changed; obligation_presence_changed     163
negation_presence_changed                                                       148
legal_modal_changed; negation_presence_changed                                   91
legal_modal_changed; party_or_right_presence_changed                             74
number_or_deadline_presence_changed; numbers_changed                          

,language,original_text,simplified_text,semantic_similarity,legal_risk_flags,preliminary_lmp_score_1_to_10
0,en,The Insurer may require repair estimates befor...,The insurer may require repair estimates befor...,0.9716,no_rule_flag,9.72
1,en,The Landlord may enter the Premises with reaso...,The landlord may enter the property with reaso...,0.9390,no_rule_flag,9.39
2,en,The Tenant shall notify the Landlord of extend...,The tenant must inform the landlord about long...,0.7279,legal_modal_changed,5.00
3,en,The Consumer shall ensure that all promotional...,You must provide proof if the company asks for...,0.6136,legal_modal_changed,5.00
4,en,The Insurer may deny claims that do not meet p...,The insurer may reject claims that do not meet...,0.9393,no_rule_flag,9.39
5,en,The Company may revoke access to licensed soft...,The company may remove access to licensed soft...,0.9500,no_rule_flag,9.50
6,en,The Company may limit access to premium features.,The company may restrict access to premium fea...,0.9670,no_rule_flag,9.67
7,en,The Tenant shall report broken handrails immed...,The tenant must immediately report broken hand...,0.8978,legal_modal_changed,5.00
8,en,The Employee shall accurately report project p...,The employee must accurately report project st...,0.8838,legal_modal_changed,5.00
9,en,User data may be used to improve account manag...,User data may be used to improve account manag...,0.9848,no_rule_flag,9.85


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import re

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

tests = [
    {
        "language": "en",
        "original": "The tenant must pay rent within 30 days.",
        "simplified": "The tenant must pay rent within 30 days.",
        "expected": "preserved"
    },
    {
        "language": "en",
        "original": "The tenant must pay rent within 30 days.",
        "simplified": "The tenant must pay rent within 90 days.",
        "expected": "changed"
    },
    {
        "language": "en",
        "original": "The tenant must pay rent.",
        "simplified": "The tenant may pay rent.",
        "expected": "changed"
    },
    {
        "language": "en",
        "original": "The tenant must not sublet the property.",
        "simplified": "The tenant may sublet the property.",
        "expected": "changed"
    },
    {
        "language": "en",
        "original": "The tenant shall pay rent on time.",
        "simplified": "The tenant must pay rent on time.",
        "expected": "preserved"
    },
    {
        "language": "hi",
        "original": "किरायेदार को 30 दिनों के भीतर किराया देना होगा।",
        "simplified": "किरायेदार को 30 दिनों के भीतर किराया देना होगा।",
        "expected": "preserved"
    },
    {
        "language": "hi",
        "original": "किरायेदार को 30 दिनों के भीतर किराया देना होगा।",
        "simplified": "किरायेदार को 90 दिनों के भीतर किराया देना होगा।",
        "expected": "changed"
    },
    {
        "language": "hi",
        "original": "किरायेदार को किराया देना होगा।",
        "simplified": "किरायेदार किराया दे सकता है।",
        "expected": "changed"
    },
    {
        "language": "hi",
        "original": "किरायेदार संपत्ति को किराए पर नहीं दे सकता।",
        "simplified": "किरायेदार संपत्ति को किराए पर दे सकता है।",
        "expected": "changed"
    }
]

results = []

for t in tests:
    embeddings = model.encode(
        [t["original"], t["simplified"]],
        normalize_embeddings=True
    )
    sim = float(np.dot(embeddings[0], embeddings[1]))

    o, s = t["original"], t["simplified"]
    flags = []

    if t["language"] == "en":
        if re.findall(r"\d+", o) != re.findall(r"\d+", s):
            flags.append("numbers_changed")

        modal = r"\b(must|shall|may|can|cannot|should)\b"
        if re.findall(modal, o.lower()) != re.findall(modal, s.lower()):
            flags.append("modal_changed")

        neg = r"\b(not|never|no|unless|without)\b"
        if bool(re.search(neg, o.lower())) != bool(re.search(neg, s.lower())):
            flags.append("negation_changed")
    else:
        if re.findall(r"\d+", o) != re.findall(r"\d+", s):
            flags.append("numbers_changed")
        if ("नहीं" in o) != ("नहीं" in s):
            flags.append("negation_changed")
        obligation_o = any(x in o for x in ["देना होगा", "करना होगा", "अनिवार्य"])
        obligation_s = any(x in s for x in ["देना होगा", "करना होगा", "अनिवार्य"])
        permission_o = any(x in o for x in ["दे सकता", "कर सकता"])
        permission_s = any(x in s for x in ["दे सकता", "कर सकता"])
        if obligation_o != obligation_s or permission_o != permission_s:
            flags.append("obligation_or_permission_changed")

    results.append({
        "language": t["language"],
        "original": o,
        "simplified": s,
        "expected": t["expected"],
        "semantic_similarity": round(sim, 4),
        "risk_flags": "; ".join(flags) if flags else "none",
        "heuristic_decision": (
            "review_for_change" if flags else "no_rule_flag"
        )
    })

test_df = pd.DataFrame(results)
out = project / "JUDGEBERT_LMP_controlled_test_v1.csv"
test_df.to_csv(out, index=False, encoding="utf-8-sig")

print("Saved:", out)
display(test_df)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_controlled_test_v1.csv


,language,original,simplified,expected,semantic_similarity,risk_flags,heuristic_decision
0,en,The tenant must pay rent within 30 days.,The tenant must pay rent within 30 days.,preserved,1.0000,none,no_rule_flag
1,en,The tenant must pay rent within 30 days.,The tenant must pay rent within 90 days.,changed,0.5796,numbers_changed,review_for_change
2,en,The tenant must pay rent.,The tenant may pay rent.,changed,0.9407,modal_changed,review_for_change
3,en,The tenant must not sublet the property.,The tenant may sublet the property.,changed,0.6818,modal_changed; negation_changed,review_for_change
4,en,The tenant shall pay rent on time.,The tenant must pay rent on time.,preserved,0.9778,modal_changed,review_for_change
5,hi,किरायेदार को 30 दिनों के भीतर किराया देना होगा।,किरायेदार को 30 दिनों के भीतर किराया देना होगा।,preserved,1.0000,none,no_rule_flag
6,hi,किरायेदार को 30 दिनों के भीतर किराया देना होगा।,किरायेदार को 90 दिनों के भीतर किराया देना होगा।,changed,0.6699,numbers_changed,review_for_change
7,hi,किरायेदार को किराया देना होगा।,किरायेदार किराया दे सकता है।,changed,0.9126,obligation_or_permission_changed,review_for_change
8,hi,किरायेदार संपत्ति को किराए पर नहीं दे सकता।,किरायेदार संपत्ति को किराए पर दे सकता है।,changed,0.6464,negation_changed,review_for_change


In [ ]:

import re
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
input_file = project / "JUDGEBERT_LMP_controlled_test_v1.csv"

df = pd.read_csv(input_file)

def normalize_english(text):
    text = str(text).lower()
    # Treat these as equivalent obligation terms for this prototype.
    text = re.sub(r"\b(must|shall)\b", "OBLIGATION", text)
    text = re.sub(r"\b(may|can)\b", "PERMISSION", text)
    text = re.sub(r"\bcannot\b", "CANNOT", text)
    return text

def revised_flags(row):
    original = str(row["original"])
    simplified = str(row["simplified"])
    lang = row["language"]
    flags = []

    if lang == "en":
        if re.findall(r"\d+", original) != re.findall(r"\d+", simplified):
            flags.append("numbers_changed")

        if normalize_english(original) != normalize_english(simplified):
            # Compare obligation/permission terms separately.
            o_obligation = bool(re.search(r"\b(must|shall)\b", original.lower()))
            s_obligation = bool(re.search(r"\b(must|shall)\b", simplified.lower()))
            o_permission = bool(re.search(r"\b(may|can)\b", original.lower()))
            s_permission = bool(re.search(r"\b(may|can)\b", simplified.lower()))

            if o_obligation != s_obligation or o_permission != s_permission:
                flags.append("modal_meaning_changed")

        neg = r"\b(not|never|no|unless|without)\b"
        if bool(re.search(neg, original.lower())) != bool(re.search(neg, simplified.lower())):
            flags.append("negation_changed")

    else:
        if re.findall(r"\d+", original) != re.findall(r"\d+", simplified):
            flags.append("numbers_changed")
        if ("नहीं" in original) != ("नहीं" in simplified):
            flags.append("negation_changed")

    return "; ".join(dict.fromkeys(flags)) if flags else "none"

df["revised_risk_flags"] = df.apply(revised_flags, axis=1)
df["revised_decision"] = df["revised_risk_flags"].apply(
    lambda x: "review_for_change" if x != "none" else "no_rule_flag"
)

out = project / "JUDGEBERT_LMP_controlled_test_v2.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

print("Saved:", out)
display(df[[
    "language", "original", "simplified", "expected",
    "revised_risk_flags", "revised_decision"
]])


Saved: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_controlled_test_v2.csv


,language,original,simplified,expected,revised_risk_flags,revised_decision
0,en,The tenant must pay rent within 30 days.,The tenant must pay rent within 30 days.,preserved,none,no_rule_flag
1,en,The tenant must pay rent within 30 days.,The tenant must pay rent within 90 days.,changed,numbers_changed,review_for_change
2,en,The tenant must pay rent.,The tenant may pay rent.,changed,modal_meaning_changed,review_for_change
3,en,The tenant must not sublet the property.,The tenant may sublet the property.,changed,modal_meaning_changed; negation_changed,review_for_change
4,en,The tenant shall pay rent on time.,The tenant must pay rent on time.,preserved,none,no_rule_flag
5,hi,किरायेदार को 30 दिनों के भीतर किराया देना होगा।,किरायेदार को 30 दिनों के भीतर किराया देना होगा।,preserved,none,no_rule_flag
6,hi,किरायेदार को 30 दिनों के भीतर किराया देना होगा।,किरायेदार को 90 दिनों के भीतर किराया देना होगा।,changed,numbers_changed,review_for_change
7,hi,किरायेदार को किराया देना होगा।,किरायेदार किराया दे सकता है।,changed,none,no_rule_flag
8,hi,किरायेदार संपत्ति को किराए पर नहीं दे सकता।,किरायेदार संपत्ति को किराए पर दे सकता है।,changed,negation_changed,review_for_change


In [ ]:

import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
path = project / "JUDGEBERT_LMP_controlled_test_v2.csv"
df = pd.read_csv(path)

# Compare system decisions with the expected outcomes in the test.
df["expected_review"] = df["expected"].astype(str).str.lower().eq("changed")
df["predicted_review"] = df["revised_decision"].eq("review_for_change")

correct = (df["expected_review"] == df["predicted_review"]).sum()
total = len(df)

print("Controlled test evaluation")
print("--------------------------")
print("Test examples:", total)
print("Correct decisions:", correct)
print("Accuracy:", round(correct / total, 3) if total else "N/A")
print("\nExpected vs predicted:")
print(pd.crosstab(
    df["expected_review"],
    df["predicted_review"],
    rownames=["Expected change"],
    colnames=["Predicted change"]
))
print("\nIncorrect examples:")
display(df.loc[
    df["expected_review"] != df["predicted_review"],
    ["language", "original", "simplified", "expected",
     "revised_risk_flags", "revised_decision"]
])


Controlled test evaluation
--------------------------
Test examples: 9
Correct decisions: 8
Accuracy: 0.889

Expected vs predicted:
Predicted change  False  True 
Expected change               
False                 3      0
True                  1      5

Incorrect examples:


,language,original,simplified,expected,revised_risk_flags,revised_decision
7,hi,किरायेदार को किराया देना होगा।,किरायेदार किराया दे सकता है।,changed,none,no_rule_flag


In [ ]:

import pandas as pd
import re
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
path = project / "JUDGEBERT_LMP_controlled_test_v2.csv"
df = pd.read_csv(path)

def hindi_risk_flags(original, simplified):
    original = str(original)
    simplified = str(simplified)
    flags = []

    # Detect changes in numbers/deadlines.
    if re.findall(r"\d+", original) != re.findall(r"\d+", simplified):
        flags.append("numbers_changed")

    # Approximate Hindi obligation and permission patterns.
    obligation = r"(होगा|होगी|होंगे|चाहिए|अनिवार्य|करना पड़ेगा)"
    permission = r"(सकता है|सकती है|सकते हैं|अनुमति है)"

    o_obligation = bool(re.search(obligation, original))
    s_obligation = bool(re.search(obligation, simplified))
    o_permission = bool(re.search(permission, original))
    s_permission = bool(re.search(permission, simplified))

    if o_obligation != s_obligation or o_permission != s_permission:
        flags.append("obligation_permission_changed")

    # Basic negation check.
    negation = r"(नहीं|कभी नहीं|बिना)"
    if bool(re.search(negation, original)) != bool(re.search(negation, simplified)):
        flags.append("negation_changed")

    return "; ".join(flags) if flags else "none"

mask = df["language"].astype(str).str.lower().isin(["hi", "hindi"])
df.loc[mask, "revised_risk_flags"] = df.loc[mask].apply(
    lambda row: hindi_risk_flags(row["original"], row["simplified"]),
    axis=1
)
df["revised_decision"] = df["revised_risk_flags"].apply(
    lambda flags: "review_for_change" if flags != "none" else "no_rule_flag"
)

out = project / "JUDGEBERT_LMP_controlled_test_v3.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

df["expected_review"] = df["expected"].astype(str).str.lower().eq("changed")
df["predicted_review"] = df["revised_decision"].eq("review_for_change")

print("Saved:", out)
print("Correct:", int((df["expected_review"] == df["predicted_review"]).sum()), "/", len(df))
print("Accuracy:", round((df["expected_review"] == df["predicted_review"]).mean(), 3))
display(df[[
    "language", "original", "simplified", "expected",
    "revised_risk_flags", "revised_decision"
]])


Saved: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_controlled_test_v3.csv
Correct: 9 / 9
Accuracy: 1.0


,language,original,simplified,expected,revised_risk_flags,revised_decision
0,en,The tenant must pay rent within 30 days.,The tenant must pay rent within 30 days.,preserved,none,no_rule_flag
1,en,The tenant must pay rent within 30 days.,The tenant must pay rent within 90 days.,changed,numbers_changed,review_for_change
2,en,The tenant must pay rent.,The tenant may pay rent.,changed,modal_meaning_changed,review_for_change
3,en,The tenant must not sublet the property.,The tenant may sublet the property.,changed,modal_meaning_changed; negation_changed,review_for_change
4,en,The tenant shall pay rent on time.,The tenant must pay rent on time.,preserved,none,no_rule_flag
5,hi,किरायेदार को 30 दिनों के भीतर किराया देना होगा।,किरायेदार को 30 दिनों के भीतर किराया देना होगा।,preserved,none,no_rule_flag
6,hi,किरायेदार को 30 दिनों के भीतर किराया देना होगा।,किरायेदार को 90 दिनों के भीतर किराया देना होगा।,changed,numbers_changed,review_for_change
7,hi,किरायेदार को किराया देना होगा।,किरायेदार किराया दे सकता है।,changed,obligation_permission_changed,review_for_change
8,hi,किरायेदार संपत्ति को किराए पर नहीं दे सकता।,किरायेदार संपत्ति को किराए पर दे सकता है।,changed,obligation_permission_changed; negation_changed,review_for_change


In [ ]:
from pathlib import Path
import pandas as pd

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

print("Project folder exists:", project.exists())
print("\nAvailable project files:")
for file in sorted(project.iterdir()):
    if file.is_file():
        print(file.name)

Project folder exists: True

Available project files:
JUDGEBERT_Hindi_simplification_pilot_10_completed.csv
JUDGEBERT_LMP_baseline_results_v1.csv
JUDGEBERT_LMP_candidate_dataset_v1.csv
JUDGEBERT_LMP_controlled_test_v1.csv
JUDGEBERT_LMP_controlled_test_v2.csv
JUDGEBERT_LMP_controlled_test_v3.csv
JUDGEBERT_LMP_dataset_audit_v1.csv
JUDGEBERT_LMP_dataset_v1.csv
hindi_original_candidates_v1.csv
legal_simplification_pairs_en_v1.csv
legal_text_pairs_v1.csv
lmp_candidate_pairs_v2.csv


In [ ]:
from pathlib import Path
import pandas as pd

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

for name in [
    "JUDGEBERT_LMP_dataset_v1.csv",
    "JUDGEBERT_LMP_baseline_results_v1.csv",
    "JUDGEBERT_LMP_controlled_test_v3.csv"
]:
    path = project / name
    print(f"\n{name}: {'FOUND' if path.exists() else 'MISSING'}")
    if path.exists():
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())


JUDGEBERT_LMP_dataset_v1.csv: FOUND
Rows: 5687
Columns: ['original_text', 'simplified_text', 'language', 'source_type', 'label_source', 'preservation_label', 'pair_id', 'human_score_1_to_10', 'review_status', 'split']

JUDGEBERT_LMP_baseline_results_v1.csv: FOUND
Rows: 5687
Columns: ['original_text', 'simplified_text', 'language', 'source_type', 'label_source', 'preservation_label', 'pair_id', 'human_score_1_to_10', 'review_status', 'split', 'semantic_similarity', 'preliminary_lmp_score_1_to_10', 'legal_risk_flags', 'score_status']

JUDGEBERT_LMP_controlled_test_v3.csv: FOUND
Rows: 9
Columns: ['language', 'original', 'simplified', 'expected', 'semantic_similarity', 'risk_flags', 'heuristic_decision', 'revised_risk_flags', 'revised_decision']


In [ ]:

import pandas as pd
import re
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
input_file = project / "JUDGEBERT_LMP_baseline_results_v1.csv"

df = pd.read_csv(input_file).fillna("")

def get_risk_flags(original, simplified, language):
    original = str(original)
    simplified = str(simplified)
    lang = str(language).lower()
    flags = []

    # Numbers and deadlines
    if re.findall(r"\d+(?:\.\d+)?", original) != re.findall(r"\d+(?:\.\d+)?", simplified):
        flags.append("numbers_changed")

    if lang in ("hi", "hindi"):
        obligation = r"(होगा|होगी|होंगे|चाहिए|अनिवार्य|करना पड़ेगा)"
        permission = r"(सकता है|सकती है|सकते हैं|अनुमति है)"
        negation = r"(नहीं|कभी नहीं|बिना)"
    else:
        obligation = r"\b(must|shall|required to|is required to)\b"
        permission = r"\b(may|can|permitted to|allowed to)\b"
        negation = r"\b(not|never|no|neither|unless|without)\b"

    if bool(re.search(obligation, original, re.I)) != bool(re.search(obligation, simplified, re.I)):
        flags.append("obligation_changed")

    if bool(re.search(permission, original, re.I)) != bool(re.search(permission, simplified, re.I)):
        flags.append("permission_changed")

    if bool(re.search(negation, original, re.I)) != bool(re.search(negation, simplified, re.I)):
        flags.append("negation_changed")

    return "; ".join(flags) if flags else "no_rule_flag"

df["lmp_risk_flags"] = df.apply(
    lambda row: get_risk_flags(
        row["original_text"],
        row["simplified_text"],
        row["language"]
    ),
    axis=1
)

# Keep the existing similarity score if available.
score_col = next(
    (c for c in ["preliminary_lmp_score_1_to_10",
                 "preliminary_score_1_to_10",
                 "risk_adjusted_demo_score"]
     if c in df.columns),
    None
)

if score_col:
    df["preliminary_lmp_score_1_to_10"] = pd.to_numeric(
        df[score_col], errors="coerce"
    ).clip(1, 10)
else:
    print("No existing score column found; scores were not invented.")

df["system_decision"] = df["lmp_risk_flags"].apply(
    lambda x: "review_for_change" if x != "no_rule_flag" else "no_rule_flag"
)

out = project / "JUDGEBERT_LMP_scoring_pipeline_v1.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")

print("Saved:", out)
print("Total pairs:", len(df))
print("\nLanguage counts:")
print(df["language"].value_counts(dropna=False))
print("\nRisk flag counts:")
print(df["lmp_risk_flags"].value_counts())
print("\nSample results:")
display(df[[
    "language", "original_text", "simplified_text",
    "lmp_risk_flags", "system_decision"
] + (["preliminary_lmp_score_1_to_10"]
     if "preliminary_lmp_score_1_to_10" in df.columns else [])].head(10))


Saved: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_scoring_pipeline_v1.csv
Total pairs: 5687

Language counts:
language
en    5676
hi      11
Name: count, dtype: int64

Risk flag counts:
lmp_risk_flags
no_rule_flag                                                3169
obligation_changed; permission_changed                      1299
obligation_changed                                           459
negation_changed                                             206
numbers_changed                                              152
obligation_changed; negation_changed                         140
permission_changed                                           128
obligation_changed; permission_changed; negation_changed      37
numbers_changed; negation_changed                             30
permission_changed; negation_changed                          23
numbers_changed; permission_changed                           22
numbers_changed; permission_changed; negation_changed         11
numbers_cha

,language,original_text,simplified_text,lmp_risk_flags,system_decision,preliminary_lmp_score_1_to_10
0,en,The Insurer may require repair estimates befor...,The insurer may require repair estimates befor...,no_rule_flag,no_rule_flag,9.72
1,en,The Landlord may enter the Premises with reaso...,The landlord may enter the property with reaso...,no_rule_flag,no_rule_flag,9.39
2,en,The Tenant shall notify the Landlord of extend...,The tenant must inform the landlord about long...,no_rule_flag,no_rule_flag,5.00
3,en,The Consumer shall ensure that all promotional...,You must provide proof if the company asks for...,no_rule_flag,no_rule_flag,5.00
4,en,The Insurer may deny claims that do not meet p...,The insurer may reject claims that do not meet...,no_rule_flag,no_rule_flag,9.39
5,en,The Company may revoke access to licensed soft...,The company may remove access to licensed soft...,no_rule_flag,no_rule_flag,9.50
6,en,The Company may limit access to premium features.,The company may restrict access to premium fea...,no_rule_flag,no_rule_flag,9.67
7,en,The Tenant shall report broken handrails immed...,The tenant must immediately report broken hand...,no_rule_flag,no_rule_flag,5.00
8,en,The Employee shall accurately report project p...,The employee must accurately report project st...,no_rule_flag,no_rule_flag,5.00
9,en,User data may be used to improve account manag...,User data may be used to improve account manag...,no_rule_flag,no_rule_flag,9.85


In [ ]:
from pathlib import Path
import pandas as pd

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

for name in [
    "JUDGEBERT_LMP_dataset_v1.csv",
    "legal_simplification_pairs_en_v1.csv",
    "JUDGEBERT_Hindi_simplification_pilot_10_completed.csv",
    "hindi_original_candidates_v1.csv",
]:
    path = project / name
    print(f"\n--- {name} ---")

    if path.exists():
        temp = pd.read_csv(path)
        print("Rows:", len(temp))
        print("Columns:", temp.columns.tolist())
        print(temp.head(2).to_string(index=False, max_colwidth=70))
    else:
        print("File not found")


--- JUDGEBERT_LMP_dataset_v1.csv ---
Rows: 5687
Columns: ['original_text', 'simplified_text', 'language', 'source_type', 'label_source', 'preservation_label', 'pair_id', 'human_score_1_to_10', 'review_status', 'split']
                                                  original_text                                                    simplified_text language              source_type           label_source  preservation_label   pair_id  human_score_1_to_10            review_status      split
The Insurer may require repair estimates before claim approval. The insurer may require repair estimates before approving a claim.       en paired_source_unverified source_pair_weak_label                   1 EN_000000                  NaN needs_human_legal_review      train
    The Landlord may enter the Premises with reasonable notice.        The landlord may enter the property with reasonable notice.       en paired_source_unverified source_pair_weak_label                   1 EN_000001             

In [ ]:
import pandas as pd
from pathlib import Path
import re

PROJECT = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
PROJECT.mkdir(parents=True, exist_ok=True)

FILES = {
    "master": PROJECT / "JUDGEBERT_LMP_dataset_v1.csv",
    "english": PROJECT / "legal_simplification_pairs_en_v1.csv",
    "hindi_pilot": PROJECT / "JUDGEBERT_Hindi_simplification_pilot_10_completed.csv",
    "hindi_candidates": PROJECT / "hindi_original_candidates_v1.csv",
}

def clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

records = []

def add_rows(path, source_name, language_hint=None):
    if not path.exists():
        print("Skipping missing file:", path.name)
        return

    df = pd.read_csv(path, encoding="utf-8-sig")
    print(f"Reading {path.name}: {len(df)} rows")

    for _, row in df.iterrows():
        original = clean_text(row.get("original_text", row.get("original", "")))
        simplified = clean_text(row.get("simplified_text", row.get("simplified", "")))

        language = clean_text(row.get("language", language_hint or "")).lower()
        if language in ["english"]:
            language = "en"
        elif language in ["hindi"]:
            language = "hi"

        # Do not create a pair when either side is missing.
        complete = bool(original and simplified)
        same_text = complete and original.casefold() == simplified.casefold()

        score = pd.to_numeric(
            pd.Series([row.get("human_score_1_to_10",
                               row.get("lmp_score_1_to_10", None))]),
            errors="coerce"
        ).iloc[0]

        label = row.get("preservation_label", None)
        label = clean_text(label).lower()

        # Do not promote similarity/model outputs to human labels.
        human_score = score if (
            clean_text(row.get("score_status", "")).lower()
            in ["human_reviewed", "human_verified", "expert_reviewed"]
        ) else None

        records.append({
            "original_text": original,
            "simplified_text": simplified,
            "language": language,
            "source_file": path.name,
            "source_type": source_name,
            "pair_complete": complete,
            "identical_text": same_text,
            "preservation_label": label if label in [
                "preserved", "changed", "1", "0"
            ] else "",
            "human_score_1_to_10": human_score,
            "review_status": "needs_review",
        })

# Master first; then add sources for missing examples.
add_rows(FILES["master"], "existing_master")
add_rows(FILES["english"], "english_simplification_source", "en")
add_rows(FILES["hindi_pilot"], "hindi_pilot", "hi")
add_rows(FILES["hindi_candidates"], "hindi_source_candidates", "hi")

master = pd.DataFrame(records)

# Normalize common label formats without inventing labels.
master["preservation_label"] = master["preservation_label"].replace({
    "1": "preserved",
    "0": "changed",
})

# Remove duplicate complete pairs, keeping the first source occurrence.
master["pair_key"] = (
    master["language"].astype(str) + "||" +
    master["original_text"].str.casefold() + "||" +
    master["simplified_text"].str.casefold()
)
complete = master[master["pair_complete"]].copy()
complete = complete.drop_duplicates("pair_key", keep="first")
incomplete = master[~master["pair_complete"]].copy()

# Keep missing-simplification candidates separate from usable pairs.
complete = complete.drop(columns=["pair_key"])
incomplete = incomplete.drop(columns=["pair_key"])

# Split only complete examples; scores and labels remain unchanged.
complete["split"] = ""
for lang, indexes in complete.groupby("language").groups.items():
    idx = list(indexes)
    n = len(idx)
    if n >= 5:
        complete.loc[idx, "split"] = "train"
        complete.loc[idx[-max(1, round(n * 0.1)):], "split"] = "test"
        n_val = max(1, round(n * 0.1))
        val_idx = idx[-max(2, round(n * 0.2)):-max(1, round(n * 0.1))]
        if val_idx:
            complete.loc[val_idx, "split"] = "validation"
    else:
        complete.loc[idx, "split"] = "needs_manual_split"

complete_path = PROJECT / "JUDGEBERT_master_pairs_v1.csv"
candidate_path = PROJECT / "JUDGEBERT_incomplete_candidates_v1.csv"
audit_path = PROJECT / "JUDGEBERT_master_dataset_audit_v1.csv"

complete.to_csv(complete_path, index=False, encoding="utf-8-sig")
incomplete.to_csv(candidate_path, index=False, encoding="utf-8-sig")

audit = pd.DataFrame([
    {"metric": "all_source_rows", "value": len(master)},
    {"metric": "complete_pairs_after_deduplication", "value": len(complete)},
    {"metric": "incomplete_candidates", "value": len(incomplete)},
    {"metric": "identical_pairs", "value": int(complete["identical_text"].sum())},
    {"metric": "human_verified_scores", "value": int(complete["human_score_1_to_10"].notna().sum())},
])
for lang, count in complete["language"].value_counts(dropna=False).items():
    audit.loc[len(audit)] = {"metric": f"complete_pairs_language_{lang}", "value": int(count)}

audit.to_csv(audit_path, index=False, encoding="utf-8-sig")

print("\n=== MASTER DATASET SUMMARY ===")
print(audit.to_string(index=False))
print("\nLabel counts:")
print(complete["preservation_label"].replace("", "unlabelled").value_counts())
print("\nSplit counts:")
print(complete["split"].value_counts())
print("\nSaved files:")
print(complete_path)
print(candidate_path)
print(audit_path)

display(complete.head(10))

Reading JUDGEBERT_LMP_dataset_v1.csv: 5687 rows
Reading legal_simplification_pairs_en_v1.csv: 6686 rows
Reading JUDGEBERT_Hindi_simplification_pilot_10_completed.csv: 10 rows
Reading hindi_original_candidates_v1.csv: 1639 rows

=== MASTER DATASET SUMMARY ===
                            metric  value
                   all_source_rows  14022
complete_pairs_after_deduplication   8165
             incomplete_candidates   1640
                   identical_pairs      0
             human_verified_scores      0
        complete_pairs_language_en   8154
        complete_pairs_language_hi     11

Label counts:
preservation_label
preserved     4217
unlabelled    2478
changed       1470
Name: count, dtype: int64

Split counts:
split
train         6532
validation     817
test           816
Name: count, dtype: int64

Saved files:
/content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_master_pairs_v1.csv
/content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_incomplete_candidates_v1.csv
/content/drive/MyDrive/JUDG

,original_text,simplified_text,language,source_file,source_type,pair_complete,identical_text,preservation_label,human_score_1_to_10,review_status,split
0,The Insurer may require repair estimates befor...,The insurer may require repair estimates befor...,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,None,needs_review,train
1,The Landlord may enter the Premises with reaso...,The landlord may enter the property with reaso...,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,None,needs_review,train
2,The Tenant shall notify the Landlord of extend...,The tenant must inform the landlord about long...,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,None,needs_review,train
3,The Consumer shall ensure that all promotional...,You must provide proof if the company asks for...,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,None,needs_review,train
4,The Insurer may deny claims that do not meet p...,The insurer may reject claims that do not meet...,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,None,needs_review,train
5,The Company may revoke access to licensed soft...,The company may remove access to licensed soft...,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,None,needs_review,train
6,The Company may limit access to premium features.,The company may restrict access to premium fea...,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,None,needs_review,train
7,The Tenant shall report broken handrails immed...,The tenant must immediately report broken hand...,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,None,needs_review,train
8,The Employee shall accurately report project p...,The employee must accurately report project st...,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,None,needs_review,train
9,User data may be used to improve account manag...,User data may be used to improve account manag...,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,None,needs_review,train


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
path = project / "JUDGEBERT_master_pairs_v1.csv"

df = pd.read_csv(path, encoding="utf-8-sig")

print("Rows:", len(df))
print("\nLanguage × label:")
print(pd.crosstab(
    df["language"],
    df["preservation_label"].fillna("").replace("", "unlabelled"),
    margins=True
))

print("\nLanguage × source:")
print(pd.crosstab(df["language"], df["source_type"]))

print("\nLabel provenance:")
print(pd.crosstab(
    df["source_type"],
    df["preservation_label"].fillna("").replace("", "unlabelled")
))

print("\nExamples marked changed:")
cols = ["language", "original_text", "simplified_text",
        "preservation_label", "source_type"]
display(df[df["preservation_label"] == "changed"][cols].head(10))

print("\nExamples marked preserved:")
display(df[df["preservation_label"] == "preserved"][cols].head(10))

Rows: 8165

Language × label:
preservation_label  changed  preserved  unlabelled   All
language                                                
en                     1468       4208        2478  8154
hi                        2          9           0    11
All                    1470       4217        2478  8165

Language × source:
source_type  english_simplification_source  existing_master
language                                                   
en                                    2478             5676
hi                                       0               11

Label provenance:
preservation_label             changed  preserved  unlabelled
source_type                                                  
english_simplification_source        0          0        2478
existing_master                   1470       4217           0

Examples marked changed:


,language,original_text,simplified_text,preservation_label,source_type
4217,en,The Tenant shall notify the Landlord of extend...,The tenant may inform the landlord about long ...,changed,existing_master
4218,en,The Consumer shall ensure that all promotional...,You may provide proof if the company asks for ...,changed,existing_master
4219,en,The Insurer may deny claims that do not meet p...,The insurer may reject claims that do meet pol...,changed,existing_master
4220,en,The Tenant shall report broken handrails immed...,The tenant may immediately report broken handr...,changed,existing_master
4221,en,The Employee shall accurately report project p...,The employee may accurately report project sta...,changed,existing_master
4222,en,The Receiving Party shall protect confidential...,The receiving party may prevent unauthorized c...,changed,existing_master
4223,en,The Purchaser shall comply with all lawful obl...,The buyer may follow legal requirements impose...,changed,existing_master
4224,en,The Policyholder shall take reasonable steps t...,The policyholder may take reasonable steps to ...,changed,existing_master
4225,en,Confidential Information shall be stored in re...,Confidential information may be stored in syst...,changed,existing_master
4226,en,The Vendor shall provide available information...,The seller may share known details about maint...,changed,existing_master



Examples marked preserved:


,language,original_text,simplified_text,preservation_label,source_type
0,en,The Insurer may require repair estimates befor...,The insurer may require repair estimates befor...,preserved,existing_master
1,en,The Landlord may enter the Premises with reaso...,The landlord may enter the property with reaso...,preserved,existing_master
2,en,The Tenant shall notify the Landlord of extend...,The tenant must inform the landlord about long...,preserved,existing_master
3,en,The Consumer shall ensure that all promotional...,You must provide proof if the company asks for...,preserved,existing_master
4,en,The Insurer may deny claims that do not meet p...,The insurer may reject claims that do not meet...,preserved,existing_master
5,en,The Company may revoke access to licensed soft...,The company may remove access to licensed soft...,preserved,existing_master
6,en,The Company may limit access to premium features.,The company may restrict access to premium fea...,preserved,existing_master
7,en,The Tenant shall report broken handrails immed...,The tenant must immediately report broken hand...,preserved,existing_master
8,en,The Employee shall accurately report project p...,The employee must accurately report project st...,preserved,existing_master
9,en,User data may be used to improve account manag...,User data may be used to improve account manag...,preserved,existing_master


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
input_file = project / "JUDGEBERT_master_pairs_v1.csv"

df = pd.read_csv(input_file, encoding="utf-8-sig")

# Keep English rows with a usable binary label.
df["language"] = df["language"].astype(str).str.lower().str.strip()
df["preservation_label"] = (
    df["preservation_label"].astype(str).str.lower().str.strip()
)

df = df[
    (df["language"] == "en") &
    (df["preservation_label"].isin(["preserved", "changed"]))
].copy()

# Remove empty text and exact duplicate pairs.
df["original_text"] = df["original_text"].fillna("").astype(str).str.strip()
df["simplified_text"] = df["simplified_text"].fillna("").astype(str).str.strip()

df = df[
    (df["original_text"].str.len() > 0) &
    (df["simplified_text"].str.len() > 0)
].copy()

df = df.drop_duplicates(
    subset=["original_text", "simplified_text"]
).reset_index(drop=True)

# Binary target: preserved = 1, changed = 0.
df["label"] = df["preservation_label"].map({
    "preserved": 1,
    "changed": 0
})

# Use pair IDs as grouping keys where available.
if "pair_id" in df.columns:
    groups = df["pair_id"].fillna("").astype(str)
    groups = groups.where(groups.str.len() > 0, df.index.astype(str))
else:
    groups = df.index.astype(str)

# Group-aware 80/10/10 split.
split1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, hold_idx = next(
    split1.split(df, df["label"], groups=groups)
)

hold = df.iloc[hold_idx].copy()
hold_groups = groups.iloc[hold_idx]

split2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_rel, test_rel = next(
    split2.split(hold, hold["label"], groups=hold_groups)
)

train = df.iloc[train_idx].copy()
validation = hold.iloc[val_rel].copy()
test = hold.iloc[test_rel].copy()

# Save separate files.
train.to_csv(project / "lmp_en_train_v1.csv", index=False)
validation.to_csv(project / "lmp_en_validation_v1.csv", index=False)
test.to_csv(project / "lmp_en_test_v1.csv", index=False)

print("PRELIMINARY ENGLISH DATASET")
print("Total labelled pairs:", len(df))
print("\nLabel counts:")
print(df["label"].value_counts().sort_index())
print("\nTrain:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))
print("\nTrain label counts:")
print(train["label"].value_counts().sort_index())
print("\nValidation label counts:")
print(validation["label"].value_counts().sort_index())
print("\nTest label counts:")
print(test["label"].value_counts().sort_index())

print("\nSaved files:")
print(project / "lmp_en_train_v1.csv")
print(project / "lmp_en_validation_v1.csv")
print(project / "lmp_en_test_v1.csv")

AttributeError: 'Index' object has no attribute 'iloc'

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
input_file = project / "JUDGEBERT_master_pairs_v1.csv"

df = pd.read_csv(input_file, encoding="utf-8-sig")

# Keep English pairs with existing binary labels.
df["language"] = df["language"].astype(str).str.lower().str.strip()
df["preservation_label"] = (
    df["preservation_label"].astype(str).str.lower().str.strip()
)

df = df[
    (df["language"] == "en") &
    (df["preservation_label"].isin(["preserved", "changed"]))
].copy()

df["original_text"] = df["original_text"].fillna("").astype(str).str.strip()
df["simplified_text"] = df["simplified_text"].fillna("").astype(str).str.strip()

df = df[
    (df["original_text"] != "") &
    (df["simplified_text"] != "")
].drop_duplicates(
    subset=["original_text", "simplified_text"]
).reset_index(drop=True)

df["label"] = df["preservation_label"].map({
    "preserved": 1,
    "changed": 0
}).astype(int)

# Use one group per pair ID to reduce data leakage.
if "pair_id" in df.columns:
    groups = df["pair_id"].fillna("").astype(str)
    groups = groups.where(groups.str.len() > 0, df.index.astype(str))
else:
    groups = pd.Series(df.index.astype(str), index=df.index)

# Split into 80% training and 20% holdout.
split1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, hold_idx = next(
    split1.split(df, df["label"], groups=groups)
)

train = df.iloc[train_idx].copy()
hold = df.iloc[hold_idx].copy()

# Split holdout into validation and test sets.
hold_groups = groups.iloc[hold_idx].reset_index(drop=True)
hold = hold.reset_index(drop=True)

split2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(
    split2.split(hold, hold["label"], groups=hold_groups)
)

validation = hold.iloc[val_idx].copy()
test = hold.iloc[test_idx].copy()

# Save all three datasets.
train.to_csv(project / "lmp_en_train_v1.csv", index=False)
validation.to_csv(project / "lmp_en_validation_v1.csv", index=False)
test.to_csv(project / "lmp_en_test_v1.csv", index=False)

print("DATASET SPLIT COMPLETE")
print("Total labelled English pairs:", len(df))
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

print("\nOverall label counts (1=preserved, 0=changed):")
print(df["label"].value_counts().sort_index())

print("\nTrain labels:")
print(train["label"].value_counts().sort_index())
print("\nValidation labels:")
print(validation["label"].value_counts().sort_index())
print("\nTest labels:")
print(test["label"].value_counts().sort_index())

print("\nSaved files:")
print(project / "lmp_en_train_v1.csv")
print(project / "lmp_en_validation_v1.csv")
print(project / "lmp_en_test_v1.csv")

DATASET SPLIT COMPLETE
Total labelled English pairs: 5676
Train: 4540
Validation: 568
Test: 568

Overall label counts (1=preserved, 0=changed):
label
0    1468
1    4208
Name: count, dtype: int64

Train labels:
label
0    1170
1    3370
Name: count, dtype: int64

Validation labels:
label
0    154
1    414
Name: count, dtype: int64

Test labels:
label
0    144
1    424
Name: count, dtype: int64

Saved files:
/content/drive/MyDrive/JUDGEBERT_LMP/lmp_en_train_v1.csv
/content/drive/MyDrive/JUDGEBERT_LMP/lmp_en_validation_v1.csv
/content/drive/MyDrive/JUDGEBERT_LMP/lmp_en_test_v1.csv


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import joblib

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

train = pd.read_csv(project / "lmp_en_train_v1.csv")
val = pd.read_csv(project / "lmp_en_validation_v1.csv")
test = pd.read_csv(project / "lmp_en_test_v1.csv")

# Represent both texts together so the classifier can compare the pair.
def make_pair_text(df):
    return (
        "ORIGINAL: " + df["original_text"].fillna("").astype(str)
        + " SIMPLIFIED: " + df["simplified_text"].fillna("").astype(str)
    )

X_train = make_pair_text(train)
y_train = train["label"].astype(int)

X_val = make_pair_text(val)
y_val = val["label"].astype(int)

X_test = make_pair_text(test)
y_test = test["label"].astype(int)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=50000,
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Validation results
val_pred = model.predict(X_val)
print("VALIDATION RESULTS")
print("Accuracy:", round(accuracy_score(y_val, val_pred), 4))
print("F1:", round(f1_score(y_val, val_pred, zero_division=0), 4))

# Final test results
test_pred = model.predict(X_test)
print("\nTEST RESULTS")
print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, test_pred, zero_division=0), 4))

print("\nClassification report:")
print(classification_report(
    y_test, test_pred,
    target_names=["Changed", "Preserved"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))

# Save the baseline model
joblib.dump(model, project / "judgebert_lmp_baseline_tfidf.joblib")
print("\nModel saved:", project / "judgebert_lmp_baseline_tfidf.joblib")

VALIDATION RESULTS
Accuracy: 0.9261
F1: 0.9499

TEST RESULTS
Accuracy: 0.9437
Precision: 0.9645
Recall: 0.9599
F1: 0.9622

Classification report:
              precision    recall  f1-score   support

     Changed       0.88      0.90      0.89       144
   Preserved       0.96      0.96      0.96       424

    accuracy                           0.94       568
   macro avg       0.92      0.93      0.93       568
weighted avg       0.94      0.94      0.94       568

Confusion matrix:
[[129  15]
 [ 17 407]]

Model saved: /content/drive/MyDrive/JUDGEBERT_LMP/judgebert_lmp_baseline_tfidf.joblib


In [ ]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import confusion_matrix

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

test = pd.read_csv(project / "lmp_en_test_v1.csv")
model = joblib.load(project / "judgebert_lmp_baseline_tfidf.joblib")

def make_pair_text(df):
    return (
        "ORIGINAL: " + df["original_text"].fillna("").astype(str)
        + " SIMPLIFIED: " + df["simplified_text"].fillna("").astype(str)
    )

test["predicted_label"] = model.predict(make_pair_text(test))
test["correct"] = test["label"].astype(int) == test["predicted_label"]

print("Incorrect predictions:", int((~test["correct"]).sum()))
print("\nIncorrect examples:\n")

display(test.loc[
    ~test["correct"],
    ["original_text", "simplified_text", "label", "predicted_label"]
].head(15))

print("\nSaved baseline remains unchanged.")

Incorrect predictions: 32

Incorrect examples:



,original_text,simplified_text,label,predicted_label
44,Any statements concerning expected rental occu...,Predictions about rental occupancy are not gua...,1,0
89,Coverage may exclude losses caused by normal w...,Coverage may not include normal wear and tear.,1,0
104,Coverage may exclude losses caused by wear and...,Coverage may not include wear and deterioration.,1,0
140,Coverage may exclude damage caused by neglect.,Coverage may not include damage caused by negl...,1,0
166,Coverage may exclude losses arising from breac...,Coverage may not include losses resulting from...,1,0
177,Business travel may be required.,Work-related travel may be required.,1,0
181,Harassment of any kind is prohibited.,Any form of harassment is not allowed.,1,0
232,Coverage may exclude losses resulting from pol...,Coverage may not include losses caused by poli...,1,0
249,The Receiving Party shall use Confidential Inf...,The receiving party may use confidential infor...,1,0
264,The Vendor shall ensure that all disclosed acc...,The seller must ensure promised access rights ...,1,0



Saved baseline remains unchanged.


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
df = pd.read_csv(project / "lmp_en_train_v1.csv")

# Terms that can affect legal meaning
patterns = {
    "modal_change": r"\b(shall|must|may|might|should|will|can)\b",
    "negation": r"\b(not|no|never|neither|nor|unless|without)\b",
    "numbers": r"\b\d+(?:\.\d+)?%?\b",
    "rights_duties": r"\b(right|duty|obligation|liable|liability|entitled|prohibited|required)\b"
}

for name, pattern in patterns.items():
    original_has = df["original_text"].str.contains(
        pattern, case=False, regex=True, na=False
    )
    simplified_has = df["simplified_text"].str.contains(
        pattern, case=False, regex=True, na=False
    )

    print(f"\n{name}:")
    print("Rows containing term in either text:",
          int((original_has | simplified_has).sum()))
    print("Rows where presence differs:",
          int((original_has != simplified_has).sum()))

# Show potential risk cases labelled preserved.
modal_pattern = patterns["modal_change"]
original_modal = df["original_text"].str.lower().str.findall(modal_pattern)
simplified_modal = df["simplified_text"].str.lower().str.findall(modal_pattern)

risk = df[
    (df["label"] == 1) &
    (original_modal != simplified_modal)
]

print("\nPreserved-labelled pairs with different modal-term lists:", len(risk))
display(risk[[
    "original_text", "simplified_text", "label"
]].head(15))

/tmp/ipykernel_3238/417578144.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  original_has = df["original_text"].str.contains(
/tmp/ipykernel_3238/417578144.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  simplified_has = df["simplified_text"].str.contains(



modal_change:
Rows containing term in either text: 4027
Rows where presence differs: 511

negation:
Rows containing term in either text: 931
Rows where presence differs: 339

numbers:
Rows containing term in either text: 194
Rows where presence differs: 95

rights_duties:
Rows containing term in either text: 290
Rows where presence differs: 185

Preserved-labelled pairs with different modal-term lists: 1841


,original_text,simplified_text,label
2,The Tenant shall notify the Landlord of extend...,The tenant must inform the landlord about long...,1
3,The Consumer shall ensure that all promotional...,You must provide proof if the company asks for...,1
7,The Employee shall accurately report project p...,The employee must accurately report project st...,1
9,The Receiving Party shall protect confidential...,The receiving party must prevent unauthorized ...,1
10,The Purchaser shall comply with all lawful obl...,The buyer must follow legal requirements impos...,1
11,The Policyholder shall take reasonable steps t...,The policyholder must take reasonable steps to...,1
16,Confidential Information shall be stored in re...,Confidential information must be stored in sys...,1
17,The Vendor shall provide available information...,The seller must share known details about main...,1
18,The Policyholder shall notify the Insurer of e...,The policyholder must report employment change...,1
19,Confidential Information shall include strateg...,Confidential information includes strategic pa...,1


In [ ]:
# Show suspicious pairs for manual review.
# This does NOT change any existing labels.

review = risk[
    ["original_text", "simplified_text", "label"]
].copy()

review["review_reason"] = "Modal terms differ; inspect legal meaning"

review = review.reset_index(drop=True)
review.insert(0, "review_id", range(1, len(review) + 1))

display(review.head(25))

review.to_csv(
    "/content/lmp_modal_review_queue.csv",
    index=False
)

print("Saved review queue:", len(review), "pairs")

,review_id,original_text,simplified_text,label,review_reason
0,1,The Tenant shall notify the Landlord of extend...,The tenant must inform the landlord about long...,1,Modal terms differ; inspect legal meaning
1,2,The Consumer shall ensure that all promotional...,You must provide proof if the company asks for...,1,Modal terms differ; inspect legal meaning
2,3,The Employee shall accurately report project p...,The employee must accurately report project st...,1,Modal terms differ; inspect legal meaning
3,4,The Receiving Party shall protect confidential...,The receiving party must prevent unauthorized ...,1,Modal terms differ; inspect legal meaning
4,5,The Purchaser shall comply with all lawful obl...,The buyer must follow legal requirements impos...,1,Modal terms differ; inspect legal meaning
5,6,The Policyholder shall take reasonable steps t...,The policyholder must take reasonable steps to...,1,Modal terms differ; inspect legal meaning
6,7,Confidential Information shall be stored in re...,Confidential information must be stored in sys...,1,Modal terms differ; inspect legal meaning
7,8,The Vendor shall provide available information...,The seller must share known details about main...,1,Modal terms differ; inspect legal meaning
8,9,The Policyholder shall notify the Insurer of e...,The policyholder must report employment change...,1,Modal terms differ; inspect legal meaning
9,10,Confidential Information shall include strateg...,Confidential information includes strategic pa...,1,Modal terms differ; inspect legal meaning


Saved review queue: 1841 pairs


In [ ]:
import pandas as pd
from IPython.display import display

review = pd.read_csv("/content/lmp_modal_review_queue.csv")

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

for i, row in review.head(5).iterrows():
    print(f"\n{'='*80}")
    print("Review ID:", row["review_id"])
    print("Current label:", row["label"])
    print("\nORIGINAL:")
    print(row["original_text"])
    print("\nSIMPLIFIED:")
    print(row["simplified_text"])


Review ID: 1
Current label: 1

ORIGINAL:
The Tenant shall notify the Landlord of extended utility interruptions.

SIMPLIFIED:
The tenant must inform the landlord about long utility outages.

Review ID: 2
Current label: 1

ORIGINAL:
The Consumer shall ensure that all promotional claims submitted are supported by documentation reasonably requested by the Company.

SIMPLIFIED:
You must provide proof if the company asks for evidence related to a promotional claim.

Review ID: 3
Current label: 1

ORIGINAL:
The Employee shall accurately report project progress.

SIMPLIFIED:
The employee must accurately report project status.

Review ID: 4
Current label: 1

ORIGINAL:
The Receiving Party shall protect confidential information from unauthorized replication.

SIMPLIFIED:
The receiving party must prevent unauthorized copying of confidential information.

Review ID: 5
Current label: 1

ORIGINAL:
The Purchaser shall comply with all lawful obligations imposed by local authorities after transfer.

S

In [ ]:
import pandas as pd
import re

# Load the existing master dataset
path = "/content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_master_pairs_v1.csv"
df = pd.read_csv(path)

# Normalize text for checks
original = df["original_text"].fillna("").str.lower()
simplified = df["simplified_text"].fillna("").str.lower()

# Flag potentially important legal meaning changes
checks = {
    "modal_terms": r"\b(shall|must|may|might|can|cannot|will|should)\b",
    "negation": r"\b(not|no|never|neither|nor|without|unless)\b",
    "numbers": r"\b\d+(?:\.\d+)?\b",
    "parties": r"\b(tenant|landlord|buyer|seller|employee|employer|insurer|insured|company|consumer|purchaser|vendor|receiving party)\b",
}

for name, pattern in checks.items():
    orig_found = original.str.findall(pattern).apply(set)
    simp_found = simplified.str.findall(pattern).apply(set)
    df[f"review_{name}"] = [
        a != b for a, b in zip(orig_found, simp_found)
    ]

review_cols = [f"review_{name}" for name in checks]
df["needs_manual_review"] = df[review_cols].any(axis=1)

print("Total pairs:", len(df))
print("Flagged for review:", int(df["needs_manual_review"].sum()))
print("Not flagged:", int((~df["needs_manual_review"]).sum()))

# Save a separate copy; do not overwrite the master
out = "/content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_pairs_review_flags_v1.csv"
df.to_csv(out, index=False)
print("Saved:", out)

display(df.loc[df["needs_manual_review"],
    ["original_text", "simplified_text", "preservation_label"] +
    review_cols
].head(10))

Total pairs: 8165
Flagged for review: 5688
Not flagged: 2477
Saved: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_pairs_review_flags_v1.csv


,original_text,simplified_text,preservation_label,review_modal_terms,review_negation,review_numbers,review_parties
2,The Tenant shall notify the Landlord of extended utility interruptions.,The tenant must inform the landlord about long utility outages.,preserved,True,False,False,False
3,The Consumer shall ensure that all promotional claims submitted are supported by documentation reasonably requested by the Company.,You must provide proof if the company asks for evidence related to a promotional claim.,preserved,True,False,False,True
7,The Tenant shall report broken handrails immediately.,The tenant must immediately report broken handrails.,preserved,True,False,False,False
8,The Employee shall accurately report project progress.,The employee must accurately report project status.,preserved,True,False,False,False
10,The Receiving Party shall protect confidential information from unauthorized replication.,The receiving party must prevent unauthorized copying of confidential information.,preserved,True,False,False,False
11,The Purchaser shall comply with all lawful obligations imposed by local authorities after transfer.,The buyer must follow legal requirements imposed by local government after ownership transfer.,preserved,True,False,False,True
12,The Policyholder shall take reasonable steps to prevent further damage.,The policyholder must take reasonable steps to prevent more damage.,preserved,True,False,False,False
17,Confidential Information shall be stored in restricted-access systems.,Confidential information must be stored in systems with restricted access.,preserved,True,False,False,False
18,The Vendor shall provide available information regarding maintenance obligations for shared infrastructure.,The seller must share known details about maintaining shared facilities.,preserved,True,False,False,True
19,The Policyholder shall notify the Insurer of employment changes when relevant.,The policyholder must report employment changes when required.,preserved,True,False,False,True


In [ ]:
import pandas as pd

project = "/content/drive/MyDrive/JUDGEBERT_LMP"

df = pd.read_csv(f"{project}/JUDGEBERT_pairs_review_flags_v1.csv")

# Keep pairs that need review in a separate file
review = df[df["needs_manual_review"]].copy()

# Add blank human-review fields
review["human_review_label"] = ""
review["human_review_notes"] = ""
review["review_status"] = "pending"

out = f"{project}/JUDGEBERT_pairs_manual_review_v1.csv"
review.to_csv(out, index=False)

print("Total dataset pairs:", len(df))
print("Pairs needing review:", len(review))
print("Saved review file:", out)

display(review[
    ["original_text", "simplified_text", "preservation_label",
     "human_review_label", "human_review_notes", "review_status"]
].head(10))

Total dataset pairs: 8165
Pairs needing review: 5688
Saved review file: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_pairs_manual_review_v1.csv


,original_text,simplified_text,preservation_label,human_review_label,human_review_notes,review_status
2,The Tenant shall notify the Landlord of extended utility interruptions.,The tenant must inform the landlord about long utility outages.,preserved,,,pending
3,The Consumer shall ensure that all promotional claims submitted are supported by documentation reasonably requested by the Company.,You must provide proof if the company asks for evidence related to a promotional claim.,preserved,,,pending
7,The Tenant shall report broken handrails immediately.,The tenant must immediately report broken handrails.,preserved,,,pending
8,The Employee shall accurately report project progress.,The employee must accurately report project status.,preserved,,,pending
10,The Receiving Party shall protect confidential information from unauthorized replication.,The receiving party must prevent unauthorized copying of confidential information.,preserved,,,pending
11,The Purchaser shall comply with all lawful obligations imposed by local authorities after transfer.,The buyer must follow legal requirements imposed by local government after ownership transfer.,preserved,,,pending
12,The Policyholder shall take reasonable steps to prevent further damage.,The policyholder must take reasonable steps to prevent more damage.,preserved,,,pending
17,Confidential Information shall be stored in restricted-access systems.,Confidential information must be stored in systems with restricted access.,preserved,,,pending
18,The Vendor shall provide available information regarding maintenance obligations for shared infrastructure.,The seller must share known details about maintaining shared facilities.,preserved,,,pending
19,The Policyholder shall notify the Insurer of employment changes when relevant.,The policyholder must report employment changes when required.,preserved,,,pending


In [ ]:
import pandas as pd

project = "/content/drive/MyDrive/JUDGEBERT_LMP"
path = f"{project}/JUDGEBERT_master_pairs_v1.csv"

df = pd.read_csv(path)

print("TOTAL ROWS:", len(df))
print("\nLANGUAGE COUNTS:")
print(df["language"].value_counts(dropna=False))

print("\nLABEL COUNTS:")
print(df["preservation_label"].value_counts(dropna=False))

print("\nLANGUAGE × LABEL:")
print(pd.crosstab(
    df["language"],
    df["preservation_label"],
    dropna=False
))

print("\nMISSING VALUES:")
print(df[[
    "original_text",
    "simplified_text",
    "language",
    "preservation_label"
]].isna().sum())

print("\nSOURCE COUNTS:")
print(df["source_type"].value_counts(dropna=False))

print("\nEXAMPLE PAIRS:")
display(df[[
    "language",
    "original_text",
    "simplified_text",
    "preservation_label"
]].head(10))

TOTAL ROWS: 8165

LANGUAGE COUNTS:
language
en    8154
hi      11
Name: count, dtype: int64

LABEL COUNTS:
preservation_label
preserved    4217
NaN          2478
changed      1470
Name: count, dtype: int64

LANGUAGE × LABEL:
preservation_label  changed  preserved   NaN
language                                    
en                     1468       4208  2478
hi                        2          9     0

MISSING VALUES:
original_text            0
simplified_text          0
language                 0
preservation_label    2478
dtype: int64

SOURCE COUNTS:
source_type
existing_master                  5687
english_simplification_source    2478
Name: count, dtype: int64

EXAMPLE PAIRS:


,language,original_text,simplified_text,preservation_label
0,en,The Insurer may require repair estimates before claim approval.,The insurer may require repair estimates before approving a claim.,preserved
1,en,The Landlord may enter the Premises with reasonable notice.,The landlord may enter the property with reasonable notice.,preserved
2,en,The Tenant shall notify the Landlord of extended utility interruptions.,The tenant must inform the landlord about long utility outages.,preserved
3,en,The Consumer shall ensure that all promotional claims submitted are supported by documentation reasonably requested by the Company.,You must provide proof if the company asks for evidence related to a promotional claim.,preserved
4,en,The Insurer may deny claims that do not meet policy requirements.,The insurer may reject claims that do not meet policy requirements.,preserved
5,en,The Company may revoke access to licensed software.,The company may remove access to licensed software.,preserved
6,en,The Company may limit access to premium features.,The company may restrict access to premium features.,preserved
7,en,The Tenant shall report broken handrails immediately.,The tenant must immediately report broken handrails.,preserved
8,en,The Employee shall accurately report project progress.,The employee must accurately report project status.,preserved
9,en,User data may be used to improve account management services.,User data may be used to improve account management.,preserved


In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_master_pairs_v1.csv"
df = pd.read_csv(path)

print("TOTAL ROWS:", len(df))

print("\nLANGUAGE COUNTS")
print(df["language"].value_counts(dropna=False))

print("\nPRESERVATION LABEL COUNTS")
print(df["preservation_label"].value_counts(dropna=False))

print("\nLANGUAGE × LABEL")
print(pd.crosstab(
    df["language"],
    df["preservation_label"],
    dropna=False
))

print("\nMISSING TEXT VALUES")
print(df[["original_text", "simplified_text"]].isna().sum())

print("\nSOURCE TYPES")
print(df["source_type"].value_counts(dropna=False))

print("\nHUMAN SCORE COUNTS")
if "human_score_1_to_10" in df.columns:
    print(df["human_score_1_to_10"].notna().sum(),
          "rows have a human score out of", len(df))

TOTAL ROWS: 8165

LANGUAGE COUNTS
language
en    8154
hi      11
Name: count, dtype: int64

PRESERVATION LABEL COUNTS
preservation_label
preserved    4217
NaN          2478
changed      1470
Name: count, dtype: int64

LANGUAGE × LABEL
preservation_label  changed  preserved   NaN
language                                    
en                     1468       4208  2478
hi                        2          9     0

MISSING TEXT VALUES
original_text      0
simplified_text    0
dtype: int64

SOURCE TYPES
source_type
existing_master                  5687
english_simplification_source    2478
Name: count, dtype: int64

HUMAN SCORE COUNTS
0 rows have a human score out of 8165


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

df = pd.read_csv(project / "JUDGEBERT_master_pairs_v1.csv")

# Keep complete English pairs with existing binary labels
clean = df[
    (df["language"] == "en") &
    (df["preservation_label"].isin(["preserved", "changed"])) &
    (df["original_text"].notna()) &
    (df["simplified_text"].notna())
].copy()

# Remove blank text and exact duplicate pairs
clean["original_text"] = clean["original_text"].astype(str).str.strip()
clean["simplified_text"] = clean["simplified_text"].astype(str).str.strip()

clean = clean[
    (clean["original_text"] != "") &
    (clean["simplified_text"] != "")
]

clean = clean.drop_duplicates(
    subset=["original_text", "simplified_text"]
).reset_index(drop=True)

# Map labels to binary values
clean["label"] = clean["preservation_label"].map({
    "changed": 0,
    "preserved": 1
})

output = project / "JUDGEBERT_english_binary_candidates_v1.csv"
clean.to_csv(output, index=False)

print("Saved:", output)
print("Total labelled English pairs:", len(clean))
print("\nLabel counts:")
print(clean["label"].value_counts().sort_index())
print("\nLabel meanings: 0 = changed, 1 = preserved")
print("\nSample pairs:")
display(clean[[
    "original_text",
    "simplified_text",
    "preservation_label",
    "label"
]].head(10))

Saved: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_english_binary_candidates_v1.csv
Total labelled English pairs: 5676

Label counts:
label
0    1468
1    4208
Name: count, dtype: int64

Label meanings: 0 = changed, 1 = preserved

Sample pairs:


,original_text,simplified_text,preservation_label,label
0,The Insurer may require repair estimates before claim approval.,The insurer may require repair estimates before approving a claim.,preserved,1
1,The Landlord may enter the Premises with reasonable notice.,The landlord may enter the property with reasonable notice.,preserved,1
2,The Tenant shall notify the Landlord of extended utility interruptions.,The tenant must inform the landlord about long utility outages.,preserved,1
3,The Consumer shall ensure that all promotional claims submitted are supported by documentation reasonably requested by the Company.,You must provide proof if the company asks for evidence related to a promotional claim.,preserved,1
4,The Insurer may deny claims that do not meet policy requirements.,The insurer may reject claims that do not meet policy requirements.,preserved,1
5,The Company may revoke access to licensed software.,The company may remove access to licensed software.,preserved,1
6,The Company may limit access to premium features.,The company may restrict access to premium features.,preserved,1
7,The Tenant shall report broken handrails immediately.,The tenant must immediately report broken handrails.,preserved,1
8,The Employee shall accurately report project progress.,The employee must accurately report project status.,preserved,1
9,User data may be used to improve account management services.,User data may be used to improve account management.,preserved,1


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
df = pd.read_csv(project / "JUDGEBERT_english_binary_candidates_v1.csv")

checks = {
    "negation": r"\b(not|no|never|neither|nor|without|unless|except)\b",
    "numbers": r"\b\d+(?:\.\d+)?%?\b",
    "obligation": r"\b(shall|must|required to|obligated to|may|can)\b",
    "parties": r"\b(landlord|tenant|insurer|insured|company|consumer|buyer|seller|employee|employer)\b"
}

for name, pattern in checks.items():
    original_has = df["original_text"].str.contains(
        pattern, case=False, regex=True, na=False
    )
    simplified_has = df["simplified_text"].str.contains(
        pattern, case=False, regex=True, na=False
    )

    flagged = df[original_has != simplified_has]

    print(f"\n{name.upper()} DIFFERENCES: {len(flagged)}")
    display(flagged[[
        "original_text",
        "simplified_text",
        "preservation_label"
    ]].head(5))


NEGATION DIFFERENCES: 451


/tmp/ipykernel_3238/2652055042.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  original_has = df["original_text"].str.contains(
/tmp/ipykernel_3238/2652055042.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  simplified_has = df["simplified_text"].str.contains(


,original_text,simplified_text,preservation_label
24,Coverage may exclude losses caused by improper use of property.,Coverage may not include losses caused by improper property use.,preserved
76,Coverage may exclude losses caused by inadequate maintenance.,Coverage may not include losses caused by poor maintenance.,preserved
88,Coverage may exclude losses related to unlawful occupation.,Coverage may not include losses from unlawful occupation.,preserved
109,The Platform may discontinue support for features dependent upon third-party services that are no longer available.,The app can remove features if outside services needed for them stop operating.,preserved
130,Confidential Information shall not be shared with competitors.,Confidential information cannot be shared with competitors.,preserved



NUMBERS DIFFERENCES: 124


,original_text,simplified_text,preservation_label
1442,Any complaint relating to damaged goods must be reported within forty-eight hours of delivery.,You must report damaged products within 48 hours after receiving them.,preserved
3483,List the five most important quick keys on a PC,1. Control C 2. Control V 3. Control Alt Delete 4. Control Print Screen 5. Control z,preserved
3484,"According to the Tourism Satellite Account, how does inbound tourism expenditure benefit Uganda's national accounts? Value Addition","Inbound tourism expenditure is recorded as a service export. According to the data for 2022, it represented a significant portion (21.6%) of Uganda's total exports, demonstrating its value to the national economy.",preserved
3488,"Without quoting the text, explain to me why the One Mile telescope was built In 1971, Sir Martin Ryle described why, in the late 1950s, radio astronomers at MRAO decided on the construction of the new One Mile telescope: ""Our object was twofold. First we wanted to extend the range of our observations far back in time to the earliest days of the Universe, and this required a large increase in both sensitivity and resolution. With greater resolution we hoped that we might be able to draw radio maps of individual radio sources with sufficient detail to give some indication of the physical processes which brought them into being.""","The One Mile Telescope was built in the late 1950s with the goal of observing the early days of the universe. The construction of the telescope allowed to get more detailed images, allowing a better understanding of the physical phenomena at work.",preserved
3499,"""repair instead of replace"" give five points to support this argument",1) saves cost 2) reduces wastage 3) good for environment 4) promotes local employment 5) efficient use of resources,preserved



OBLIGATION DIFFERENCES: 760


... (2 outputs left out)

/tmp/ipykernel_3238/2652055042.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  original_has = df["original_text"].str.contains(



PARTIES DIFFERENCES: 429


/tmp/ipykernel_3238/2652055042.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  simplified_has = df["simplified_text"].str.contains(


,original_text,simplified_text,preservation_label
11,The Purchaser shall comply with all lawful obligations imposed by local authorities after transfer.,The buyer must follow legal requirements imposed by local government after ownership transfer.,preserved
18,The Vendor shall provide available information regarding maintenance obligations for shared infrastructure.,The seller must share known details about maintaining shared facilities.,preserved
19,The Policyholder shall notify the Insurer of employment changes when relevant.,The policyholder must report employment changes when required.,preserved
34,Any customer feedback submitted may be used internally to improve products and services.,The company may use your feedback to improve its offerings.,preserved
51,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,preserved


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

df = pd.read_csv(
    project / "JUDGEBERT_english_binary_candidates_v1.csv"
)

# Patterns that need closer review
patterns = {
    "negation": r"\b(not|no|never|neither|nor|without|unless|except)\b",
    "numbers": r"\b\d+(?:\.\d+)?%?\b",
    "obligation": r"\b(shall|must|required to|obligated to|may|can)\b",
    "parties": r"\b(landlord|tenant|insurer|insured|company|consumer|buyer|seller|employee|employer|purchaser|vendor|policyholder|cardholder)\b"
}

review = df.copy()
review["review_reasons"] = ""

for name, pattern in patterns.items():
    original_has = review["original_text"].str.contains(
        pattern, case=False, regex=True, na=False
    )
    simplified_has = review["simplified_text"].str.contains(
        pattern, case=False, regex=True, na=False
    )

    different = original_has != simplified_has
    review.loc[different, "review_reasons"] += name + "; "

# Flag pairs that may be unrelated to legal text
legal_pattern = (
    r"\b(agreement|contract|court|law|legal|claim|liability|"
    r"tenant|landlord|insurer|policy|party|obligation|shall|"
    r"must|rights|duties|purchaser|vendor|consumer)\b"
)

both_without_legal_terms = (
    ~review["original_text"].str.contains(
        legal_pattern, case=False, regex=True, na=False
    )
    &
    ~review["simplified_text"].str.contains(
        legal_pattern, case=False, regex=True, na=False
    )
)

review.loc[both_without_legal_terms, "review_reasons"] += (
    "check_domain_relevance; "
)

review = review[review["review_reasons"] != ""].copy()

out = project / "JUDGEBERT_english_review_queue_v1.csv"
review.to_csv(out, index=False)

print("Review queue saved:", out)
print("Pairs flagged for review:", len(review))
print("\nReview reasons:")
print(
    review["review_reasons"]
    .str.rstrip("; ")
    .str.split("; ")
    .explode()
    .value_counts()
)

display(review[
    ["original_text", "simplified_text",
     "preservation_label", "review_reasons"]
].head(15))

/tmp/ipykernel_3238/1545498179.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  original_has = review["original_text"].str.contains(
/tmp/ipykernel_3238/1545498179.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  simplified_has = review["simplified_text"].str.contains(
/tmp/ipykernel_3238/1545498179.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  original_has = review["original_text"].str.contains(
/tmp/ipykernel_3238/1545498179.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  simplified_has = review["simplified_text"].str.contains(
/tmp/ipykernel_3238/1545498179.py:22: UserWarning: This pattern is interpreted as a regu

Review queue saved: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_english_review_queue_v1.csv
Pairs flagged for review: 2888

Review reasons:
review_reasons
check_domain_relevance    2037
obligation                 760
negation                   451
parties                    272
numbers                    124
Name: count, dtype: int64


/tmp/ipykernel_3238/1545498179.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~review["original_text"].str.contains(
/tmp/ipykernel_3238/1545498179.py:44: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~review["simplified_text"].str.contains(


,original_text,simplified_text,preservation_label,review_reasons
5,The Company may revoke access to licensed software.,The company may remove access to licensed software.,preserved,check_domain_relevance;
6,The Company may limit access to premium features.,The company may restrict access to premium features.,preserved,check_domain_relevance;
9,User data may be used to improve account management services.,User data may be used to improve account management.,preserved,check_domain_relevance;
14,Personal information may be used to restore lost account access.,Personal information may be used to recover account access.,preserved,check_domain_relevance;
15,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,preserved,check_domain_relevance;
16,The Service may provide communication tools.,The service may offer communication tools.,preserved,check_domain_relevance;
20,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,preserved,obligation;
21,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,preserved,obligation;
22,The Company may restrict collaboration access.,The company may limit collaboration access.,preserved,check_domain_relevance;
23,Personal information may be used to improve account security measures.,Personal information may be used to improve account security.,preserved,check_domain_relevance;


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

review_path = project / "JUDGEBERT_english_review_queue_v1.csv"
review = pd.read_csv(review_path)

print("Total pairs flagged for review:", len(review))

print("\nReview reason counts:")
reasons = (
    review["review_reasons"]
    .fillna("")
    .str.rstrip("; ")
    .str.split("; ")
    .explode()
)
print(reasons[reasons != ""].value_counts())

# Show pairs with potential changes to negation, numbers,
# or the parties involved.
critical_pattern = r"negation|numbers|parties"

critical = review[
    review["review_reasons"].str.contains(
        critical_pattern, case=False, na=False
    )
]

print("\nPairs requiring closer meaning checks:", len(critical))

display(
    critical[
        [
            "original_text",
            "simplified_text",
            "preservation_label",
            "review_reasons",
        ]
    ].head(15)
)

Total pairs flagged for review: 2888

Review reason counts:
review_reasons
check_domain_relevance    2037
obligation                 760
negation                   451
parties                    272
numbers                    124
Name: count, dtype: int64

Pairs requiring closer meaning checks: 779


,original_text,simplified_text,preservation_label,review_reasons
10,Coverage may exclude losses caused by improper use of property.,Coverage may not include losses caused by improper property use.,preserved,negation; check_domain_relevance;
14,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,preserved,obligation; parties;
16,Any customer feedback submitted may be used internally to improve products and services.,The company may use your feedback to improve its offerings.,preserved,parties; check_domain_relevance;
24,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,preserved,obligation; parties;
30,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,preserved,obligation; parties;
32,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,preserved,parties; check_domain_relevance;
35,The Cardholder shall review account statements promptly upon receipt.,You should check your statements as soon as you receive them.,preserved,obligation; parties;
37,Coverage may exclude losses caused by inadequate maintenance.,Coverage may not include losses caused by poor maintenance.,preserved,negation; check_domain_relevance;
45,Coverage may exclude losses related to unlawful occupation.,Coverage may not include losses from unlawful occupation.,preserved,negation; check_domain_relevance;
57,The Platform may discontinue support for features dependent upon third-party services that are no longer available.,The app can remove features if outside services needed for them stop operating.,preserved,negation;


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
input_file = project / "JUDGEBERT_english_review_queue_v1.csv"

review = pd.read_csv(input_file)

print("Total pairs needing review:", len(review))
print("\nAvailable columns:")
print(review.columns.tolist())

# Display 10 examples for manual review
display(
    review[
        [
            "original_text",
            "simplified_text",
            "preservation_label",
            "review_reasons"
        ]
    ].head(10)
)

Total pairs needing review: 2888

Available columns:
['original_text', 'simplified_text', 'language', 'source_file', 'source_type', 'pair_complete', 'identical_text', 'preservation_label', 'human_score_1_to_10', 'review_status', 'split', 'label', 'review_reasons']


,original_text,simplified_text,preservation_label,review_reasons
0,The Company may revoke access to licensed software.,The company may remove access to licensed software.,preserved,check_domain_relevance;
1,The Company may limit access to premium features.,The company may restrict access to premium features.,preserved,check_domain_relevance;
2,User data may be used to improve account management services.,User data may be used to improve account management.,preserved,check_domain_relevance;
3,Personal information may be used to restore lost account access.,Personal information may be used to recover account access.,preserved,check_domain_relevance;
4,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,preserved,check_domain_relevance;
5,The Service may provide communication tools.,The service may offer communication tools.,preserved,check_domain_relevance;
6,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,preserved,obligation;
7,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,preserved,obligation;
8,The Company may restrict collaboration access.,The company may limit collaboration access.,preserved,check_domain_relevance;
9,Personal information may be used to improve account security measures.,Personal information may be used to improve account security.,preserved,check_domain_relevance;


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
input_file = project / "JUDGEBERT_english_review_queue_v1.csv"

review = pd.read_csv(input_file)

print("Total pairs needing review:", len(review))
print("\nAvailable columns:")
print(review.columns.tolist())

# Display 10 examples for manual review
display(
    review[
        [
            "original_text",
            "simplified_text",
            "preservation_label",
            "review_reasons"
        ]
    ].head(10)
)

Total pairs needing review: 2888

Available columns:
['original_text', 'simplified_text', 'language', 'source_file', 'source_type', 'pair_complete', 'identical_text', 'preservation_label', 'human_score_1_to_10', 'review_status', 'split', 'label', 'review_reasons']


,original_text,simplified_text,preservation_label,review_reasons
0,The Company may revoke access to licensed software.,The company may remove access to licensed software.,preserved,check_domain_relevance;
1,The Company may limit access to premium features.,The company may restrict access to premium features.,preserved,check_domain_relevance;
2,User data may be used to improve account management services.,User data may be used to improve account management.,preserved,check_domain_relevance;
3,Personal information may be used to restore lost account access.,Personal information may be used to recover account access.,preserved,check_domain_relevance;
4,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,preserved,check_domain_relevance;
5,The Service may provide communication tools.,The service may offer communication tools.,preserved,check_domain_relevance;
6,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,preserved,obligation;
7,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,preserved,obligation;
8,The Company may restrict collaboration access.,The company may limit collaboration access.,preserved,check_domain_relevance;
9,Personal information may be used to improve account security measures.,Personal information may be used to improve account security.,preserved,check_domain_relevance;


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
input_file = project / "JUDGEBERT_english_review_queue_v1.csv"

review = pd.read_csv(input_file)

# Select a reproducible batch of 50 pairs
batch = review.sample(
    n=min(50, len(review)),
    random_state=42
).copy()

# Add columns for your manual decision
batch["human_review_label"] = ""
batch["review_notes"] = ""

output_file = project / "JUDGEBERT_manual_review_batch_50.csv"
batch.to_csv(output_file, index=False)

print("Review batch created:", output_file)
print("Number of pairs:", len(batch))

display(
    batch[
        [
            "original_text",
            "simplified_text",
            "preservation_label",
            "review_reasons",
            "human_review_label",
            "review_notes",
        ]
    ].head(10)
)

Review batch created: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_manual_review_batch_50.csv
Number of pairs: 50


,original_text,simplified_text,preservation_label,review_reasons,human_review_label,review_notes
471,The Platform may require acceptance of revised community guidelines as a condition of continued participation.,You may need to accept updated community rules to keep using certain features.,preserved,check_domain_relevance;,,
1206,Personal information may be used to maintain compliance documentation.,Personal information may be used to maintain compliance records.,preserved,check_domain_relevance;,,
2381,"From this text, find out the pros and cons of dredging Dredging is the excavation of material from a water environment. Possible reasons for dredging include improving existing water features; reshaping land and water features to alter drainage, navigability, and commercial use; constructing dams, dikes, and other controls for streams and shorelines; and recovering valuable mineral deposits or marine life having commercial value. In all but a few situations the excavation is undertaken by a specialist floating plant, known as a dredger. Dredging is carried out in many different locations and for many different purposes, but the main objectives are usually to recover material of value or use, or to create a greater depth of water. Dredges have been classified as suction or mechanical. Dredging systems can either be shore-based, brought to a location based on barges, or built into purpose built vessels. Dredging has significant environmental impacts: it can disturb marine sediments, leading to both short- and long-term water pollution, destroy important seabed ecosystems, and can release legacy human-sourced toxins captured in the sediment. These environmental impacts can significantly hurt marine wildlife populations, contaminate sources of drinking water and interrupt economic activities such as fishing.","The benefits of dredging are: 1. improves existing water features for better drainage, navigability, and commercial use 2. important for constructing dams, dikes, and other controls for streams and shorelines 3. helps in recovering valuable mineral deposits or marine life having commercial value The demerits of dredging are related to its severe environmental impact, for example: 1. it can disturb marine sediments, leading to both short- and long-term water pollution 2. it can destroy important seabed ecosystems 3. it can release legacy human-sourced toxins captured in the sediment All the above impacts can significantly hurt marine wildlife populations, contaminate sources of drinking water and interrupt economic activities such as fishing.",preserved,numbers; check_domain_relevance;,,
2103,Give me some examples sedimentary rocks.,Sandstone and chalk are sedimentary rocks.,preserved,check_domain_relevance;,,
2884,Many people are hesitant to invest resources toward slowing down or even reversing the effects of climate change because they think it does not make sense if only a small portion of the population invests resources. They think it is stupid to do so until everyone agreed to do so or is forced by policy. What is the reason that some people still do so?,"Some people do want to wait until everyone agreed or a policy is in place because they want to start or increase the momentum of slowing down or reversing the effects of climate change. They don't want to wait until it is potentially too late but rather they want to do what they can right now. Some of them might hope that those actions motivate others to follow them which might in turn increase the chance of more and more people doing the same and thus likely starting a bandwagon effect. Some of the activists might also want to prevent living with a bad conscience and thus, they might prefer to spend their resources on climate change related causes instead of other alternatives.",changed,negation; obligation;,,
1271,The Company may limit access to customer benefits where account verification remains incomplete.,The company can restrict benefits until your account is v

In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

input_file = project / "JUDGEBERT_english_review_queue_v1.csv"
df = pd.read_csv(input_file)

# Combine both sentences for keyword screening
text = (
    df["original_text"].fillna("") + " " +
    df["simplified_text"].fillna("")
).str.lower()

# Preliminary legal-domain keyword filter
legal_pattern = (
    r"\blaw\b|\blegal\b|\bcourt\b|\bjudge\b|"
    r"\bcontract\b|\bagreement\b|\btenant\b|\blandlord\b|"
    r"\binsurer\b|\binsurance\b|\bclaim\b|\bliability\b|"
    r"\bobligation\b|\bshall\b|\bstatute\b|\bclause\b|"
    r"\bparty\b|\bparties\b|\bpolicyholder\b|\bconsumer\b|"
    r"\bjurisdiction\b|\bplaintiff\b|\bdefendant\b|"
    r"\bpayment\b|\bbreach\b|\bconfidential\b|\bissuer\b"
)

legal_candidates = df[text.str.contains(legal_pattern, regex=True)].copy()

output_file = project / "JUDGEBERT_legal_review_candidates_v1.csv"
legal_candidates.to_csv(output_file, index=False)

print("Original review pairs:", len(df))
print("Keyword-filtered candidates:", len(legal_candidates))
print("Saved to:", output_file)

display(
    legal_candidates[
        ["original_text", "simplified_text",
         "preservation_label", "review_reasons"]
    ].head(15)
)

Original review pairs: 2888
Keyword-filtered candidates: 908
Saved to: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_legal_review_candidates_v1.csv


,original_text,simplified_text,preservation_label,review_reasons
4,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,preserved,check_domain_relevance;
6,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,preserved,obligation;
7,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,preserved,obligation;
13,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,preserved,obligation;
14,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,preserved,obligation; parties;
22,Products shall remain subject to all applicable restrictions.,Products are subject to applicable restrictions.,preserved,obligation;
24,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,preserved,obligation; parties;
26,The Issuer may decline recurring payment requests where account status no longer permits such transactions.,The bank can reject recurring payments if your account no longer qualifies for them.,preserved,check_domain_relevance;
30,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,preserved,obligation; parties;
32,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,preserved,parties; check_domain_relevance;


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

input_file = project / "JUDGEBERT_legal_review_candidates_v1.csv"
df = pd.read_csv(input_file)

# Select 30 candidates for manual review
review = df.head(30).copy()

# Add blank human-review fields
review["human_label"] = ""
review["human_score_1_to_10"] = ""
review["human_notes"] = ""

output_file = project / "JUDGEBERT_manual_review_v1.csv"
review.to_csv(output_file, index=False)

print("Pairs prepared for manual review:", len(review))
print("Review file saved to:", output_file)

display(review[
    ["original_text", "simplified_text",
     "preservation_label", "human_label",
     "human_score_1_to_10", "human_notes"]
].head(10))

Pairs prepared for manual review: 30
Review file saved to: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_manual_review_v1.csv


,original_text,simplified_text,preservation_label,human_label,human_score_1_to_10,human_notes
0,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,preserved,,,
1,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,preserved,,,
2,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,preserved,,,
3,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,preserved,,,
4,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,preserved,,,
5,Products shall remain subject to all applicable restrictions.,Products are subject to applicable restrictions.,preserved,,,
6,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,preserved,,,
7,The Issuer may decline recurring payment requests where account status no longer permits such transactions.,The bank can reject recurring payments if your account no longer qualifies for them.,preserved,,,
8,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,preserved,,,
9,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,preserved,,,


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
input_file = project / "JUDGEBERT_manual_review_v1.csv"

review = pd.read_csv(input_file)

suggested = [
    "uncertain",
    "preserved",
    "uncertain",
    "uncertain",
    "preserved",
    "preserved",
    "changed",
    "uncertain",
    "uncertain",
    "uncertain"
]

review.loc[:9, "human_label"] = suggested
review.loc[:9, "human_notes"] = "Preliminary review suggestion; verify before final use."

output_file = project / "JUDGEBERT_manual_review_v1.csv"
review.to_csv(output_file, index=False)

print("Saved review file:", output_file)
display(review[["original_text", "simplified_text", "human_label"]].head(10))

Saved review file: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_manual_review_v1.csv


/tmp/ipykernel_3238/1116461354.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['uncertain', 'preserved', 'uncertain', 'uncertain', 'preserved', 'preserved', 'changed', 'uncertain', 'uncertain', 'uncertain']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review.loc[:9, "human_label"] = suggested
/tmp/ipykernel_3238/1116461354.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Preliminary review suggestion; verify before final use.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review.loc[:9, "human_notes"] = "Preliminary review suggestion; verify before final use."


,original_text,simplified_text,human_label
0,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,uncertain
1,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,preserved
2,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,uncertain
3,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,uncertain
4,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,preserved
5,Products shall remain subject to all applicable restrictions.,Products are subject to applicable restrictions.,preserved
6,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,changed
7,The Issuer may decline recurring payment requests where account status no longer permits such transactions.,The bank can reject recurring payments if your account no longer qualifies for them.,uncertain
8,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,uncertain
9,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,uncertain


In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
project.mkdir(parents=True, exist_ok=True)

rubric = """JUDGEBERT - Legal Meaning Preservation (LMP) Scoring Rubric

Purpose:
Compare an original legal text with its simplified version in the SAME language.

Score: 10
Meaning fully preserved. Only wording or sentence structure changes.

Score: 9
Meaning preserved, with a very minor difference that does not affect legal interpretation.

Score: 7-8
Main meaning preserved, but a minor legal detail is less explicit.

Score: 5-6
Some important detail is missing, ambiguous, or weakened.

Score: 3-4
A major right, obligation, condition, exception, deadline, or party relationship changes.

Score: 1-2
Meaning is substantially reversed or seriously distorted.

Review these carefully:
- Negation (not, never, prohibited)
- Must/shall versus may
- Rights and obligations
- Parties and who performs an action
- Conditions and exceptions
- Dates, deadlines, and amounts
- Scope and limitations

Important:
- Similar wording does not guarantee preserved meaning.
- Do not invent scores or call unverified labels expert annotations.
- Mark uncertain cases for human review.
- English pairs must be compared in English; Hindi pairs must be compared in Hindi.
- This rubric is a preliminary project rubric, not a validated legal standard.
"""

out = project / "JUDGEBERT_LMP_scoring_rubric.txt"
out.write_text(rubric, encoding="utf-8")

print("Rubric saved successfully:", out)
print(rubric)

Rubric saved successfully: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_scoring_rubric.txt
JUDGEBERT - Legal Meaning Preservation (LMP) Scoring Rubric

Purpose:
Compare an original legal text with its simplified version in the SAME language.

Score: 10
Meaning fully preserved. Only wording or sentence structure changes.

Score: 9
Meaning preserved, with a very minor difference that does not affect legal interpretation.

Score: 7-8
Main meaning preserved, but a minor legal detail is less explicit.

Score: 5-6
Some important detail is missing, ambiguous, or weakened.

Score: 3-4
A major right, obligation, condition, exception, deadline, or party relationship changes.

Score: 1-2
Meaning is substantially reversed or seriously distorted.

Review these carefully:
- Negation (not, never, prohibited)
- Must/shall versus may
- Rights and obligations
- Parties and who performs an action
- Conditions and exceptions
- Dates, deadlines, and amounts
- Scope and limitations

Important:
- Simil

In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")

# Load the current review queue
review_file = project / "JUDGEBERT_manual_review_v1.csv"
df = pd.read_csv(review_file)

print("Total pairs:", len(df))
print("\nColumns:")
print(df.columns.tolist())

# Show the current label distribution
label_col = "human_label" if "human_label" in df.columns else "preservation_label"
print("\nLabel counts:")
print(df[label_col].fillna("unlabelled").value_counts())

# Check whether any 1–10 scores have been assigned
score_col = "human_score_1_to_10"
if score_col in df.columns:
    print("\nActual 1–10 scores:", df[score_col].notna().sum())
else:
    print("\nNo 1–10 score column found yet.")

display(df.head(5))

Total pairs: 30

Columns:
['original_text', 'simplified_text', 'language', 'source_file', 'source_type', 'pair_complete', 'identical_text', 'preservation_label', 'human_score_1_to_10', 'review_status', 'split', 'label', 'review_reasons', 'human_label', 'human_notes']

Label counts:
human_label
unlabelled    20
uncertain      6
preserved      3
changed        1
Name: count, dtype: int64

Actual 1–10 scores: 0


,original_text,simplified_text,language,source_file,source_type,pair_complete,identical_text,preservation_label,human_score_1_to_10,review_status,split,label,review_reasons,human_label,human_notes
0,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,NaN,needs_review,train,1,check_domain_relevance;,uncertain,Preliminary review suggestion; verify before final use.
1,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,NaN,needs_review,train,1,obligation;,preserved,Preliminary review suggestion; verify before final use.
2,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,NaN,needs_review,train,1,obligation;,uncertain,Preliminary review suggestion; verify before final use.
3,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,NaN,needs_review,train,1,obligation;,uncertain,Preliminary review suggestion; verify before final use.
4,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,en,JUDGEBERT_LMP_dataset_v1.csv,existing_master,True,False,preserved,NaN,needs_review,train,1,obligation; parties;,preserved,Preliminary review suggestion; verify before final use.


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
file_path = project / "JUDGEBERT_manual_review_v1.csv"

df = pd.read_csv(file_path)

# Ensure the numeric score column exists
if "human_score_1_to_10" not in df.columns:
    df["human_score_1_to_10"] = pd.NA

# Keep uncertain pairs for review; do not assign them scores automatically
print("Pairs available:", len(df))
print("Pairs with numeric scores:", df["human_score_1_to_10"].notna().sum())

display(df[[
    "original_text",
    "simplified_text",
    "human_label",
    "human_score_1_to_10"
]].head(10))

Pairs available: 30
Pairs with numeric scores: 0


,original_text,simplified_text,human_label,human_score_1_to_10
0,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,uncertain,NaN
1,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,preserved,NaN
2,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,uncertain,NaN
3,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,uncertain,NaN
4,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,preserved,NaN
5,Products shall remain subject to all applicable restrictions.,Products are subject to applicable restrictions.,preserved,NaN
6,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,changed,NaN
7,The Issuer may decline recurring payment requests where account status no longer permits such transactions.,The bank can reject recurring payments if your account no longer qualifies for them.,uncertain,NaN
8,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,uncertain,NaN
9,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,uncertain,NaN


In [ ]:
  import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
file_path = project / "JUDGEBERT_manual_review_v1.csv"

df = pd.read_csv(file_path)

# Add preliminary scores for rows 0–9.
# Review and change these values if you disagree.
suggested_scores = [4, 10, 8, 7, 8, 10, 5, 7, 6, 4]

df["human_score_1_to_10"] = pd.to_numeric(
    df.get("human_score_1_to_10", pd.Series(index=df.index, dtype="float64")),
    errors="coerce"
)

df.loc[df.index[:10], "human_score_1_to_10"] = suggested_scores

# Record that these are preliminary suggestions, not expert-verified scores.
if "score_status" not in df.columns:
    df["score_status"] = "not_reviewed"

df.loc[df.index[:10], "score_status"] = "suggested_needs_human_review"

df.to_csv(file_path, index=False)

print("Saved:", file_path)
print("Rows with preliminary numeric scores:",
      df["human_score_1_to_10"].notna().sum())

display(df[[
    "original_text",
    "simplified_text",
    "human_label",
    "human_score_1_to_10",
    "score_status"
]].head(10))

Saved: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_manual_review_v1.csv
Rows with preliminary numeric scores: 10


,original_text,simplified_text,human_label,human_score_1_to_10,score_status
0,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,uncertain,4.0,suggested_needs_human_review
1,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,preserved,10.0,suggested_needs_human_review
2,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,uncertain,8.0,suggested_needs_human_review
3,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,uncertain,7.0,suggested_needs_human_review
4,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,preserved,8.0,suggested_needs_human_review
5,Products shall remain subject to all applicable restrictions.,Products are subject to applicable restrictions.,preserved,10.0,suggested_needs_human_review
6,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,changed,5.0,suggested_needs_human_review
7,The Issuer may decline recurring payment requests where account status no longer permits such transactions.,The bank can reject recurring payments if your account no longer qualifies for them.,uncertain,7.0,suggested_needs_human_review
8,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,uncertain,6.0,suggested_needs_human_review
9,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,uncertain,4.0,suggested_needs_human_review


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
file_path = project / "JUDGEBERT_manual_review_v1.csv"

df = pd.read_csv(file_path)

print("TOTAL PAIRS:", len(df))
print("\nLANGUAGE COUNTS:")
print(df["language"].value_counts(dropna=False))

print("\nLABEL COUNTS:")
print(df["human_label"].value_counts(dropna=False))

print("\nSCORE STATUS:")
print(df["score_status"].value_counts(dropna=False))

scores = pd.to_numeric(df["human_score_1_to_10"], errors="coerce")
print("\nPAIRS WITH NUMERIC SCORES:", scores.notna().sum())
print("PAIRS WITHOUT NUMERIC SCORES:", scores.isna().sum())

print("\nSCORED PAIRS BY STATUS:")
print(
    df.assign(numeric_score=scores.notna())
      .groupby("score_status", dropna=False)["numeric_score"]
      .agg(["count", "sum"])
)

display(
    df.loc[scores.notna(), [
        "original_text",
        "simplified_text",
        "human_label",
        "human_score_1_to_10",
        "score_status"
    ]].head(10)
)

TOTAL PAIRS: 30

LANGUAGE COUNTS:
language
en    30
Name: count, dtype: int64

LABEL COUNTS:
human_label
NaN          20
uncertain     6
preserved     3
changed       1
Name: count, dtype: int64

SCORE STATUS:
score_status
not_reviewed                    20
suggested_needs_human_review    10
Name: count, dtype: int64

PAIRS WITH NUMERIC SCORES: 10
PAIRS WITHOUT NUMERIC SCORES: 20

SCORED PAIRS BY STATUS:
                              count  sum
score_status                            
not_reviewed                     20    0
suggested_needs_human_review     10   10


,original_text,simplified_text,human_label,human_score_1_to_10,score_status
0,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,uncertain,4.0,suggested_needs_human_review
1,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,preserved,10.0,suggested_needs_human_review
2,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,uncertain,8.0,suggested_needs_human_review
3,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,uncertain,7.0,suggested_needs_human_review
4,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,preserved,8.0,suggested_needs_human_review
5,Products shall remain subject to all applicable restrictions.,Products are subject to applicable restrictions.,preserved,10.0,suggested_needs_human_review
6,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,changed,5.0,suggested_needs_human_review
7,The Issuer may decline recurring payment requests where account status no longer permits such transactions.,The bank can reject recurring payments if your account no longer qualifies for them.,uncertain,7.0,suggested_needs_human_review
8,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,uncertain,6.0,suggested_needs_human_review
9,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,uncertain,4.0,suggested_needs_human_review


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
input_file = project / "JUDGEBERT_manual_review_v1.csv"

df = pd.read_csv(input_file)

# Select 10 pairs for manual review
review = df.head(10).copy()

# Keep the suggested scores visible, but do not treat them as verified
review["reviewer_score_1_to_10"] = ""
review["reviewer_label"] = ""
review["reviewer_notes"] = ""

output_file = project / "JUDGEBERT_first10_review.csv"
review.to_csv(output_file, index=False)

print("Review sheet saved to:", output_file)
print("Pairs to review:", len(review))

display(review[[
    "original_text",
    "simplified_text",
    "human_label",
    "human_score_1_to_10",
    "reviewer_score_1_to_10",
    "reviewer_label",
    "reviewer_notes"
]])

Review sheet saved to: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_first10_review.csv
Pairs to review: 10


,original_text,simplified_text,human_label,human_score_1_to_10,reviewer_score_1_to_10,reviewer_label,reviewer_notes
0,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,uncertain,4.0,,,
1,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,preserved,10.0,,,
2,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,uncertain,8.0,,,
3,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,uncertain,7.0,,,
4,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,preserved,8.0,,,
5,Products shall remain subject to all applicable restrictions.,Products are subject to applicable restrictions.,preserved,10.0,,,
6,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,changed,5.0,,,
7,The Issuer may decline recurring payment requests where account status no longer permits such transactions.,The bank can reject recurring payments if your account no longer qualifies for them.,uncertain,7.0,,,
8,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,uncertain,6.0,,,
9,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,uncertain,4.0,,,


In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/drive/MyDrive/JUDGEBERT_LMP")
file_path = project / "JUDGEBERT_first10_review.csv"

review = pd.read_csv(file_path)

# Show the first 10 rows with editable review fields
for i, row in review.iterrows():
    print(f"\nROW {i}")
    print("Original:", row["original_text"])
    print("Simplified:", row["simplified_text"])


ROW 0
Original: The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.
Simplified: The bank may charge a fee when your card is used in another currency.

ROW 1
Original: Confidential Information shall include strategic partnerships.
Simplified: Confidential information includes strategic partnerships.

ROW 2
Original: Products shall be offered according to the terms of this Agreement.
Simplified: Products are offered under this agreement.

ROW 3
Original: Requests for exceptions to academic requirements shall be considered on a case-by-case basis.
Simplified: Requests to waive academic rules will be reviewed individually.

ROW 4
Original: The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.
Simplified: You should inform the bank if you move to a new place that affects your account.

ROW 5
Original: Products shall remain subject to all applicable restrictions.
Simplified:

In [ ]:
# Example review entries — change these after checking each pair.
scores = [4, 10, 7, 5, 7, 10, 5, 5, None, None]
labels = [
    "uncertain",
    "preserved",
    "uncertain",
    "changed",
    "uncertain",
    "preserved",
    "changed",
    "uncertain",
    "",
    ""
]

review["reviewer_score_1_to_10"] = scores
review["reviewer_label"] = labels

review.to_csv(file_path, index=False)

print("Reviews saved to:", file_path)

Reviews saved to: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_first10_review.csv


In [ ]:
display(review[[
    "original_text",
    "simplified_text",
    "reviewer_score_1_to_10",
    "reviewer_label"
]])

,original_text,simplified_text,reviewer_score_1_to_10,reviewer_label
0,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,4.0,uncertain
1,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,10.0,preserved
2,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,7.0,uncertain
3,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,5.0,changed
4,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,7.0,uncertain
5,Products shall remain subject to all applicable restrictions.,Products are subject to applicable restrictions.,10.0,preserved
6,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,5.0,changed
7,The Issuer may decline recurring payment requests where account status no longer permits such transactions.,The bank can reject recurring payments if your account no longer qualifies for them.,5.0,uncertain
8,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,NaN,
9,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,NaN,


In [ ]:
from google.colab import files

review.to_csv(
    "/content/JUDGEBERT_first10_review_completed.csv",
    index=False
)

files.download("/content/JUDGEBERT_first10_review_completed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

!pip -q install sentence-transformers pandas

import pandas as pd
import re
from sentence_transformers import SentenceTransformer, util

# Load the multilingual baseline model
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Small test set: same-language original/simplified pairs
pairs = [
    {
        "language": "en",
        "original_text": "The tenant must pay rent by the fifth day of each month.",
        "simplified_text": "The tenant must pay rent by the fifth day of each month."
    },
    {
        "language": "en",
        "original_text": "The tenant must pay rent by the fifth day of each month.",
        "simplified_text": "The tenant may pay rent whenever they want."
    },
    {
        "language": "hi",
        "original_text": "किरायेदार को हर महीने की पाँच तारीख तक किराया देना होगा।",
        "simplified_text": "किरायेदार को हर महीने की पाँच तारीख तक किराया देना होगा।"
    },
    {
        "language": "hi",
        "original_text": "किरायेदार को हर महीने की पाँच तारीख तक किराया देना होगा।",
        "simplified_text": "किरायेदार जब चाहे किराया दे सकता है।"
    }
]

results = []

for pair in pairs:
    embeddings = model.encode(
        [pair["original_text"], pair["simplified_text"]],
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    similarity = util.cos_sim(
        embeddings[0], embeddings[1]
    ).item()

    results.append({
        **pair,
        "semantic_similarity": round(similarity, 4)
    })

results_df = pd.DataFrame(results)
display(results_df)

results_df.to_csv(
    "/content/JUDGEBERT_baseline_demo.csv",
    index=False
)

print("Saved: /content/JUDGEBERT_baseline_demo.csv")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,language,original_text,simplified_text,semantic_similarity
0,en,The tenant must pay rent by the fifth day of each month.,The tenant must pay rent by the fifth day of each month.,1.0000
1,en,The tenant must pay rent by the fifth day of each month.,The tenant may pay rent whenever they want.,0.7160
2,hi,किरायेदार को हर महीने की पाँच तारीख तक किराया देना होगा।,किरायेदार को हर महीने की पाँच तारीख तक किराया देना होगा।,1.0000
3,hi,किरायेदार को हर महीने की पाँच तारीख तक किराया देना होगा।,किरायेदार जब चाहे किराया दे सकता है।,0.6346


Saved: /content/JUDGEBERT_baseline_demo.csv


In [ ]:

import pandas as pd
import os
import re
from sentence_transformers import util

# Find the actual English simplification dataset
possible_files = [
    "/content/legal_simplification_pairs_en_filtered.csv",
    "/content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_legal_review_candidates_v1.csv",
    "/content/lmp_candidate_pairs_v2.csv"
]

file_path = next((p for p in possible_files if os.path.exists(p)), None)

if file_path is None:
    raise FileNotFoundError(
        "Dataset not found. Check the CSV files in the Colab Files panel."
    )

df = pd.read_csv(file_path)
print("Dataset:", file_path)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# Identify the text columns
original_col = next(
    (c for c in ["original_text", "Input", "original", "source_text"]
     if c in df.columns), None
)
simplified_col = next(
    (c for c in ["simplified_text", "Output", "simplified", "target_text"]
     if c in df.columns), None
)

if original_col is None or simplified_col is None:
    raise ValueError(
        "Could not identify the original/simplified text columns. "
        "Use the printed column names to adjust the code."
    )

# Keep valid pairs and sample up to 500 for a first run
work = df[[original_col, simplified_col]].copy()
work.columns = ["original_text", "simplified_text"]
work = work.dropna()
work["original_text"] = work["original_text"].astype(str)
work["simplified_text"] = work["simplified_text"].astype(str)
work = work[
    (work["original_text"].str.strip() != "") &
    (work["simplified_text"].str.strip() != "")
].drop_duplicates()

work = work.head(500).copy()

# Calculate semantic similarity in batches
original_embeddings = model.encode(
    work["original_text"].tolist(),
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)
simplified_embeddings = model.encode(
    work["simplified_text"].tolist(),
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)

work["semantic_similarity"] = util.cos_sim(
    original_embeddings, simplified_embeddings
).diagonal().cpu().numpy().round(4)

# Flag possible legally important changes.
# These are warning signals, not automatic legal judgments.
patterns = {
    "negation": r"\b(not|no|never|neither|nor|cannot|can't|unless|without)\b",
    "modality": r"\b(must|shall|required|may|might|can|cannot|should|will|will not)\b",
    "numbers_dates_money": r"\b\d+(?:[.,]\d+)?%?\b|[$₹€£]",
    "legal_parties": r"\b(tenant|landlord|lessor|lessee|plaintiff|defendant|"
                     r"claimant|insurer|insured|issuer|cardholder|consumer|"
                     r"employer|employee|buyer|seller)\b"
}

def normalized_matches(text, pattern):
    return sorted(set(
        re.findall(pattern, text.lower())
    ))

for name, pattern in patterns.items():
    work[name + "_original"] = work["original_text"].apply(
        lambda x: normalized_matches(x, pattern)
    )
    work[name + "_simplified"] = work["simplified_text"].apply(
        lambda x: normalized_matches(x, pattern)
    )
    work[name + "_warning"] = work.apply(
        lambda r: r[name + "_original"] != r[name + "_simplified"],
        axis=1
    )

warning_cols = [name + "_warning" for name in patterns]
work["legal_detail_warning_count"] = work[warning_cols].sum(axis=1)

work["review_priority"] = work.apply(
    lambda r: "HIGH - check legal details"
    if r["legal_detail_warning_count"] >= 2
    else (
        "CHECK"
        if r["legal_detail_warning_count"] == 1
        or r["semantic_similarity"] < 0.75
        else "LOWER - still verify"
    ),
    axis=1
)

# Deliberately do not invent a 1–10 score or claim labels are verified.
output_path = "/content/judgebert_baseline_results_v2.csv"
work.to_csv(output_path, index=False)

print("\nSaved results to:", output_path)
print("\nReview priority counts:")
display(work["review_priority"].value_counts().to_frame("pairs"))

print("\nExample results:")
display(work[
    ["original_text", "simplified_text",
     "semantic_similarity", "legal_detail_warning_count",
     "review_priority"]
].head(10))


Dataset: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_legal_review_candidates_v1.csv
Rows: 908
Columns: ['original_text', 'simplified_text', 'language', 'source_file', 'source_type', 'pair_complete', 'identical_text', 'preservation_label', 'human_score_1_to_10', 'review_status', 'split', 'label', 'review_reasons']


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


Saved results to: /content/judgebert_baseline_results_v2.csv

Review priority counts:


,pairs
review_priority,
CHECK,263
HIGH - check legal details,226
LOWER - still verify,11



Example results:


,original_text,simplified_text,semantic_similarity,legal_detail_warning_count,review_priority
0,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,0.6675,1,CHECK
1,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,0.9670,1,CHECK
2,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,0.9035,1,CHECK
3,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,0.5724,1,CHECK
4,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,0.5109,2,HIGH - check legal details
5,Products shall remain subject to all applicable restrictions.,Products are subject to applicable restrictions.,0.6681,1,CHECK
6,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,0.5381,2,HIGH - check legal details
7,The Issuer may decline recurring payment requests where account status no longer permits such transactions.,The bank can reject recurring payments if your account no longer qualifies for them.,0.8089,2,HIGH - check legal details
8,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,0.7281,2,HIGH - check legal details
9,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,0.7344,1,CHECK


In [ ]:

import pandas as pd
import os

results_path = "/content/judgebert_baseline_results_v2.csv"
drive_folder = "/content/drive/MyDrive/JUDGEBERT_LMP"

os.makedirs(drive_folder, exist_ok=True)

results = pd.read_csv(results_path)

priority_summary = (
    results["review_priority"]
    .value_counts()
    .rename_axis("review_priority")
    .reset_index(name="pair_count")
)

priority_summary["percentage"] = (
    priority_summary["pair_count"] / len(results) * 100
).round(2)

print("JUDGEBERT BASELINE EVALUATION")
print("=" * 40)
print("Pairs evaluated:", len(results))
print("Mean semantic similarity:",
      round(results["semantic_similarity"].mean(), 4))
print("Median semantic similarity:",
      round(results["semantic_similarity"].median(), 4))
print("\nReview-priority breakdown:")
display(priority_summary)

print("\nLegal-detail warning frequencies:")
warning_columns = [
    "negation_warning",
    "modality_warning",
    "numbers_dates_money_warning",
    "legal_parties_warning"
]

warning_summary = pd.DataFrame({
    "warning_type": warning_columns,
    "pairs_flagged": [
        int(results[c].sum()) for c in warning_columns
    ]
})
display(warning_summary)

# Save full evaluation and summaries to Drive
results.to_csv(
    f"{drive_folder}/JUDGEBERT_baseline_results_v2.csv",
    index=False
)

priority_summary.to_csv(
    f"{drive_folder}/JUDGEBERT_priority_summary_v1.csv",
    index=False
)

warning_summary.to_csv(
    f"{drive_folder}/JUDGEBERT_warning_summary_v1.csv",
    index=False
)

print("\nSaved all three files in:", drive_folder)


JUDGEBERT BASELINE EVALUATION
Pairs evaluated: 500
Mean semantic similarity: 0.8066
Median semantic similarity: 0.8439

Review-priority breakdown:


,review_priority,pair_count,percentage
0,CHECK,263,52.6
1,HIGH - check legal details,226,45.2
2,LOWER - still verify,11,2.2



Legal-detail warning frequencies:


,warning_type,pairs_flagged
0,negation_warning,125
1,modality_warning,460
2,numbers_dates_money_warning,0
3,legal_parties_warning,145



Saved all three files in: /content/drive/MyDrive/JUDGEBERT_LMP


In [ ]:

import pandas as pd
import os

# Load the baseline results already generated
input_path = "/content/judgebert_baseline_results_v2.csv"
df = pd.read_csv(input_path)

def calculate_lmp_score(row):
    similarity = float(row["semantic_similarity"])
    warnings = int(row["legal_detail_warning_count"])

    # Convert similarity to an initial 1–10 estimate
    score = round(1 + 9 * max(0, min(1, similarity)))

    # Apply conservative penalties for possible legal-detail changes
    if row.get("negation_warning", False):
        score -= 3

    if row.get("modality_warning", False):
        score -= 2

    if row.get("numbers_dates_money_warning", False):
        score -= 2

    if row.get("legal_parties_warning", False):
        score -= 2

    score = max(1, min(10, score))

    if warnings > 0:
        status = "FLAGGED_FOR_LEGAL_REVIEW"
    else:
        status = "PRELIMINARY_ESTIMATE"

    return pd.Series({
        "lmp_score_1_to_10": score,
        "score_status": status
    })

df[["lmp_score_1_to_10", "score_status"]] = df.apply(
    calculate_lmp_score, axis=1
)

# Save results
output_path = "/content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_scored_v1.csv"
df.to_csv(output_path, index=False)

print("Scoring component completed.")
print("Pairs scored:", len(df))
print("Saved to:", output_path)

print("\nScore distribution:")
display(
    df["lmp_score_1_to_10"]
    .value_counts()
    .sort_index()
    .rename_axis("score")
    .reset_index(name="pair_count")
)

print("\nExample scored pairs:")
display(
    df[[
        "original_text",
        "simplified_text",
        "semantic_similarity",
        "legal_detail_warning_count",
        "lmp_score_1_to_10",
        "score_status"
    ]].head(10)
)


Scoring component completed.
Pairs scored: 500
Saved to: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_LMP_scored_v1.csv

Score distribution:


,score,pair_count
0,1,33
1,2,42
2,3,54
3,4,103
4,5,39
5,6,36
6,7,94
7,8,88
8,9,1
9,10,10



Example scored pairs:


,original_text,simplified_text,semantic_similarity,legal_detail_warning_count,lmp_score_1_to_10,score_status
0,The Issuer may impose a foreign currency conversion charge on transactions settled in non-Indian currencies.,The bank may charge a fee when your card is used in another currency.,0.6675,1,5,FLAGGED_FOR_LEGAL_REVIEW
1,Confidential Information shall include strategic partnerships.,Confidential information includes strategic partnerships.,0.9670,1,8,FLAGGED_FOR_LEGAL_REVIEW
2,Products shall be offered according to the terms of this Agreement.,Products are offered under this agreement.,0.9035,1,7,FLAGGED_FOR_LEGAL_REVIEW
3,Requests for exceptions to academic requirements shall be considered on a case-by-case basis.,Requests to waive academic rules will be reviewed individually.,0.5724,1,4,FLAGGED_FOR_LEGAL_REVIEW
4,The Cardholder shall promptly notify the Issuer of any change in residency that may affect account administration.,You should inform the bank if you move to a new place that affects your account.,0.5109,2,2,FLAGGED_FOR_LEGAL_REVIEW
5,Products shall remain subject to all applicable restrictions.,Products are subject to applicable restrictions.,0.6681,1,5,FLAGGED_FOR_LEGAL_REVIEW
6,The Consumer shall be solely responsible for maintaining confidentiality of account access credentials.,You are responsible for keeping your login information secure.,0.5381,2,2,FLAGGED_FOR_LEGAL_REVIEW
7,The Issuer may decline recurring payment requests where account status no longer permits such transactions.,The bank can reject recurring payments if your account no longer qualifies for them.,0.8089,2,4,FLAGGED_FOR_LEGAL_REVIEW
8,The Consumer acknowledges that personalized product recommendations are generated through automated systems.,Product suggestions may be created automatically using computer systems.,0.7281,2,4,FLAGGED_FOR_LEGAL_REVIEW
9,The Cardholder acknowledges that transaction settlement dates may differ from transaction authorization dates.,The date a payment is completed may be different from the date it was approved.,0.7344,1,6,FLAGGED_FOR_LEGAL_REVIEW


In [ ]:

import re
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from sentence_transformers import util

# Reuse the multilingual model already loaded in this notebook.
# If the runtime was restarted, load it again:
try:
    model
except NameError:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

def assess_pair(original, simplified):
    original = original.strip()
    simplified = simplified.strip()

    if not original or not simplified:
        return None, ["Enter both texts before scoring."]

    embeddings = model.encode(
        [original, simplified],
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    similarity = float(util.cos_sim(
        embeddings[0], embeddings[1]
    ).item())

    patterns = {
        "Negation": r"\b(not|no|never|neither|nor|cannot|can't|unless|without)\b",
        "Legal obligation or permission": r"\b(must|shall|required|may|might|can|cannot|should|will)\b",
        "Numbers, dates or money": r"\b\d+(?:[.,]\d+)?%?\b|[$₹€£]",
        "Legal parties": r"\b(tenant|landlord|lessor|lessee|plaintiff|defendant|"
                        r"claimant|insurer|insured|issuer|cardholder|consumer|"
                        r"employer|employee|buyer|seller)\b"
    }

    warnings = []
    for name, pattern in patterns.items():
        a = set(re.findall(pattern, original.lower()))
        b = set(re.findall(pattern, simplified.lower()))
        if a != b:
            warnings.append(name + " terms differ; inspect carefully.")

    score = round(1 + 9 * similarity)
    score -= sum(
        3 if w.startswith("Negation") else
        2 if w.startswith("Legal obligation") else
        2 if w.startswith("Numbers") else
        2
        for w in warnings
    )
    score = max(1, min(10, score))

    if not warnings:
        warnings = [
            "No rule-based differences detected. This does not prove legal meaning is preserved."
        ]

    result = {
        "semantic_similarity": round(similarity, 4),
        "preliminary_lmp_score": score,
        "warnings": warnings
    }
    return result, []

language = widgets.Dropdown(
    options=[("English", "en"), ("Hindi", "hi")],
    value="en",
    description="Language:"
)

original_box = widgets.Textarea(
    placeholder="Enter the original legal sentence...",
    description="Original:",
    layout=widgets.Layout(width="95%", height="110px")
)

simplified_box = widgets.Textarea(
    placeholder="Enter the simplified version in the same language...",
    description="Simplified:",
    layout=widgets.Layout(width="95%", height="110px")
)

score_button = widgets.Button(
    description="Assess legal meaning",
    button_style="primary"
)

output = widgets.Output()

def on_score_clicked(_):
    with output:
        clear_output()
        result, errors = assess_pair(
            original_box.value,
            simplified_box.value
        )

        if errors:
            print(errors[0])
            return

        print("JUDGEBERT — PRELIMINARY ASSESSMENT")
        print("Language:", language.label if language.value == "en" else "Hindi")
        print("Semantic similarity:", result["semantic_similarity"])
        print("Preliminary LMP score:", str(result["preliminary_lmp_score"]) + "/10")
        print("\nWarnings:")
        for warning in result["warnings"]:
            print("-", warning)

        print("\nResearch prototype only — not legal advice or a validated score.")

score_button.on_click(on_score_clicked)

display(language, original_box, simplified_box, score_button, output)


Dropdown(description='Language:', options=(('English', 'en'), ('Hindi', 'hi')), value='en')

Textarea(value='', description='Original:', layout=Layout(height='110px', width='95%'), placeholder='Enter the…

Textarea(value='', description='Simplified:', layout=Layout(height='110px', width='95%'), placeholder='Enter t…

Button(button_style='primary', description='Assess legal meaning', style=ButtonStyle())

Output()

In [ ]:

import csv
import io
from datetime import datetime
from IPython.display import display, clear_output
import ipywidgets as widgets

assessment_history = []

reset_button = widgets.Button(description="Reset texts")
download_button = widgets.Button(description="Download assessment history")
history_output = widgets.Output()

def save_current_assessment(_):
    original = original_box.value.strip()
    simplified = simplified_box.value.strip()

    if not original or not simplified:
        with history_output:
            clear_output()
            print("Enter both sentences and click Assess legal meaning first.")
        return

    result, errors = assess_pair(original, simplified)

    if errors:
        with history_output:
            clear_output()
            print(errors[0])
        return

    assessment_history.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "language": language.value,
        "original_text": original,
        "simplified_text": simplified,
        "semantic_similarity": result["semantic_similarity"],
        "preliminary_lmp_score": result["preliminary_lmp_score"],
        "warnings": " | ".join(result["warnings"])
    })

    with history_output:
        clear_output()
        print("Assessment added to download history.")
        print("Assessments saved in this session:", len(assessment_history))

def reset_texts(_):
    original_box.value = ""
    simplified_box.value = ""
    with output:
        clear_output()
    with history_output:
        clear_output()
        print("Text fields reset.")

def download_history(_):
    with history_output:
        clear_output()
        if not assessment_history:
            print("No assessments saved yet. Click Assess legal meaning, then Save assessment.")
            return

        import pandas as pd
        from google.colab import files

        history_df = pd.DataFrame(assessment_history)
        path = "/content/JUDGEBERT_assessment_history.csv"
        history_df.to_csv(path, index=False)
        print("Downloading assessment history...")
        files.download(path)

save_button = widgets.Button(
    description="Save assessment",
    button_style="success"
)
save_button.on_click(save_current_assessment)
reset_button.on_click(reset_texts)
download_button.on_click(download_history)

display(widgets.HBox([save_button, reset_button, download_button]))
display(history_output)


Output()

In [ ]:

import os
import pandas as pd

project_folder = "/content/drive/MyDrive/JUDGEBERT_LMP"
os.makedirs(project_folder, exist_ok=True)

# Save the interactive demo history if it exists
if "assessment_history" in globals() and assessment_history:
    history_df = pd.DataFrame(assessment_history)
    history_path = os.path.join(
        project_folder, "JUDGEBERT_demo_assessment_history.csv"
    )
    history_df.to_csv(history_path, index=False)
    print("Demo history saved:", history_path)
    print("Assessments:", len(history_df))
else:
    print("No saved demo assessments in the current session yet.")

# Confirm the baseline and scored datasets are available
for filename in [
    "JUDGEBERT_baseline_results_v2.csv",
    "JUDGEBERT_LMP_scored_v1.csv",
    "JUDGEBERT_priority_summary_v1.csv",
    "JUDGEBERT_warning_summary_v1.csv"
]:
    path = os.path.join(project_folder, filename)
    print(("FOUND  " if os.path.exists(path) else "MISSING "), filename)


Demo history saved: /content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_demo_assessment_history.csv
Assessments: 1
FOUND   JUDGEBERT_baseline_results_v2.csv
FOUND   JUDGEBERT_LMP_scored_v1.csv
FOUND   JUDGEBERT_priority_summary_v1.csv
FOUND   JUDGEBERT_warning_summary_v1.csv


In [ ]:

import os
import pandas as pd
from datetime import date

folder = "/content/drive/MyDrive/JUDGEBERT_LMP"
results_path = os.path.join(folder, "JUDGEBERT_baseline_results_v2.csv")

results = pd.read_csv(results_path)

priority_counts = results["review_priority"].value_counts()
warning_columns = [
    "negation_warning",
    "modality_warning",
    "numbers_dates_money_warning",
    "legal_parties_warning"
]

def count_warning(col):
    return int(results[col].fillna(False).astype(bool).sum())

report = f"""# JUDGEBERT: Legal Meaning Preservation (LMP)

**Project report draft**
**Prepared:** {date.today().isoformat()}
**Languages targeted:** English and Hindi

## 1. Problem statement

Legal documents often contain complex terminology, obligations, exceptions,
conditions, deadlines, and rights. Simplifying legal language can improve
readability, but careless rewriting may alter the original legal meaning.

JUDGEBERT aims to compare an original legal passage with a simplified version
in the same language and provide an initial indication of possible meaning
preservation issues.

## 2. Objectives

- Compare original and simplified text using multilingual sentence embeddings.
- Flag possible changes to negation, legal obligation or permission terms,
  numeric details, and references to legal parties.
- Produce a preliminary 1–10 score for demonstration.
- Support an English/Hindi workflow, subject to appropriate same-language
  evaluation data and further validation.

## 3. Current methodology

The prototype uses the
`paraphrase-multilingual-MiniLM-L12-v2` sentence-transformer model to compute
semantic similarity between the two texts.

A rule-based layer checks for differences in:
- Negation terms.
- Obligation or permission terms such as must, shall, and may.
- Numbers, dates, and currency expressions.
- Selected legal party terms.

The current score combines semantic similarity with manually chosen penalties
for detected differences. These penalties have not been empirically calibrated.
The score is therefore a preliminary heuristic, not a validated legal rating.

## 4. Evaluation sample

The current evaluation file contains **{len(results)} pairs**.

### Review-priority counts

- CHECK: {int(priority_counts.get('CHECK', 0))}
- HIGH - check legal details: {int(priority_counts.get('HIGH - check legal details', 0))}
- LOWER - still verify: {int(priority_counts.get('LOWER - still verify', 0))}

### Rule warning counts

- Negation differences: {count_warning('negation_warning')}
- Obligation/permission differences: {count_warning('modality_warning')}
- Numbers, dates, or currency differences: {count_warning('numbers_dates_money_warning')}
- Legal party term differences: {count_warning('legal_parties_warning')}

These are warning counts, not confirmed legal errors. A rule may flag harmless
rewriting, and a rule may miss important meaning changes.

## 5. Limitations

1. Semantic similarity does not guarantee legal meaning preservation.
2. The score mapping and warning penalties have not been validated against a
   sufficiently large, reliable set of human-rated examples.
3. The evaluation sample is not evidence of performance on all legal documents.
4. The English/Hindi capability requires dedicated evaluation in each language.
5. The current rules cover only selected lexical patterns and may miss changes
   involving exceptions, scope, actors, cross-references, or legal interpretation.
6. The prototype is not legal advice and should not be used to make legal
   decisions without qualified human review.

## 6. Future work

- Obtain a reliable set of same-language original/simplified legal pairs.
- Collect independent human ratings using a documented rubric.
- Evaluate agreement between model outputs and human ratings.
- Calibrate score thresholds and penalties using held-out data.
- Expand Hindi-specific testing and legal terminology coverage.
- Compare the baseline against stronger models and document the results.
- Develop a user interface with clear warnings and exportable results.

## 7. Conclusion

JUDGEBERT currently demonstrates a multilingual semantic similarity baseline,
rule-based legal-detail warnings, and a preliminary scoring workflow. Its
current value is as a research prototype for identifying text pairs that
may need closer review. Further labelled-data collection and evaluation are
required before making claims about legal meaning preservation accuracy.
"""

report_path = os.path.join(folder, "JUDGEBERT_Project_Report_Draft.md")

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print("Report saved successfully:")
print(report_path)
print("\nEvaluation sample size:", len(results))
print("This is a draft; review it before final submission.")


Report saved successfully:
/content/drive/MyDrive/JUDGEBERT_LMP/JUDGEBERT_Project_Report_Draft.md

Evaluation sample size: 500
This is a draft; review it before final submission.


In [ ]:

import os

folder = "/content/drive/MyDrive/JUDGEBERT_LMP"
os.makedirs(folder, exist_ok=True)

readme = r"""# JUDGEBERT — Legal Meaning Preservation (LMP)

## Overview
JUDGEBERT is a research prototype that compares an original legal sentence
with a simplified version in the same language. It combines multilingual
sentence similarity with rule-based warnings for selected legal-detail changes.

## Current features
- English and Hindi text input in the interactive demo.
- Multilingual sentence embeddings using
  `paraphrase-multilingual-MiniLM-L12-v2`.
- Semantic similarity calculation.
- Warnings for possible differences in negation, obligation/permission terms,
  numbers/dates/currency, and selected legal-party terms.
- Preliminary heuristic score from 1 to 10.
- CSV export of evaluation and demo results.

## Method
1. Encode the original and simplified text.
2. Calculate cosine similarity between their sentence embeddings.
3. Compare selected legal-sensitive terms.
4. Produce a preliminary score and warning flags.

## Important limitations
- The score is heuristic and has not been calibrated against a sufficiently
  large, reliable set of human-rated examples.
- Semantic similarity does not prove legal meaning preservation.
- Rule warnings can include harmless wording changes and may miss other risks.
- English and Hindi performance must be evaluated separately with suitable data.
- This prototype is for research and demonstration, not legal advice.

## Project files
The Google Drive project folder contains the working notebook outputs,
baseline results, score results, summaries, and draft project report.
Copy only files suitable for public release into the GitHub repository.

## Future work
- Create a reliable set of same-language simplification pairs.
- Collect human ratings using a documented rubric.
- Evaluate agreement with human ratings and calibrate the score.
- Expand Hindi evaluation and legal-detail detection.
- Compare with stronger baselines.

## Status
Working prototype; further evaluation and validation are required.
"""

readme_path = os.path.join(folder, "README.md")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme)

print("README saved:")
print(readme_path)
print("\nNext: copy this README into your GitHub repository.")


README saved:
/content/drive/MyDrive/JUDGEBERT_LMP/README.md

Next: copy this README into your GitHub repository.


In [ ]:

import os
import zipfile

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

folder = "/content/drive/MyDrive/JUDGEBERT_LMP"
zip_path = "/content/JUDGEBERT_LMP_submission.zip"

files_to_include = [
    "README.md",
    "JUDGEBERT_Project_Report_Draft.md",
    "JUDGEBERT_baseline_results_v2.csv",
    "JUDGEBERT_LMP_scored_v1.csv",
    "JUDGEBERT_priority_summary_v1.csv",
    "JUDGEBERT_warning_summary_v1.csv",
    "JUDGEBERT_demo_assessment_history.csv",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in files_to_include:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            archive.write(path, arcname=filename)
            print("Included:", filename)
        else:
            print("Skipped (not found):", filename)

print("\nZIP created:", zip_path)

from google.colab import files
files.download(zip_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Included: README.md
Included: JUDGEBERT_Project_Report_Draft.md
Included: JUDGEBERT_baseline_results_v2.csv
Included: JUDGEBERT_LMP_scored_v1.csv
Included: JUDGEBERT_priority_summary_v1.csv
Included: JUDGEBERT_warning_summary_v1.csv
Included: JUDGEBERT_demo_assessment_history.csv

ZIP created: /content/JUDGEBERT_LMP_submission.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>